# From First Principles to Modern Large Language Models

**Author:** Pablo Leyva

An unbroken chain of derivation from set theory to modern LLMs.


## Table of contents

### A — Foundations
- Chapter 1: Sets, functions, logic, proofs
- Chapter 2: Numbers, sequences, limits, completeness
- Chapter 3: Continuity, univariate differentiation, chain rule
- Chapter 4: Multivariate calculus: partials, gradients, Jacobians
- Chapter 5: Linear algebra I: vector spaces, basis, linear maps
- Chapter 6: Linear algebra II: inner products, norms, eigenvalues, SVD
- Chapter 7: Convexity and optimization; gradient descent convergence

### B — Probability and Information
- Chapter 8: Probability foundations: sample spaces, sigma-algebras, Kolmogorov axioms
- Chapter 9: Random variables, distributions, CDF/PMF/PDF
- Chapter 10: Expectation, variance, covariance; Jensen's inequality
- Chapter 11: Information theory: self-information, entropy, cross-entropy, KL
- Chapter 12: Statistical inference: likelihood, MLE, ERM, bias-variance

### C — Stochastic Optimization
- Chapter 13: SGD: stochastic-approximation theorem; mini-batching; convergence sketch
- Chapter 14: Momentum, RMSProp, AdamW: derivation and bias-correction proof

### D — Neural Networks
- Chapter 15: MLPs as compositional functions; universal approximation
- Chapter 16: Activation functions: ReLU/GELU/softmax with derivatives
- Chapter 17: Loss functions: MSE, cross-entropy; gradients from first principles
- Chapter 18: Backpropagation: chain rule applied; reverse-mode AD as a graph algorithm

### E — Sequence Models and Attention
- Chapter 19: Embeddings: token to vector; lookup as a linear map; weight tying
- Chapter 20: RNN intuition; vanishing-gradient proof; why we need attention
- Chapter 21: Scaled dot-product attention: derivation, softmax-temperature analysis
- Chapter 22: Multi-head attention: parallel heads as concat-then-project; complexity
- Chapter 23: Transformer block: residual + LayerNorm/RMSNorm + FFN + attention; gradient-flow argument
- Chapter 24: Positional encoding: sinusoidal derivation, RoPE construction

### F — Pre-training
- Chapter 25: Causal masking; next-token prediction loss as MLE on the empirical distribution
- Chapter 26: Tokenization: BPE algorithm; greedy merge correctness
- Chapter 27: Pre-training pipeline: AdamW + warmup + cosine decay + gradient clipping; tiny-GPT training run

### G — Post-training
- Chapter 28: SFT, RLHF (PPO/GRPO), and DPO; train + post-train a tiny GPT

### H — Reinforcement Learning
- Chapter 29: MDP foundations: Bellman equations, value iteration, tabular Q-learning
- Chapter 30: Value-based deep RL: function approximation, DQN, max-entropy framework
- Chapter 31: Policy gradient, GRPO, and the RLHF/DPO bridge: tiny-GPT alignment loop



# Block A — Foundations


# Chapter 1 — Sets, functions, logic, proofs

We need a precise grammar for membership, functions, and proof before we can define real numbers (Ch. 5), linear maps (Ch. 8), probability (Ch. 15), or token embeddings (Ch. 19).

**Key definitions.** A *set* $S$ is a collection of distinct elements; $x \in S$ means $x$ is an element of $S$. The *power set* is $\mathcal{P}(S) = \{T : T \subset S\}$. A *function* $f : A \to B$ assigns each $a \in A$ exactly one $f(a) \in B$.


In [ ]:
from itertools import chain, combinations

def power_set(S):
    S = list(S)
    return [set(c) for r in range(len(S) + 1) for c in combinations(S, r)]

S = {'a', 'b', 'c'}
P = power_set(S)
for T in P:
    print(sorted(T))
print('|P(S)| =', len(P), '   2**|S| =', 2 ** len(S))
assert len(P) == 2 ** len(S), 'power-set cardinality identity failed'


## De Morgan's laws

For $A, B \subset U$:
$$(A \cup B)^c = A^c \cap B^c, \qquad (A \cap B)^c = A^c \cup B^c.$$


In [ ]:
U = set(range(1, 9))
A = {1, 3, 5, 7}
B = {2, 3, 5, 7}

comp = lambda X: U - X

lhs1, rhs1 = comp(A | B), comp(A) & comp(B)
lhs2, rhs2 = comp(A & B), comp(A) | comp(B)

print('(A u B)^c =', sorted(lhs1), '  A^c n B^c =', sorted(rhs1))
print('(A n B)^c =', sorted(lhs2), '  A^c u B^c =', sorted(rhs2))
assert lhs1 == rhs1 and lhs2 == rhs2, 'De Morgan failed'
print('Both De Morgan identities verified.')


## Injection, surjection, bijection

$f : A \to B$ is *injective* iff $f(a_1) = f(a_2) \Rightarrow a_1 = a_2$, *surjective* iff every $b \in B$ has a preimage, *bijective* iff both.


In [ ]:
def is_injective(f, A):
    seen = {}
    for a in A:
        b = f[a]
        if b in seen:
            return False
        seen[b] = a
    return True

def is_surjective(f, A, B):
    return set(f[a] for a in A) == set(B)

def is_bijective(f, A, B):
    return is_injective(f, A) and is_surjective(f, A, B)

A = [0, 1, 2, 3]
B = [0, 1, 2, 3, 4]
f = {0: 1, 1: 3, 2: 0, 3: 4}   # injective into B, not surjective
print('f =', f)
print('injective?', is_injective(f, A))
print('surjective onto B?', is_surjective(f, A, B))
print('bijective A -> B?', is_bijective(f, A, B))

# A bijection A -> A
g = {0: 2, 1: 0, 2: 3, 3: 1}
print('\ng =', g)
print('bijective A -> A?', is_bijective(g, A, A))


## Induction

Claim: $\sum_{k=1}^{n} k = n(n+1)/2$.

*Base:* $n = 1$: LHS $= 1 =$ RHS. *Step:* assume $S(n) = n(n+1)/2$; then
$S(n+1) = S(n) + (n+1) = n(n+1)/2 + (n+1) = (n+1)(n+2)/2$.


In [ ]:
import numpy as np

def S_direct(n):
    return sum(range(1, n + 1))

def S_closed(n):
    return n * (n + 1) // 2

# Direct check up to n = 20
for n in range(1, 21):
    assert S_direct(n) == S_closed(n), f'mismatch at n={n}'
print('Direct verification 1..20: OK')

# Numerical induction-step check: S_closed(n+1) - S_closed(n) == n+1
ns = np.arange(1, 1001)
lhs = np.array([S_closed(n + 1) - S_closed(n) for n in ns])
rhs = ns + 1
assert np.array_equal(lhs, rhs), 'induction step failed'
print('Base case S(1) =', S_closed(1))
print('Induction step S(n+1) - S(n) = n+1 verified for n = 1..1000')


## Connection to LLMs

A vocabulary $\mathcal{V}$ is a finite set of tokens. The embedding map $E : \mathcal{V} \to \mathbb{R}^d$ is a function; its lookup-table implementation requires the index $\mathcal{V} \to \{0, \ldots, |\mathcal{V}|-1\}$ to be a **bijection**. We revisit this in Chapter 19.


In [ ]:
import numpy as np
np.random.seed(0)

vocab = ['<bos>', '<eos>', 'the', 'cat', 'sat', 'on', 'mat']
V = len(vocab)
d = 4

tok2id = {tok: i for i, tok in enumerate(vocab)}
id2tok = {i: tok for tok, i in tok2id.items()}

# Bijectivity of vocabulary index
assert len(set(tok2id.values())) == V                  # injective
assert set(tok2id.values()) == set(range(V))           # surjective onto {0,...,V-1}
print('vocabulary index is a bijection V <-> {0,...,V-1}')

E = np.random.randn(V, d).astype(np.float32)
print('embedding matrix shape:', E.shape)
for tok in ['cat', 'mat']:
    print(f'E[{tok!r}] =', E[tok2id[tok]])


# Chapter 2 — Numbers, sequences, limits, completeness

We build $\mathbb{N} \subset \mathbb{Z} \subset \mathbb{Q} \subset \mathbb{R}$ and isolate the **completeness axiom** of $\mathbb{R}$: every nonempty subset that is bounded above has a supremum in $\mathbb{R}$. This single axiom is what powers every convergence theorem we will need later for SGD and Adam.


## Convergence: $\varepsilon$–$N$ on $S_n = \sum_{k=1}^n 1/k^2 \to \pi^2/6$

We compute partial sums, the gap $|S_n - \pi^2/6|$, and for each $\varepsilon$ the smallest $N$ such that $n \geq N \Rightarrow |S_n - \pi^2/6| < \varepsilon$.


In [ ]:
import numpy as np

target = np.pi**2 / 6
n_max = 2_000_000
ks = np.arange(1, n_max + 1, dtype=np.float64)
partial = np.cumsum(1.0 / ks**2)

for n in [1, 5, 10, 100, 1_000, 10_000, 100_000]:
    print(f'S_{n:>7d} = {partial[n-1]:.10f}   gap = {abs(partial[n-1] - target):.2e}')

gap = np.abs(partial - target)
for eps in [1e-2, 1e-4, 1e-6]:
    idx = np.argmax(gap < eps)
    # Verify monotone: from idx onward, gap stays < eps (true here since gap ~ 1/n).
    assert gap[idx] < eps
    print(f'eps = {eps:.0e}   smallest N = {idx + 1}')


## Cauchy criterion

A real sequence is Cauchy iff for every $\varepsilon > 0$ there exists $N$ with $|a_m - a_n| < \varepsilon$ for all $m, n \geq N$. We test $a_n = 1/n$ (Cauchy) against $b_n = (-1)^n$ (not Cauchy).


In [ ]:
import numpy as np

def is_cauchy(seq, eps):
    """Return smallest N such that sup_{m,n>=N} |seq[m]-seq[n]| < eps, or None."""
    seq = np.asarray(seq, dtype=np.float64)
    L = len(seq)
    for N in range(L):
        tail = seq[N:]
        if tail.max() - tail.min() < eps:
            return N
    return None

a = 1.0 / np.arange(1, 5001)
b = (-1.0) ** np.arange(5000)

for eps in [1e-1, 1e-2, 1e-3]:
    Na = is_cauchy(a, eps)
    Nb = is_cauchy(b, eps)
    print(f'eps = {eps:.0e}   a_n=1/n: N = {Na}    b_n=(-1)^n: N = {Nb}')


## $\sqrt{2} \notin \mathbb{Q}$ and Newton's method

**Proof recap.** If $\sqrt{2} = p/r$ in lowest terms, then $p^2 = 2r^2$ forces $p$ even, then $r$ even, contradicting $\gcd(p,r)=1$.

So $\sqrt{2}$ lives in $\mathbb{R} \setminus \mathbb{Q}$ — and the Newton iteration $x_{n+1} = (x_n + 2/x_n)/2$ produces a Cauchy sequence of rationals whose limit is $\sqrt{2}$, witnessing why we needed completeness in the first place.


In [ ]:
import numpy as np

x = 1.0
true = np.sqrt(2.0)
print(f'{ "n":>3}  {"x_n":>20}  {"|x_n - sqrt(2)|":>18}')
for n in range(11):
    print(f'{n:>3}  {x:>20.16f}  {abs(x - true):>18.2e}')
    x = 0.5 * (x + 2.0 / x)


## Bounded monotone convergence: telescoping series

$a_n = \sum_{k=1}^n \tfrac{1}{k(k+1)} = 1 - \tfrac{1}{n+1}$ is monotone increasing and bounded above by $1$. By Theorem 2.8 it must converge — and indeed to $\sup_n a_n = 1$.


In [ ]:
import numpy as np

n_max = 100_000
ks = np.arange(1, n_max + 1, dtype=np.float64)
terms = 1.0 / (ks * (ks + 1.0))
a = np.cumsum(terms)

monotone = bool(np.all(np.diff(a) >= 0))
bounded = bool(np.all(a <= 1.0))
print(f'monotone increasing: {monotone}')
print(f'bounded above by 1 : {bounded}')
for n in [1, 10, 100, 1_000, 10_000, 100_000]:
    print(f'a_{n:>6d} = {a[n-1]:.10f}   gap to 1 = {1 - a[n-1]:.2e}')
print(f'sup_n a_n (numerical) = {a.max():.12f}')


## Forward link

When we prove SGD converges in Chapter 13, the core lemma is: the loss sequence $(L(\theta_t))$ is monotone decreasing in expectation and bounded below by $0$, hence convergent — by **exactly** Theorem 2.8 of this chapter.


## Continuity, differentiation, chain rule

We numerically explore the four anchors of Chapter 3:
1. $\varepsilon$-$\delta$ continuity at a point.
2. Derivative as a limit (forward difference).
3. Chain rule.
4. Mean value theorem.

**Recall:** $f$ is continuous at $a$ iff $\forall\,\varepsilon>0\;\exists\,\delta>0:\;|x-a|<\delta \Rightarrow |f(x)-f(a)|<\varepsilon$.


In [ ]:
import numpy as np
np.random.seed(0)

# Numerical eps-delta certificate for f(x) = x^2 at a = 2.
# Strategy: for each eps, binary-search the largest delta in (0, 1] for which
# sup_{|x-a|<delta} |f(x)-f(a)| < eps holds (sampled densely).

def f(x):
    return x * x

a = 2.0
fa = f(a)

def violation(delta, n=4001):
    xs = np.linspace(a - delta, a + delta, n)
    return np.max(np.abs(f(xs) - fa))

def find_delta(eps, lo=0.0, hi=1.0, iters=60):
    # halving search: largest hi with violation(hi) < eps
    if violation(hi) < eps:
        return hi
    for _ in range(iters):
        mid = 0.5 * (lo + hi)
        if violation(mid) < eps:
            lo = mid
        else:
            hi = mid
    return lo

print(f'{"eps":>10} {"delta found":>15} {"sup|f(x)-f(a)|":>20}  ok?')
for eps in (1e-1, 1e-2, 1e-3):
    d = find_delta(eps)
    sup = violation(d)
    print(f'{eps:>10.0e} {d:>15.8e} {sup:>20.8e}  {sup < eps}')


## Derivative as a limit

$f'(a) = \lim_{h\to 0} \frac{f(a+h)-f(a)}{h}$.

The forward-difference truncation error for smooth $f$ is $O(h)$ (Taylor expansion). Below we plot it on a log-log axis if matplotlib is available; otherwise we print a table.


In [ ]:
import numpy as np

# Forward-difference derivative of sin at a grid of points.
xs = np.linspace(0.1, np.pi - 0.1, 200)
true = np.cos(xs)

hs = np.array([10.0 ** k for k in range(-1, -13, -1)])
errors = np.array([np.max(np.abs((np.sin(xs + h) - np.sin(xs)) / h - true)) for h in hs])

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(5, 3.5))
    ax.loglog(hs, errors, 'o-', label='forward diff error')
    ax.loglog(hs, hs, 'k--', alpha=0.5, label='O(h) reference')
    ax.set_xlabel('h'); ax.set_ylabel('max error'); ax.legend(); ax.grid(True, which='both', alpha=0.3)
    fig.tight_layout(); fig.savefig('fd_error.png', dpi=120)
    print('Saved fd_error.png')
except Exception as e:
    print(f'(matplotlib unavailable: {e}) -- printing table instead')

print(f'{"h":>12} {"max |fd - cos|":>20}')
for h, err in zip(hs, errors):
    print(f'{h:>12.1e} {err:>20.6e}')


## Chain rule

If $g$ is differentiable at $a$ and $f$ is differentiable at $g(a)$, then
$$ (f\circ g)'(a) = f'(g(a)) \cdot g'(a). $$

**Sketch (Carath\'eodory):** define $\phi(y)=(f(y)-f(b))/(y-b)$ for $y\ne b$ and $\phi(b)=f'(b)$. Then $\phi$ is continuous at $b=g(a)$ and $f(y)-f(b)=\phi(y)(y-b)$ identically. Substitute $y=g(x)$, divide by $x-a$, take $x\to a$.

We verify on $f(x)=\sin(x^2)$ at $x=1$: analytic value is $\cos(1)\cdot 2 \approx 1.0806$.


In [ ]:
import numpy as np

x = 1.0
F = lambda t: np.sin(t * t)

analytic = np.cos(x * x) * 2.0 * x  # f'(g(x)) * g'(x) with f=sin, g=x^2

# Central difference is O(h^2) and minimizes truncation+roundoff at h ~ 1e-5.
h = 1e-5
numeric = (F(x + h) - F(x - h)) / (2 * h)

print(f'analytic   f\'(g(x)) * g\'(x) = {analytic:.12f}')
print(f'numeric    central diff h=1e-5 = {numeric:.12f}')
print(f'abs error                       = {abs(analytic - numeric):.3e}')
assert abs(analytic - numeric) < 1e-7, 'chain rule check failed'
print('chain rule verified to <1e-7')


## Mean value theorem

If $f$ is continuous on $[a,b]$ and differentiable on $(a,b)$, then $\exists c \in (a,b)$ with
$$ f'(c) = \frac{f(b)-f(a)}{b-a}. $$

We find such a $c$ for $f(x)=x^3-2x$ on $[0,2]$ by bisection on $f'(x)-\text{slope}$.


In [ ]:
import numpy as np

def f(x):  return x**3 - 2*x
def fp(x): return 3*x**2 - 2

a, b = 0.0, 2.0
slope = (f(b) - f(a)) / (b - a)
g = lambda x: fp(x) - slope

# fp is continuous; g(0) = -2 - 1 = -3, g(2) = 10 - 1 = 9, sign change => bisection works.
lo, hi = a, b
assert g(lo) * g(hi) < 0
for _ in range(80):
    mid = 0.5 * (lo + hi)
    if g(lo) * g(mid) <= 0:
        hi = mid
    else:
        lo = mid
c = 0.5 * (lo + hi)

print(f'slope (f(b)-f(a))/(b-a) = {slope:.10f}')
print(f'witness c              = {c:.10f}')
print(f'f\'(c)                  = {fp(c):.10f}')
print(f'|f\'(c) - slope|        = {abs(fp(c) - slope):.3e}')
# Closed form: 3c^2 - 2 = 2  =>  c = sqrt(4/3)
print(f'closed form sqrt(4/3)   = {np.sqrt(4/3):.10f}')


## Multivariate calculus: partials, gradients, Jacobians

We move from $f:\mathbb{R}\to\mathbb{R}$ (Chapter 3) to $f:\mathbb{R}^n\to\mathbb{R}^m$. The right notion of differentiability is **Fréchet**: there exists a linear $L$ with $\|f(\mathbf{a}+\mathbf{h})-f(\mathbf{a})-L\mathbf{h}\| = o(\|\mathbf{h}\|)$. The matrix of $L$ is the **Jacobian** $J_f(\mathbf{a}) \in \mathbb{R}^{m\times n}$, whose $j$-th column is $\partial f/\partial x_j(\mathbf{a})$ (Theorem 4.5). When $m=1$, $\nabla f = J_f^\top$.

Below we verify the key facts numerically with finite differences.

In [ ]:
import numpy as np
np.random.seed(0)

# f: R^2 -> R, f(x,y) = x^2 y + sin(x+y)
def f(x, y):
    return x**2 * y + np.sin(x + y)

# Analytic gradient: df/dx = 2xy + cos(x+y), df/dy = x^2 + cos(x+y)
def grad_f_analytic(x, y):
    return np.array([2*x*y + np.cos(x+y), x**2 + np.cos(x+y)])

def grad_f_numeric(x, y, h=1e-6):
    dx = (f(x+h, y) - f(x-h, y)) / (2*h)
    dy = (f(x, y+h) - f(x, y-h)) / (2*h)
    return np.array([dx, dy])

x0, y0 = 1.0, 2.0
ga = grad_f_analytic(x0, y0)
gn = grad_f_numeric(x0, y0)
print(f'analytic grad at (1,2): {ga}')
print(f'numeric  grad at (1,2): {gn}')
print(f'max abs error: {np.max(np.abs(ga-gn)):.2e}')

## Jacobians by column

For $\mathbf{f}:\mathbb{R}^n\to\mathbb{R}^m$, the $j$-th column of $J_f(\mathbf{a})$ is $\partial \mathbf{f}/\partial x_j(\mathbf{a})$, computable as the central difference $(\mathbf{f}(\mathbf{a}+h\mathbf{e}_j)-\mathbf{f}(\mathbf{a}-h\mathbf{e}_j))/(2h)$. We test on $\mathbf{f}(x,y,z)=(xyz,\ x^2+\sin y+e^z)$, whose analytic Jacobian is

$$J_f = \begin{pmatrix} yz & xz & xy \\ 2x & \cos y & e^z \end{pmatrix}.$$

In [ ]:
import numpy as np

def F(v):
    x, y, z = v
    return np.array([x*y*z, x**2 + np.sin(y) + np.exp(z)])

def J_analytic(v):
    x, y, z = v
    return np.array([
        [y*z,      x*z,      x*y],
        [2*x,      np.cos(y), np.exp(z)],
    ])

def J_numeric(F, v, h=1e-6):
    v = np.asarray(v, dtype=float)
    n = v.size
    cols = []
    for j in range(n):
        ej = np.zeros(n); ej[j] = 1.0
        cols.append((F(v + h*ej) - F(v - h*ej)) / (2*h))
    return np.stack(cols, axis=1)

v0 = np.array([1.0, 0.5, -0.3])
Ja = J_analytic(v0)
Jn = J_numeric(F, v0)
print('analytic J:'); print(Ja)
print('numeric  J:'); print(Jn)
print(f'max abs error: {np.max(np.abs(Ja-Jn)):.2e}')

## Multivariate chain rule

Theorem 4.7: $J_{f\circ g}(\mathbf{a}) = J_f(g(\mathbf{a}))\, J_g(\mathbf{a})$. Take $g(t)=(\cos t,\sin t)$ and $f(x,y)=x^2+y^2$. Then $f\circ g \equiv 1$, so $(f\circ g)'(t)=0$ for every $t$. The chain rule must produce the same answer.

In [ ]:
import numpy as np

def g(t):
    return np.array([np.cos(t), np.sin(t)])

def gprime(t):
    return np.array([-np.sin(t), np.cos(t)])  # column vector (R -> R^2 has 2x1 Jacobian)

def grad_f(p):
    x, y = p
    return np.array([2*x, 2*y])  # row of J_f

ts = np.linspace(0, 2*np.pi, 7)
for t in ts:
    direct = 0.0  # f(g(t)) = 1, derivative is 0
    chain  = float(grad_f(g(t)) @ gprime(t))   # 1x2 @ 2x1
    print(f't={t:6.3f}  direct={direct:+.2e}  chain-rule={chain:+.2e}')

## Schwarz / Clairaut: equality of mixed partials

For $C^2$ functions, $\partial_x\partial_y f = \partial_y\partial_x f$ (Theorem 4.8). Numerically, both can be approximated by the second-order central difference

$$\partial_x\partial_y f(a,b) \approx \frac{f(a+h,b+k)-f(a+h,b-k)-f(a-h,b+k)+f(a-h,b-k)}{4hk}.$$

We test on $f(x,y) = x^3 y^2 + \sin(xy)$ at $(1,1)$. Analytically,
$\partial_y f = 2x^3 y + x\cos(xy)$, so $\partial_x\partial_y f = 6x^2 y + \cos(xy) - xy\sin(xy)$, which at $(1,1)$ is $6 + \cos(1) - \sin(1)$.

In [ ]:
import numpy as np

def fxy(x, y):
    return x**3 * y**2 + np.sin(x*y)

def mixed_partial(F, a, b, h=1e-3, k=1e-3):
    return (F(a+h, b+k) - F(a+h, b-k) - F(a-h, b+k) + F(a-h, b-k)) / (4*h*k)

a, b = 1.0, 1.0
dxdy = mixed_partial(fxy, a, b)
dydx = mixed_partial(lambda y, x: fxy(x, y), b, a)  # swap roles
analytic = 6*a**2*b + np.cos(a*b) - a*b*np.sin(a*b)

print(f'numeric  d/dx d/dy f at (1,1): {dxdy:.10f}')
print(f'numeric  d/dy d/dx f at (1,1): {dydx:.10f}')
print(f'analytic value             : {analytic:.10f}')
print(f'|dxdy - dydx| = {abs(dxdy-dydx):.2e}')
print(f'|num - analytic| = {abs(dxdy-analytic):.2e}')

## Connection to LLMs

A transformer is a composition $F = F_L\circ\cdots\circ F_1$. Theorem 4.7 says the gradient of the loss with respect to layer-$\ell$ parameters is a product of Jacobians from the loss back to that layer. Backpropagation never materializes those matrices: it propagates a *row vector* $\mathbf{v}^\top$ right-to-left via vector–Jacobian products (VJPs), one per layer, each at $O(\text{forward cost})$. We will derive reverse-mode autodiff formally in Chapter 18.

# Chapter 5 — Linear algebra I: vector spaces, basis, linear maps

Every transformer layer is a linear map between finite-dimensional real vector spaces, framed by bias terms and nonlinearities. Before attention (Ch. 21) or embeddings (Ch. 19), we need vector spaces, bases, dimension, kernels, images, and the rank–nullity theorem.

**Eight axioms of a vector space $V$ over a field $\mathbb{F}$.** For $u, v, w \in V$, $a, b \in \mathbb{F}$:

1. $(u+v)+w = u+(v+w)$
2. $u+v = v+u$
3. $\exists\, 0 \in V$ with $v+0=v$
4. $\forall v\,\exists (-v)$ with $v+(-v)=0$
5. $a(u+v) = au + av$
6. $(a+b)v = av + bv$
7. $(ab)v = a(bv)$
8. $1\cdot v = v$

The canonical example is $V = \mathbb{R}^d$ over $\mathbb{F} = \mathbb{R}$, which is exactly the embedding space used by language models with hidden dimension $d$.

In [ ]:
import numpy as np

def is_linearly_independent(vectors):
    """Vectors is a list/array of row vectors. Independent iff rank == count."""
    M = np.asarray(vectors, dtype=float)
    return int(np.linalg.matrix_rank(M)) == M.shape[0]

def dim_span(vectors):
    """Dimension of the span of a list of vectors."""
    M = np.asarray(vectors, dtype=float)
    return int(np.linalg.matrix_rank(M))

S = [(1, 0, 0), (0, 1, 0), (1, 1, 0)]
print('vectors:', S)
print('linearly independent?', is_linearly_independent(S))
print('dim of span:', dim_span(S))
assert dim_span(S) == 2, 'expected 2 since (1,1,0) = (1,0,0) + (0,1,0)'

## Basis and dimension

A **basis** of $V$ is a linearly independent spanning set. By the **Steinitz exchange lemma**, every basis of a finite-dimensional $V$ has the same cardinality, called $\dim V$. Below we compute the rank of a $4 \times 6$ matrix (the dimension of its column span / image) and extract a basis of its **null space** (kernel) via the right singular vectors of $A$ associated to zero singular values.

In [ ]:
import numpy as np
np.random.seed(0)

A = np.random.randint(-3, 4, size=(4, 6)).astype(float)
print('A =\n', A)

U, sigma, Vt = np.linalg.svd(A)
print('singular values:', np.round(sigma, 4))

# Numerical rank: count singular values above tolerance.
tol = max(A.shape) * np.finfo(float).eps * sigma.max()
rank = int((sigma > tol).sum())
print('rank(A) =', rank)

# Right singular vectors (rows of Vt) corresponding to zero singular values
# span the null space. There are n - rank of them, where n = 6.
n = A.shape[1]
null_basis = Vt[rank:]  # shape (n - rank, n)
print('dim ker(A) =', null_basis.shape[0])

# Verify A @ v == 0 for each null-space basis vector v.
for i, v in enumerate(null_basis):
    Av = A @ v
    print(f'A v_{i} norm = {np.linalg.norm(Av):.2e}')
    assert np.allclose(Av, 0, atol=1e-10)

## Linear maps and matrix representation

A linear map $T: V \to W$ satisfies $T(au+bv) = aTu + bTv$. In bases $(e_j)$ of $V = \mathbb{R}^n$ and $(f_i)$ of $W = \mathbb{R}^m$, the matrix $A \in \mathbb{R}^{m\times n}$ has $j$-th column equal to the coordinates of $T(e_j)$. If we change basis on $V = W = \mathbb{R}^n$ via an invertible $P$, the same map $T$ is represented in the new basis by $\tilde{A} = P^{-1} A P$. Numerically: for any $v \in \mathbb{R}^n$ we should have $A v = P\,\tilde{A}\,P^{-1} v$.

In [ ]:
import numpy as np
np.random.seed(0)

A = np.random.randn(3, 3)
# Build a random invertible P. Re-draw if singular (essentially never happens).
while True:
    P = np.random.randn(3, 3)
    if abs(np.linalg.det(P)) > 1e-6:
        break
P_inv = np.linalg.inv(P)
A_tilde = P_inv @ A @ P

v = np.random.randn(3)
lhs = A @ v
rhs = P @ A_tilde @ P_inv @ v
print('A v       =', lhs)
print('P A~ Pi v =', rhs)
print('max abs diff:', np.max(np.abs(lhs - rhs)))
assert np.allclose(lhs, rhs)

## Rank–nullity

**Theorem.** For $T: V \to W$ with $\dim V < \infty$, $\dim \ker T + \dim \mathrm{im}\,T = \dim V$. Equivalently, for $A \in \mathbb{R}^{m\times n}$, $\mathrm{rank}(A) + \dim \ker(A) = n$. We verify this for the $4 \times 6$ matrix above (so the sum should equal $6$).

In [ ]:
import numpy as np
np.random.seed(0)

A = np.random.randint(-3, 4, size=(4, 6)).astype(float)
U, sigma, Vt = np.linalg.svd(A)
tol = max(A.shape) * np.finfo(float).eps * sigma.max()
rank = int((sigma > tol).sum())
nullity = A.shape[1] - rank  # by definition of SVD null-space basis
print(f'rank(A)    = {rank}')
print(f'nullity(A) = {nullity}')
print(f'sum        = {rank + nullity}')
print(f'n (cols)   = {A.shape[1]}')
assert rank + nullity == A.shape[1], 'rank-nullity violated'

## Connection to LLMs

A transformer with hidden dimension $d$ operates in $\mathbb{R}^d$. The query/key/value projections in attention (Ch. 21) are linear maps $\mathbb{R}^d \to \mathbb{R}^{d_k}$. Token embeddings (Ch. 19) are a linear map $\mathbb{R}^{|\mathcal{V}|} \to \mathbb{R}^d$ applied to a one-hot input. LoRA constrains weight *updates* to a low-rank subspace, an explicit application of $\dim \mathrm{im}\,T \le \min(m, n)$. Rank–nullity will reappear whenever we count free parameters or degrees of freedom.

## Inner products, norms, and Cauchy-Schwarz

The dot product on $\mathbb{R}^n$ is $\langle x,y\rangle=\sum_i x_iy_i$, and its induced norm is $\|x\|=\sqrt{\langle x,x\rangle}$. Cauchy-Schwarz says $|\langle x,y\rangle|\le\|x\|\,\|y\|$. We numerically verify this on 50 random pairs in $\mathbb{R}^{10}$ by computing the ratio $|\langle x,y\rangle|/(\|x\|\|y\|)$ -- it must always be $\le 1$.


In [ ]:
import numpy as np

np.random.seed(0)
n_pairs, dim = 50, 10
ratios = []
for _ in range(n_pairs):
    x = np.random.randn(dim)
    y = np.random.randn(dim)
    inner = float(np.dot(x, y))
    nx = float(np.linalg.norm(x))
    ny = float(np.linalg.norm(y))
    ratios.append(abs(inner) / (nx * ny))

ratios = np.array(ratios)
print(f'pairs tested        : {n_pairs}')
print(f'max  |<x,y>|/(|x||y|): {ratios.max():.6f}')
print(f'mean |<x,y>|/(|x||y|): {ratios.mean():.6f}')
assert ratios.max() <= 1.0 + 1e-12, 'Cauchy-Schwarz violated!'
print('Cauchy-Schwarz holds for all 50 pairs.')


## Eigenvalues of symmetric matrices

The spectral theorem says every real symmetric matrix $A$ has an orthonormal eigenbasis: $A=Q\Lambda Q^\top$. Below we build a symmetric $5\times5$ matrix $A=M+M^\top$, diagonalize with `np.linalg.eigh` (which exploits symmetry), and verify both $A v_i=\lambda_i v_i$ for every $i$ and $V^\top V=I$ (orthonormal eigenvectors).


In [ ]:
import numpy as np

np.random.seed(0)
M = np.random.randn(5, 5)
A = M + M.T
assert np.allclose(A, A.T)

eigvals, V = np.linalg.eigh(A)
print('eigenvalues:', np.round(eigvals, 6))

max_eig_residual = 0.0
for i in range(A.shape[0]):
    lhs = A @ V[:, i]
    rhs = eigvals[i] * V[:, i]
    max_eig_residual = max(max_eig_residual, float(np.linalg.norm(lhs - rhs)))
print(f'max ||A v_i - lambda_i v_i|| : {max_eig_residual:.2e}')

ortho_err = float(np.linalg.norm(V.T @ V - np.eye(5)))
print(f'||V^T V - I||_F              : {ortho_err:.2e}')

assert max_eig_residual < 1e-10
assert ortho_err < 1e-10
print('Spectral theorem verified numerically.')


## Singular value decomposition

Every $A\in\mathbb{R}^{m\times n}$ admits an SVD $A=U\Sigma V^\top$ with $U,V$ orthogonal and $\Sigma$ diagonal with nonnegative entries. We build a random $5\times3$ matrix, run `np.linalg.svd`, and reconstruct $A$ to high precision.


In [ ]:
import numpy as np

np.random.seed(0)
A = np.random.randn(5, 3)
U, s, Vt = np.linalg.svd(A, full_matrices=True)

Sigma = np.zeros_like(A)
Sigma[:len(s), :len(s)] = np.diag(s)
A_reco = U @ Sigma @ Vt

print('singular values        :', np.round(s, 6))
print('U shape, Sigma shape, Vt shape:', U.shape, Sigma.shape, Vt.shape)
print(f'||A - U Sigma V^T||_F  : {np.linalg.norm(A - A_reco):.2e}')
print(f'||U^T U - I||_F        : {np.linalg.norm(U.T @ U - np.eye(5)):.2e}')
print(f'||V V^T - I||_F        : {np.linalg.norm(Vt @ Vt.T - np.eye(3)):.2e}')

assert np.linalg.norm(A - A_reco) < 1e-10
print('SVD reconstruction verified.')


## Eckart-Young: best low-rank approximation

Eckart-Young says the best rank-$k$ approximation of $A$ in Frobenius norm is the truncated SVD $A_k=\sum_{i=1}^k\sigma_i u_i v_i^\top$, with squared error $\sum_{i>k}\sigma_i^2$. We compute rank-1 and rank-2 truncations of the same $5\times3$ matrix and observe a strict monotone decrease in error, matching the predicted tail sum of squared singular values.


In [ ]:
import numpy as np

np.random.seed(0)
A = np.random.randn(5, 3)
U, s, Vt = np.linalg.svd(A, full_matrices=False)

errors = {}
for k in (1, 2, 3):
    A_k = (U[:, :k] * s[:k]) @ Vt[:k, :]
    err = float(np.linalg.norm(A - A_k))
    predicted = float(np.sqrt(np.sum(s[k:] ** 2)))
    errors[k] = err
    print(f'rank {k}: ||A - A_k||_F = {err:.6f}   predicted sqrt(sum sigma_i^2 for i>k) = {predicted:.6f}')

assert errors[1] >= errors[2] >= errors[3]
assert errors[3] < 1e-10
print('Frobenius error decreases monotonically with rank, as Eckart-Young predicts.')


# Chapter 7 — Convexity and gradient descent

Why convex? Because it is the **only** setting where we can write down honest convergence rates for $x_{t+1} = x_t - \eta \nabla f(x_t)$. Real transformer losses are non-convex (Chapter 13, 14, 27), but the convex rates supply the vocabulary we use to reason about them: $L$-smoothness, condition number $\kappa = L/\mu$, contraction.

**Definitions.** $f$ is *convex* iff $f(t x + (1-t) y) \le t f(x) + (1-t) f(y)$. It is *$L$-smooth* iff $\|\nabla f(x) - \nabla f(y)\| \le L\|x - y\|$. It is *$\mu$-strongly convex* iff $f(y) \ge f(x) + \langle \nabla f(x), y - x\rangle + \tfrac{\mu}{2}\|y - x\|^2$.


In [ ]:
import numpy as np

# A strongly convex quadratic f(x, y) = (x - 1)^2 + 2 (y + 1)^2
# Hessian = diag(2, 4), so mu = 2, L = 4, kappa = 2, minimum at x* = (1, -1).
def f(z):
    return (z[0] - 1.0)**2 + 2.0 * (z[1] + 1.0)**2

def grad_f(z):
    return np.array([2.0 * (z[0] - 1.0), 4.0 * (z[1] + 1.0)])

x_star = np.array([1.0, -1.0])

# Numerical convexity check: midpoint inequality on random pairs.
np.random.seed(0)
violations = 0
for _ in range(2000):
    a, b = np.random.randn(2) * 3, np.random.randn(2) * 3
    t = np.random.rand()
    if f(t * a + (1 - t) * b) > t * f(a) + (1 - t) * f(b) + 1e-9:
        violations += 1
print(f'Convexity violations in 2000 samples: {violations}')

# Contour plot, with table fallback if matplotlib is unavailable.
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    xs = np.linspace(-2, 4, 80); ys = np.linspace(-4, 2, 80)
    X, Y = np.meshgrid(xs, ys)
    Z = (X - 1.0)**2 + 2.0 * (Y + 1.0)**2
    plt.figure(figsize=(5, 4))
    plt.contour(X, Y, Z, levels=20)
    plt.scatter([1], [-1], c='red', label='x*')
    plt.title('f(x,y) = (x-1)^2 + 2(y+1)^2'); plt.legend()
    plt.savefig('ch07_contour.png', dpi=80, bbox_inches='tight'); plt.close()
    print('Saved contour to ch07_contour.png')
except Exception as e:
    print(f'matplotlib unavailable ({e}); printing slice table:')
    for x in [-1, 0, 1, 2, 3]:
        row = [f(np.array([x, y])) for y in [-3, -2, -1, 0, 1]]
        print(f'x={x:+d}: ' + '  '.join(f'{v:6.2f}' for v in row))


## Descent lemma and the $1/L$ step

**Descent lemma.** $L$-smoothness implies $f(y) \le f(x) + \langle \nabla f(x), y - x\rangle + \tfrac{L}{2}\|y - x\|^2$. Plugging $y = x - \tfrac{1}{L}\nabla f(x)$ yields
$$f(x_{t+1}) \le f(x_t) - \tfrac{1}{2L}\|\nabla f(x_t)\|^2,$$
i.e. **monotone descent**. For $\mu$-strongly convex $f$ this strengthens to $\|x_{t+1} - x^*\|^2 \le (1 - \mu/L)\|x_t - x^*\|^2$, a *geometric* contraction.


In [ ]:
import numpy as np

def f(z):
    return (z[0] - 1.0)**2 + 2.0 * (z[1] + 1.0)**2
def grad_f(z):
    return np.array([2.0 * (z[0] - 1.0), 4.0 * (z[1] + 1.0)])
x_star = np.array([1.0, -1.0])

# GD on the strongly convex quadratic with eta = 1/L.
L = 4.0; mu = 2.0; eta = 1.0 / L
x = np.array([3.5, 1.5])
history = []
for t in range(200):
    history.append(np.linalg.norm(x - x_star)**2)
    x = x - eta * grad_f(x)
history.append(np.linalg.norm(x - x_star)**2)

predicted_factor = 1.0 - mu / L  # 0.5
print(f'Predicted contraction per step: {predicted_factor}')
for t in [0, 1, 2, 5, 10, 20, 50, 100, 200]:
    pred = history[0] * predicted_factor**t
    print(f't={t:3d}  ||x_t - x*||^2 = {history[t]:.3e}   predicted bound = {pred:.3e}')

ratios = [history[t+1] / history[t] for t in range(50) if history[t] > 1e-30]
print(f'Median empirical per-step ratio over first 50 steps: {np.median(ratios):.4f}')


## $O(1/T)$ rate without strong convexity

If $f$ is $L$-smooth and convex but **not** $\mu$-strongly convex, the rate degrades from geometric to $f(x_T) - f^* \le \tfrac{L \|x_0 - x^*\|^2}{2T}$. We exhibit this on a least-squares problem $f(x) = \|Ax - b\|^2$ where $A \in \mathbb{R}^{20 \times 10}$ has a tiny smallest singular value: in the slow direction the effective $\mu$ is essentially zero.


In [ ]:
import numpy as np

np.random.seed(0)
U, _ = np.linalg.qr(np.random.randn(20, 20))
V, _ = np.linalg.qr(np.random.randn(10, 10))
sigma = np.linspace(1.0, 1e-3, 10)  # smallest singular value 1e-3 -> tiny mu
S = np.zeros((20, 10)); np.fill_diagonal(S, sigma)
A = U @ S @ V.T
b = np.random.randn(20)

x_star_ls, *_ = np.linalg.lstsq(A, b, rcond=None)
f_star = float(np.linalg.norm(A @ x_star_ls - b)**2)

L_ls = 2.0 * (sigma.max()**2)
print(f'sigma_max={sigma.max():.4f}, sigma_min={sigma.min():.4f}, L={L_ls:.4f}')

x = np.zeros(10); eta = 1.0 / L_ls
gaps = []
for t in range(1000):
    r = A @ x - b
    g = 2.0 * (A.T @ r)
    x = x - eta * g
    gaps.append(float(np.linalg.norm(A @ x - b)**2) - f_star)

Ts = [1, 2, 5, 10, 50, 100, 500, 1000]
print('   T       f(x_T)-f*        bound L||x0-x*||^2/(2T)')
C = L_ls * float(np.linalg.norm(x_star_ls)**2) / 2.0
for T in Ts:
    print(f'  {T:4d}    {gaps[T-1]:.3e}      {C/T:.3e}')

tail_T = np.arange(100, 1001)
tail_g = np.array(gaps[99:1000])
tail_g = np.maximum(tail_g, 1e-20)
slope, intercept = np.polyfit(np.log(tail_T), np.log(tail_g), 1)
print(f'log-log slope on T in [100, 1000]: {slope:.3f}  (theory predicts ~ -1)')

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    plt.figure(figsize=(5, 4))
    plt.loglog(np.arange(1, 1001), np.maximum(gaps, 1e-20), label='f(x_T) - f*')
    plt.loglog(np.arange(1, 1001), C / np.arange(1, 1001), '--', label='L||x0-x*||^2/(2T)')
    plt.xlabel('T'); plt.ylabel('suboptimality'); plt.legend(); plt.title('GD on ill-conditioned LS')
    plt.savefig('ch07_rate.png', dpi=80, bbox_inches='tight'); plt.close()
    print('Saved log-log rate plot to ch07_rate.png')
except Exception as e:
    print(f'(plot skipped: {e})')


## Strong convexity: $\eta = 1/L$ vs. the optimal $\eta = 2/(L + \mu)$

For a quadratic with eigenvalues in $[\mu, L]$ the per-step contraction with $\eta = 1/L$ is $1 - \mu/L$, while with the optimal $\eta = 2/(L + \mu)$ it improves to $((\kappa - 1)/(\kappa + 1))^2$ where $\kappa = L/\mu$. Both rates are linear; the optimal one has a strictly smaller constant.


In [ ]:
import numpy as np
L = 4.0; mu = 2.0
x_star = np.array([1.0, -1.0])
def grad_f(z):
    return np.array([2.0 * (z[0] - 1.0), 4.0 * (z[1] + 1.0)])

def run(eta, T=40):
    x = np.array([3.5, 1.5])
    err = [np.linalg.norm(x - x_star)**2]
    for _ in range(T):
        x = x - eta * grad_f(x)
        err.append(np.linalg.norm(x - x_star)**2)
    return err

eta_basic = 1.0 / L
eta_opt = 2.0 / (L + mu)
h_basic = run(eta_basic)
h_opt = run(eta_opt)
kappa = L / mu
rate_basic = 1.0 - mu / L
rate_opt = ((kappa - 1.0) / (kappa + 1.0))**2
print(f'theory: eta=1/L contracts by {rate_basic:.3f}/step; eta=2/(L+mu) by {rate_opt:.3f}/step')
print(f'  t   ||x_t-x*||^2 (1/L)    ||x_t-x*||^2 (2/(L+mu))')
for t in [0, 5, 10, 20, 40]:
    print(f'  {t:3d}     {h_basic[t]:.3e}             {h_opt[t]:.3e}')

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    plt.figure(figsize=(5, 4))
    plt.semilogy(h_basic, label='eta = 1/L')
    plt.semilogy(h_opt, label='eta = 2/(L+mu) (optimal)')
    plt.xlabel('t'); plt.ylabel('||x_t - x*||^2'); plt.legend(); plt.title('Step-size comparison')
    plt.savefig('ch07_steps.png', dpi=80, bbox_inches='tight'); plt.close()
    print('Saved step-size comparison to ch07_steps.png')
except Exception as e:
    print(f'(plot skipped: {e})')


## Forward to the LLM chapters

Transformer training is non-convex, but the same skeleton recurs: a descent-lemma-style inequality (now in expectation) plus a contraction or telescoping argument. SGD (Chapter 13) replaces $(\star)$ with $\mathbb{E}[f(x_{t+1})] \le f(x_t) - \tfrac{\eta}{2}\|\nabla f(x_t)\|^2 + \tfrac{L \eta^2 \sigma^2}{2}$. AdamW (Chapter 14) builds preconditioners on top. The pre-training pipeline (Chapter 27) uses warmup precisely because the *local* $L$ is large early in training, exactly the regime where the descent lemma forbids a large step.


# Block B — Probability and Information


## Probability foundations

A **probability space** is a triple $(\Omega, \mathcal{F}, \mathbb{P})$ where

- $\Omega$ is a non-empty *sample space*;
- $\mathcal{F} \subseteq 2^{\Omega}$ is a *$\sigma$-algebra* (contains $\Omega$, closed under complements and countable unions);
- $\mathbb{P}: \mathcal{F} \to [0, 1]$ satisfies the **Kolmogorov axioms**: non-negativity, $\mathbb{P}(\Omega) = 1$, and countable additivity for disjoint events.

We instantiate the smallest non-trivial example: two fair dice with $\Omega = \{1,\dots,6\}^2$, $\mathcal{F} = 2^{\Omega}$, and uniform $\mathbb{P}(A) = |A|/36$.


In [ ]:
import numpy as np
from itertools import product

# Build Omega = {1,...,6}^2 explicitly as a frozenset of tuples.
Omega = frozenset(product(range(1, 7), repeat=2))
assert len(Omega) == 36

def P(A):
    """Uniform probability measure on Omega: P(A) = |A| / |Omega|."""
    A = set(A)
    assert A.issubset(Omega), 'A must be a subset of Omega'
    return len(A) / len(Omega)

# (K2) normalization
print('P(Omega) =', P(Omega))

# (K3) finite additivity on two disjoint events:
# A = first die equals 1; B = first die equals 2 -- disjoint.
A = {w for w in Omega if w[0] == 1}
B = {w for w in Omega if w[0] == 2}
assert A.isdisjoint(B)
print('P(A) + P(B) =', P(A) + P(B), '   P(A union B) =', P(A | B))
assert abs((P(A) + P(B)) - P(A | B)) < 1e-12


### Inclusion-exclusion and the union bound

For any events $A, B$,
$$\mathbb{P}(A \cup B) = \mathbb{P}(A) + \mathbb{P}(B) - \mathbb{P}(A \cap B).$$

For any countable family,
$$\mathbb{P}\!\left(\bigcup_n A_n\right) \leq \sum_n \mathbb{P}(A_n).$$

Equality fails as soon as the events overlap.


In [ ]:
# Inclusion-exclusion: A = first die = 6, B = second die = 6.
A = {w for w in Omega if w[0] == 6}
B = {w for w in Omega if w[1] == 6}
lhs = P(A | B)
rhs = P(A) + P(B) - P(A & B)
print(f'P(A cup B) = {lhs:.6f}   P(A) + P(B) - P(A cap B) = {rhs:.6f}')
assert abs(lhs - rhs) < 1e-12

# Union bound on three events:
# A1 = first die >= 5; A2 = second die >= 5; A3 = sum >= 10.
A1 = {w for w in Omega if w[0] >= 5}
A2 = {w for w in Omega if w[1] >= 5}
A3 = {w for w in Omega if w[0] + w[1] >= 10}
union = A1 | A2 | A3
print(f'P(union) = {P(union):.6f}   sum P(Ai) = {P(A1) + P(A2) + P(A3):.6f}')
assert P(union) <= P(A1) + P(A2) + P(A3) + 1e-12


### Conditional probability and Bayes' theorem

For $\mathbb{P}(B) > 0$, $\mathbb{P}(A \mid B) = \mathbb{P}(A \cap B) / \mathbb{P}(B)$ and
$$\mathbb{P}(A \mid B) = \frac{\mathbb{P}(B \mid A)\, \mathbb{P}(A)}{\mathbb{P}(B)}.$$

**Disease testing.** Prior $\mathbb{P}(D) = 0.001$, sensitivity $\mathbb{P}(+\mid D) = 0.99$, specificity $\mathbb{P}(-\mid D^c) = 0.95$. We compute $\mathbb{P}(D \mid +)$ analytically and verify by Monte Carlo.


In [ ]:
import numpy as np

prior = 0.001
sens  = 0.99    # P(+ | D)
spec  = 0.95    # P(- | not D), so false-positive rate = 1 - spec = 0.05
fpr   = 1 - spec

# Law of total probability: P(+) = P(+|D)P(D) + P(+|notD)P(notD)
p_pos = sens * prior + fpr * (1 - prior)
# Bayes: P(D | +)
p_d_given_pos = sens * prior / p_pos
print(f'Analytic P(D | +) = {p_d_given_pos:.6f}')

# Monte Carlo verification.
np.random.seed(0)
n = 10_000
diseased = np.random.rand(n) < prior
test_pos = np.where(
    diseased,
    np.random.rand(n) < sens,    # given disease, true-positive
    np.random.rand(n) < fpr,     # given no disease, false-positive
)
n_pos = test_pos.sum()
if n_pos > 0:
    mc = (diseased & test_pos).sum() / n_pos
else:
    mc = float('nan')
print(f'Monte Carlo P(D | +)  = {mc:.6f}   (n_pos = {int(n_pos)} of {n})')
print('Note: with prior 0.001 and n=10000, the MC estimate is noisy but order-of-magnitude consistent.')


### Independence

Events $A, B$ are **independent** iff $\mathbb{P}(A \cap B) = \mathbb{P}(A)\mathbb{P}(B)$. This is a *property of the measure*, not a property of the sets.

On the two-dice space, the events "first die = 6" and "second die = 6" are independent (the dice are physically uncoupled in the uniform measure). The events "first die = 6" and "sum = 12" are *not* independent: knowing the sum is 12 forces both dice to be 6.


In [ ]:
A = {w for w in Omega if w[0] == 6}            # first die = 6
B = {w for w in Omega if w[1] == 6}            # second die = 6
C = {w for w in Omega if w[0] + w[1] == 12}    # sum = 12

print('Independence test for (A, B):')
print(f'  P(A) * P(B)    = {P(A) * P(B):.6f}')
print(f'  P(A cap B)     = {P(A & B):.6f}   --> independent\n')

print('Independence test for (A, C):')
print(f'  P(A) * P(C)    = {P(A) * P(C):.6f}')
print(f'  P(A cap C)     = {P(A & C):.6f}   --> NOT independent')

assert abs(P(A & B) - P(A) * P(B)) < 1e-12
assert abs(P(A & C) - P(A) * P(C)) > 1e-6


### Looking ahead

A causal language model (Chapter 25) defines, for each context $c$, a probability measure on the finite vocabulary $\mathcal{V}$ via the softmax output. Every claim in this chapter -- additivity, the union bound, Bayes, total probability, continuity -- transfers verbatim to that measure. Sampling, beam search, nucleus truncation, and importance-weighted training are all operations on a Kolmogorov probability space.


## Random Variables, Distributions, CDF / PMF / PDF

A random variable $X:\Omega\to\mathbb{R}$ pushes the probability $\mathbb{P}$ on $(\Omega,\mathcal{F})$ to a distribution $\mu_X$ on $\mathbb{R}$. We will (1) build a discrete RV (categorical) by softmaxing fixed logits, (2) compute its PMF and CDF, and (3) sample from it via inverse-CDF.


In [ ]:
import numpy as np

np.random.seed(0)

# Categorical with K=5 outcomes from fixed logits (think: tiny LM head).
logits = np.array([2.0, 1.0, 0.5, -0.5, 0.0])
exp_z = np.exp(logits - logits.max())  # numerical stability
pmf = exp_z / exp_z.sum()

print('PMF :', np.round(pmf, 4))
print('Sum :', pmf.sum())  # must be 1 (Theorem 9.2)

cdf = np.cumsum(pmf)
print('CDF :', np.round(cdf, 4))

# Inverse-CDF sampling (Theorem 9.4): draw U ~ Uniform[0,1], output min k with CDF[k] >= U.
n = 10_000
u = np.random.rand(n)
samples = np.searchsorted(cdf, u, side='left')

empirical = np.bincount(samples, minlength=5) / n
print('Empirical :', np.round(empirical, 4))
print('True PMF  :', np.round(pmf, 4))
print('Max |emp - true| :', np.max(np.abs(empirical - pmf)))


### Continuous distributions and densities

For a continuous RV, $\mathbb{P}(X\in A) = \int_A f_X(x)\,dx$. The standard normal has $f_X(x)=\tfrac{1}{\sqrt{2\pi}}e^{-x^2/2}$. Theorem 9.2 forces $\int_{\mathbb{R}} f_X = 1$. We verify by Riemann sum and build the CDF by cumulative sum.


In [ ]:
import numpy as np

def standard_normal_pdf(x):
    return np.exp(-0.5 * x * x) / np.sqrt(2 * np.pi)

# Truncate to [-8, 8]: the tail mass beyond is < 1e-15.
x_grid = np.linspace(-8.0, 8.0, 16_001)
dx = x_grid[1] - x_grid[0]
f_vals = standard_normal_pdf(x_grid)

total_mass = np.sum(f_vals) * dx  # midpoint/Riemann sum
print(f'Numerical integral of f_X : {total_mass:.10f}  (target 1)')

# CDF via cumulative sum of f_vals * dx.
F_vals = np.cumsum(f_vals) * dx
for x in [-2.0, -1.0, 0.0, 1.0, 2.0]:
    idx = int(np.searchsorted(x_grid, x))
    print(f'F_X({x:+.1f}) = {F_vals[idx]:.6f}')


### Change of variables

Theorem 9.3: if $Y = g(X)$ with $g$ strictly monotone $C^1$, then $f_Y(y) = f_X(g^{-1}(y))\,|(g^{-1})'(y)|$.

Take $X\sim\mathrm{Uniform}[0,1]$ so $f_X = 1$ on $[0,1]$, and $g(x) = -\ln x$ on $(0,1]$. Then $g^{-1}(y) = e^{-y}$, $(g^{-1})'(y) = -e^{-y}$, and
$$f_Y(y) = 1 \cdot |-e^{-y}| = e^{-y},\quad y \geq 0,$$
i.e. $Y\sim\mathrm{Exp}(1)$. We confirm with a histogram of 10000 samples.


In [ ]:
import numpy as np

np.random.seed(0)

x_samples = np.random.rand(10_000)         # X ~ U[0,1]
y_samples = -np.log(x_samples)             # Y = g(X) = -ln X

# Empirical histogram on [0, 6]
edges = np.linspace(0, 6, 25)
counts, _ = np.histogram(y_samples, bins=edges)
centers = 0.5 * (edges[:-1] + edges[1:])
widths = np.diff(edges)
emp_density = counts / (counts.sum() * widths)
true_density = np.exp(-centers)            # f_Y(y) = e^{-y}

print(' y center   empirical f_Y   true f_Y   |diff|')
for c, e, t in zip(centers[:8], emp_density[:8], true_density[:8]):
    print(f'  {c:5.3f}      {e:8.4f}     {t:8.4f}   {abs(e-t):.4f}')

print(f'\nSample mean of Y : {y_samples.mean():.4f}  (true E[Y] = 1)')


### Inverse-CDF sampling, revisited

We re-run inverse-CDF sampling on the categorical from cell 2 and watch the empirical PMF converge to the true PMF as $n$ grows. This is exactly how a language model converts a softmax row into a token id (modulo temperature/top-$k$/top-$p$): cumulative-sum the probabilities, draw $U\sim U[0,1]$, output the first index whose cumulative probability $\geq U$.


In [ ]:
import numpy as np

np.random.seed(0)

logits = np.array([2.0, 1.0, 0.5, -0.5, 0.0])
exp_z = np.exp(logits - logits.max())
pmf = exp_z / exp_z.sum()
cdf = np.cumsum(pmf)

def inverse_cdf_sample(cdf, n, rng):
    u = rng.random(n)
    return np.searchsorted(cdf, u, side='left')

rng = np.random.default_rng(0)
for n in [100, 1_000, 10_000, 100_000]:
    s = inverse_cdf_sample(cdf, n, rng)
    emp = np.bincount(s, minlength=len(pmf)) / n
    err = np.max(np.abs(emp - pmf))
    print(f'n = {n:>7}  max |emp - true| = {err:.4f}')

print('\nTrue PMF:', np.round(pmf, 4))


# Chapter 10 — Expectation, Variance, Covariance, Jensen

We compress a distribution into summary numbers: the **expectation** $\mathbb{E}[X]$ and the **variance** $\mathrm{Var}(X) = \mathbb{E}[(X - \mathbb{E}X)^2]$. We then verify the most useful structural facts numerically: linearity (no independence required), the Jensen inequality for convex $\phi$, Markov, Chebyshev, and the weak law of large numbers.


In [ ]:
import numpy as np
np.random.seed(0)

# Discrete distribution on 5 outcomes.
values = np.array([-2.0, -1.0, 0.0, 1.0, 3.0])
probs  = np.array([0.10, 0.25, 0.30, 0.25, 0.10])
assert np.isclose(probs.sum(), 1.0)

EX_analytic   = float(np.sum(values * probs))
VarX_analytic = float(np.sum((values - EX_analytic)**2 * probs))

# Sample 10000 times.
samples = np.random.choice(values, size=10000, p=probs)
EX_emp   = float(samples.mean())
VarX_emp = float(samples.var(ddof=0))

print(f'E[X]  analytic = {EX_analytic:.6f}   empirical = {EX_emp:.6f}')
print(f'Var X analytic = {VarX_analytic:.6f}   empirical = {VarX_emp:.6f}')


## Linearity of expectation — without independence

$\mathbb{E}[aX + bY] = a\mathbb{E}[X] + b\mathbb{E}[Y]$ holds even when $X$ and $Y$ are *perfectly* dependent. Below, $Y = 2X + 1$, so $X$ determines $Y$ exactly; linearity is unaffected.


In [ ]:
import numpy as np
np.random.seed(0)

p = 0.3
n = 10000
X = (np.random.rand(n) < p).astype(np.float64)   # Bernoulli(p)
Y = 2.0 * X + 1.0                                # perfectly determined by X

EX     = X.mean()
EY     = Y.mean()
EX_pY  = (X + Y).mean()

print(f'E[X]        ~ {EX:.4f}    (analytic = {p})')
print(f'E[Y]        ~ {EY:.4f}    (analytic = {2*p + 1})')
print(f'E[X + Y]    ~ {EX_pY:.4f}  vs  E[X] + E[Y] = {EX + EY:.4f}')

# Covariance is maximal here, but linearity still holds.
cov_XY = np.cov(X, Y, ddof=0)[0, 1]
print(f'Cov(X, Y) = {cov_XY:.4f}   (X, Y are perfectly correlated)')


## Jensen's inequality

If $\phi$ is convex, then $\phi(\mathbb{E}[X]) \le \mathbb{E}[\phi(X)]$.

We test two convex functions: $\phi(x) = e^x$ on $X \sim \mathrm{Unif}\{-1,+1\}$, and $\phi(p) = -\log p$ on a categorical distribution (which connects directly to the entropy of Chapter 11).


In [ ]:
import numpy as np
np.random.seed(0)

# (1) phi(x) = exp(x), X uniform on {-1, +1}
X = np.random.choice([-1.0, 1.0], size=20000)
phi_of_EX = np.exp(X.mean())              # ~ exp(0) = 1
E_phi_X   = np.exp(X).mean()              # ~ (e + 1/e)/2 ~ 1.5431
print(f'phi(E[X]) = {phi_of_EX:.6f}')
print(f'E[phi(X)] = {E_phi_X:.6f}   (analytic = {(np.e + 1/np.e)/2:.6f})')
print(f'Jensen holds (E[phi(X)] >= phi(E[X])): {E_phi_X >= phi_of_EX}')

# (2) phi(p) = -log p, applied to outcomes drawn from a categorical p_true.
# E[-log p_true(X)] = entropy H(p_true).  By Jensen, H(p) >= -log E[p(X)] = -log sum p^2.
p_true = np.array([0.5, 0.25, 0.15, 0.10])
K = len(p_true)
draws  = np.random.choice(K, size=50000, p=p_true)
neg_log_p_X = -np.log(p_true[draws])
H_emp = neg_log_p_X.mean()
H_an  = -float(np.sum(p_true * np.log(p_true)))
jensen_lb = -np.log(np.sum(p_true**2))      # >= 0; tight only at uniform
print(f'\nEntropy H(p_true)  empirical = {H_emp:.6f}   analytic = {H_an:.6f}')
print(f'Jensen lower bound -log E[p(X)] = {jensen_lb:.6f}   (<= H)')


## Markov, Chebyshev, and the weak law of large numbers

Markov: $\mathbb{P}(X \ge a) \le \mathbb{E}[X]/a$ for $X \ge 0$.

Chebyshev: $\mathbb{P}(|X - \mu| \ge k\sigma) \le 1/k^2$.

Weak LLN: for i.i.d. $X_i$ with finite variance, $\bar X_n \to \mu$ in probability — proved directly via Chebyshev applied to $\bar X_n$, since $\mathrm{Var}(\bar X_n) = \sigma^2/n$.


In [ ]:
import numpy as np
np.random.seed(0)

# X_i ~ Uniform[0, 1], so mu = 1/2 and sigma^2 = 1/12.
mu, sigma2 = 0.5, 1.0/12.0
ns = np.unique(np.round(np.geomspace(10, 10000, num=12)).astype(int))

rows = []
for n in ns:
    Xn = np.random.uniform(0.0, 1.0, size=n)
    bar = float(Xn.mean())
    band = float(np.sqrt(sigma2 / n))           # one std-dev of bar X_n
    rows.append((n, bar, band))

print(f'{"n":>7} {"bar X_n":>12} {"|bar - mu|":>14} {"sigma/sqrt(n)":>16}')
for n, bar, band in rows:
    print(f'{n:>7d} {bar:>12.6f} {abs(bar-mu):>14.6f} {band:>16.6f}')

# Try to plot; fall back to the table above if matplotlib is unavailable.
try:
    import matplotlib.pyplot as plt
    ns_arr   = np.array([r[0] for r in rows], dtype=float)
    bars_arr = np.array([r[1] for r in rows], dtype=float)
    band_arr = np.array([r[2] for r in rows], dtype=float)
    plt.figure(figsize=(7, 4))
    plt.semilogx(ns_arr, bars_arr, 'o-', label=r'$\bar X_n$')
    plt.fill_between(ns_arr, mu - 2*band_arr, mu + 2*band_arr,
                     alpha=0.2, label=r'$\mu \pm 2\sigma/\sqrt{n}$ (Chebyshev / CLT band)')
    plt.axhline(mu, linestyle='--', label=r'$\mu = 1/2$')
    plt.xlabel('n'); plt.ylabel('sample mean'); plt.legend()
    plt.title('Weak law of large numbers, Uniform[0,1]')
    plt.tight_layout()
    plt.show()
except Exception as exc:
    print(f'matplotlib unavailable ({exc!r}); the table above is the fallback view.')


## Connection to LLMs

Training loss is $\mathcal{L}(\theta) = \mathbb{E}_{x \sim \mathcal{D}}[\ell(\theta; x)]$. Mini-batch SGD replaces this expectation by a sample mean of size $B$. By **linearity of expectation**, the SGD gradient is unbiased: $\mathbb{E}[\nabla \hat{\mathcal{L}}] = \nabla \mathcal{L}$. By the **variance-of-a-sum** identity (with independence within a batch), $\mathrm{Var}(\nabla \hat{\mathcal{L}}) = \Theta(1/B)$ — larger batches give a lower-variance estimator, and the **weak LLN** says we recover the true loss in probability as $B \to \infty$. **Jensen's inequality** powers the ELBO of variational inference (Chapter 22). Variance-reduction techniques for the SGD gradient are revisited in Chapter 13.


# Chapter 11 — Information theory: entropy, cross-entropy, KL

We use the natural log $\ln$ throughout. Self-information is $I(x) = -\ln p(x)$; entropy is $H(p) = \mathbb{E}_{x\sim p}[-\ln p(x)]$; cross-entropy is $H(p, q) = \mathbb{E}_{x \sim p}[-\ln q(x)]$; KL is $D_{\mathrm{KL}}(p \| q) = \mathbb{E}_{x \sim p}[\ln p(x)/q(x)]$. The two non-trivial facts proved in the chapter are: (1) **Gibbs**, $D_{\mathrm{KL}}(p \| q) \ge 0$; (2) the **decomposition** $H(p, q) = H(p) + D_{\mathrm{KL}}(p \| q)$. Everything else follows.


In [ ]:
import numpy as np

EPS = 1e-12

def entropy(p):
    p = np.asarray(p, dtype=float)
    # convention 0 ln 0 = 0 via clipping; contributions of zeros are zero anyway
    return float(-np.sum(np.where(p > 0, p * np.log(np.clip(p, EPS, 1.0)), 0.0)))

def cross_entropy(p, q):
    p = np.asarray(p, dtype=float); q = np.asarray(q, dtype=float)
    return float(-np.sum(np.where(p > 0, p * np.log(np.clip(q, EPS, 1.0)), 0.0)))

def kl(p, q):
    p = np.asarray(p, dtype=float); q = np.asarray(q, dtype=float)
    ratio = np.log(np.clip(p, EPS, 1.0)) - np.log(np.clip(q, EPS, 1.0))
    return float(np.sum(np.where(p > 0, p * ratio, 0.0)))

# Two categoricals over 5 tokens.
p = np.array([0.05, 0.10, 0.50, 0.20, 0.15])
q = np.array([0.20, 0.20, 0.20, 0.20, 0.20])
assert np.isclose(p.sum(), 1.0) and np.isclose(q.sum(), 1.0)

Hp = entropy(p)
Hpq = cross_entropy(p, q)
Dpq = kl(p, q)
print(f'H(p)        = {Hp:.6f} nats')
print(f'H(p, q)     = {Hpq:.6f} nats')
print(f'D_KL(p||q)  = {Dpq:.6f} nats')
print(f'H(p) + D_KL = {Hp + Dpq:.6f} nats')
print(f'decomposition residual: {abs(Hpq - (Hp + Dpq)):.2e}')


## Gibbs' inequality

$D_{\mathrm{KL}}(p \| q) \ge 0$ for every pair of PMFs on the same support, with equality iff $p = q$. Proof by Jensen on the convex map $-\ln$ (Chapter 10). We sample 50 random Dirichlet pairs and confirm.


In [ ]:
rng = np.random.default_rng(0)
K = 5
alpha = np.ones(K)
ps = rng.dirichlet(alpha, size=50)
qs = rng.dirichlet(alpha, size=50)

kls = np.array([kl(p_i, q_i) for p_i, q_i in zip(ps, qs)])
self_kls = np.array([kl(p_i, p_i) for p_i in ps])

print(f'min  D_KL(p_i || q_i) over 50 pairs: {kls.min():.6f}')
print(f'mean D_KL(p_i || q_i) over 50 pairs: {kls.mean():.6f}')
print(f'all D_KL >= 0?                       {bool(np.all(kls >= -1e-12))}')
print(f'max |D_KL(p_i || p_i)|:              {np.max(np.abs(self_kls)):.2e}')


## Maximum entropy

On a finite support of size $K$, $H(p) \le \ln K$ with equality iff $p$ is uniform. Proof: $D_{\mathrm{KL}}(p \| u) = \ln K - H(p) \ge 0$ by Gibbs.


In [ ]:
K = 8
u = np.full(K, 1.0 / K)
Hu = entropy(u)
print(f'H(uniform on {K}) = {Hu:.6f} nats')
print(f'ln {K}            = {np.log(K):.6f} nats')
print(f'difference        = {abs(Hu - np.log(K)):.2e}')

rng = np.random.default_rng(0)
ps = rng.dirichlet(np.ones(K), size=100)
Hs = np.array([entropy(p_i) for p_i in ps])
print(f'\n100 random non-uniform p\'s on K={K}:')
print(f'  max H(p) = {Hs.max():.6f}  (must be < ln K = {np.log(K):.6f})')
print(f'  all H(p) < ln K? {bool(np.all(Hs < np.log(K) - 1e-12))}')


## Mutual information

$I(X; Y) = H(X) + H(Y) - H(X, Y) = D_{\mathrm{KL}}(p_{X,Y} \| p_X \otimes p_Y)$. Both formulas must agree, and both must vanish when $X, Y$ are independent.


In [ ]:
# Correlated joint on {0,1}^2: P(X=Y) is large.
# Rows = X in {0,1}, cols = Y in {0,1}.
P = np.array([[0.40, 0.10],
              [0.10, 0.40]])
assert np.isclose(P.sum(), 1.0)

pX = P.sum(axis=1)
pY = P.sum(axis=0)
P_indep = np.outer(pX, pY)

H_X  = entropy(pX)
H_Y  = entropy(pY)
H_XY = entropy(P.flatten())

I_via_entropies = H_X + H_Y - H_XY
I_via_kl        = kl(P.flatten(), P_indep.flatten())

print('Correlated joint:')
print(f'  H(X)        = {H_X:.6f}')
print(f'  H(Y)        = {H_Y:.6f}')
print(f'  H(X, Y)     = {H_XY:.6f}')
print(f'  I via H     = {I_via_entropies:.6f}')
print(f'  I via KL    = {I_via_kl:.6f}')
print(f'  agreement   = {abs(I_via_entropies - I_via_kl):.2e}')

# Independent joint: I should be 0.
P_prod = P_indep
I_prod = (entropy(P_prod.sum(axis=1)) + entropy(P_prod.sum(axis=0))
          - entropy(P_prod.flatten()))
I_prod_kl = kl(P_prod.flatten(), np.outer(P_prod.sum(axis=1), P_prod.sum(axis=0)).flatten())
print('\nIndependent joint p_X \u2297 p_Y:')
print(f'  I via H     = {I_prod:.2e}')
print(f'  I via KL    = {I_prod_kl:.2e}')


**Takeaway.** Cross-entropy $H(p_{\mathrm{data}}, p_\theta)$ is the next-token loss; minimizing it minimizes $D_{\mathrm{KL}}(p_{\mathrm{data}} \| p_\theta)$ since $H(p_{\mathrm{data}})$ is $\theta$-independent (Chapter 17, 25). KL appears again as the trust-region penalty in RLHF / DPO (Chapter 28). Entropy of the model's softmax controls sampling diversity at decoding time.


# Chapter 12 — Statistical inference: likelihood, MLE, ERM, bias-variance

We treat data as iid samples from a parametric family $\{p_\theta\}$ and use the **likelihood** to pick the best $\theta$. We then verify three claims numerically: (i) MLE for a Gaussian mean equals the sample mean; (ii) MLE for a categorical model equals the empirical distribution and minimizes cross-entropy; (iii) MSE = Bias$^2$ + Variance.


In [ ]:
import numpy as np

# --- Gaussian MLE: simulate N(mu=2, sigma=1), compute log-likelihood on a mu-grid ---
np.random.seed(0)
mu_true, sigma = 2.0, 1.0
n = 1000
X = np.random.normal(mu_true, sigma, size=n)

def log_lik_gauss(mu, X, sigma=1.0):
    return -0.5 * n * np.log(2*np.pi*sigma**2) - 0.5/sigma**2 * np.sum((X - mu)**2)

grid = np.linspace(1.0, 3.0, 4001)
ll = np.array([log_lik_gauss(m, X, sigma) for m in grid])
mu_hat_grid = grid[int(np.argmax(ll))]
mu_hat_closed = X.mean()

print(f'sample mean (closed form MLE): {mu_hat_closed:.6f}')
print(f'argmax over grid             : {mu_hat_grid:.6f}')
print(f'|difference|                 : {abs(mu_hat_grid - mu_hat_closed):.4e}')
assert abs(mu_hat_grid - mu_hat_closed) < 1e-3


## MLE = minimum cross-entropy (categorical case)

For a categorical model with classes $\{1,\dots,K\}$, the log-likelihood divided by $n$ is $\sum_k \hat p_n(k)\log p_\theta(k) = -H(\hat p_n, p_\theta)$. Maximizing the likelihood is therefore identical to minimizing cross-entropy, and the unique minimizer (subject to $\sum_k \theta_k = 1$) is $\theta = \hat p_n$ — provable by Lagrange multipliers, verified here numerically.


In [ ]:
import numpy as np

np.random.seed(0)
K = 4
p_true = np.array([0.1, 0.2, 0.3, 0.4])
n = 500
samples = np.random.choice(K, size=n, p=p_true)
p_hat = np.bincount(samples, minlength=K) / n
print(f'empirical distribution p_hat: {p_hat}')

def cross_entropy(p, q, eps=1e-12):
    return -np.sum(p * np.log(q + eps))

# Sweep theta on a small grid around p_hat (a tiny convex perturbation toward each vertex)
best_H, best_theta = np.inf, None
for alpha in np.linspace(0.0, 0.5, 11):
    for k in range(K):
        e_k = np.zeros(K); e_k[k] = 1.0
        theta = (1 - alpha) * p_hat + alpha * e_k
        H = cross_entropy(p_hat, theta)
        if H < best_H:
            best_H, best_theta = H, theta

H_at_phat = cross_entropy(p_hat, p_hat)
print(f'H(p_hat, p_hat)         = {H_at_phat:.6f}')
print(f'min H over grid         = {best_H:.6f}')
print(f'argmin theta over grid  = {best_theta}')
assert np.allclose(best_theta, p_hat)
assert abs(best_H - H_at_phat) < 1e-9


## Bias-variance decomposition

We estimate $\mu = 0$ for $X_i \sim \mathcal{N}(0,1)$ with two estimators:
- $\hat\mu_1 = \bar X_n$ (unbiased, variance $1/n$);
- $\hat\mu_2 = (\bar X_n + 1)/2$ (biased toward $1/2$, variance $1/(4n)$).

Run $T = 1000$ trials at $n = 10$, compute empirical $\mathrm{Bias}^2$, $\mathrm{Var}$, $\mathrm{MSE}$, and verify $\mathrm{MSE} = \mathrm{Bias}^2 + \mathrm{Var}$.


In [ ]:
import numpy as np

np.random.seed(0)
T, n, mu_true = 1000, 10, 0.0
samples = np.random.normal(mu_true, 1.0, size=(T, n))
xbar = samples.mean(axis=1)
mu1 = xbar
mu2 = (xbar + 1.0) / 2.0

def report(name, est, mu_true):
    bias = est.mean() - mu_true
    var = est.var(ddof=0)
    mse = ((est - mu_true)**2).mean()
    print(f'{name:>10s}: bias^2={bias**2:.5f}  var={var:.5f}  '
          f'bias^2+var={bias**2+var:.5f}  MSE={mse:.5f}')
    assert abs((bias**2 + var) - mse) < 1e-12
    return bias**2, var, mse

report('mu1 = Xbar', mu1, mu_true)
report('mu2 biased', mu2, mu_true)
print('Decomposition Bias^2 + Var = MSE verified for both estimators.')


## ERM as a generalization of MLE

Replace $-\log p_\theta(x)$ by an arbitrary loss $\ell(\theta;x)$ and minimize the empirical mean. Linear regression with squared loss $\ell(w; (x,y)) = (y - wx)^2$ is the canonical example; the ERM solution is the **normal equation** $\hat w = (\sum_i x_i y_i)/(\sum_i x_i^2)$.


In [ ]:
import numpy as np

np.random.seed(0)
w_true, sigma_eps, n = 3.0, 0.5, 100
x = np.random.normal(0.0, 1.0, size=n)
eps = np.random.normal(0.0, sigma_eps, size=n)
y = w_true * x + eps

# Normal equation in 1D: w_hat = (x . y) / (x . x)
w_hat = float(np.dot(x, y) / np.dot(x, x))
emp_risk = float(np.mean((y - w_hat * x)**2))
print(f'true w        : {w_true}')
print(f'ERM estimate  : {w_hat:.6f}')
print(f'|w_hat - w*|  : {abs(w_hat - w_true):.4e}')
print(f'empirical risk: {emp_risk:.6f}  (noise variance {sigma_eps**2:.4f})')
assert abs(w_hat - w_true) < 0.1


## Connection to LLMs

An autoregressive language model factors $p_\theta(x_{1:T}) = \prod_t p_\theta(x_t \mid x_{<t})$. The pre-training loss is
$$-\frac{1}{N}\sum_{\text{seq}}\sum_t \log p_\theta(x_t \mid x_{<t}) \;=\; H(\hat p_n,\,p_\theta),$$
i.e. cross-entropy with the empirical token distribution. By Theorem 12.1 this is exactly MLE; by its corollary it is KL projection of the model onto the empirical corpus. Bias-variance (Theorem 12.3) then governs the under/overfitting trade-off explored in Chapter 27.


# Block C — Stochastic Optimization


## SGD: stochastic-approximation theorem; mini-batching; convergence sketch

Full-batch gradient descent (Chapter 7) requires a pass over every data point per step. For a corpus of $10^{13}$ tokens that is unaffordable. Stochastic gradient descent (SGD) replaces $\nabla F(\theta) = \mathbb{E}_\xi \nabla f(\theta;\xi)$ with a sample estimate $\hat g_B = \frac{1}{B}\sum_{j=1}^B \nabla f(\theta;\xi_j)$ and accepts noise in exchange for cheap steps.

We illustrate four facts from this chapter on tiny problems:
1. SGD is noisier than GD but still converges.
2. Variance of $\hat g_B$ scales as $1/B$.
3. On a smooth non-convex toy, $\frac{1}{T}\sum_t \|\nabla F(\theta_t)\|^2 = O(1/\sqrt{T})$.
4. Constant step plateaus at a noise floor; diminishing step (Robbins--Monro) converges.


In [ ]:
import numpy as np
try:
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception:
    HAVE_MPL = False

np.random.seed(0)
n = 1000
y = np.random.normal(loc=3.0, scale=1.0, size=n)  # F(theta) = (1/n) sum (theta - y_i)^2; min at mean(y)
theta_star = y.mean()

def loss(theta):
    return float(np.mean((theta - y) ** 2))

def full_grad(theta):
    return 2.0 * (theta - y.mean())

def stoch_grad(theta, B=1, rng=None):
    rng = rng or np.random
    idx = rng.randint(0, n, size=B)
    return 2.0 * (theta - y[idx].mean())

T = 200
eta = 0.05
theta_gd = 0.0
loss_gd = []
for t in range(T):
    loss_gd.append(loss(theta_gd))
    theta_gd -= eta * full_grad(theta_gd)

rng = np.random.RandomState(0)
theta_sgd = 0.0
loss_sgd = []
for t in range(T):
    loss_sgd.append(loss(theta_sgd))
    theta_sgd -= eta * stoch_grad(theta_sgd, B=1, rng=rng)

print(f'optimum theta* = {theta_star:.4f}')
print(f'GD final theta  = {theta_gd:.4f}, loss = {loss_gd[-1]:.4f}')
print(f'SGD final theta = {theta_sgd:.4f}, loss = {loss_sgd[-1]:.4f}')
if HAVE_MPL:
    plt.figure()
    plt.plot(loss_gd, label='Full-batch GD')
    plt.plot(loss_sgd, label='SGD (B=1)', alpha=0.7)
    plt.xlabel('iteration'); plt.ylabel('loss'); plt.legend(); plt.title('GD vs SGD on least squares')
    plt.show()


### Variance reduction by batching

Theorem 13.1: $\mathbb{E}\|\hat g_B - \nabla F\|^2 \leq \sigma^2/B$. We verify this on the same least-squares problem at $\theta = 0$ by drawing $200$ random batches for each $B$ and measuring the empirical variance of the resulting gradient estimates.


In [ ]:
import numpy as np
np.random.seed(0)
n = 1000
y = np.random.normal(loc=3.0, scale=1.0, size=n)
true_grad_at_0 = 2.0 * (0.0 - y.mean())

rng = np.random.RandomState(0)
Bs = [1, 8, 64, 256]
K = 200  # number of random batches per B
print(f'{"B":>5}  {"empirical Var":>14}  {"sigma^2 / B":>14}  {"ratio":>8}')
sigma2 = 4.0 * np.var(y)  # population variance of g(theta=0;xi) = 2*(0 - y_i)
for B in Bs:
    estimates = np.array([2.0 * (0.0 - y[rng.randint(0, n, size=B)].mean()) for _ in range(K)])
    var_emp = float(np.var(estimates))
    pred = sigma2 / B
    print(f'{B:>5}  {var_emp:>14.5f}  {pred:>14.5f}  {var_emp/pred:>8.3f}')


### SGD on a smooth non-convex toy

Let $F(\theta) = \theta^2/2 + 0.5\sin(5\theta)$. This is $L$-smooth (with $L = 1 + 12.5 = 13.5$, since $|F''| \leq 1 + 12.5$) but non-convex. Theorem 13.2 predicts the running average of $\|\nabla F(\theta_t)\|^2$ decays like $1/\sqrt{T}$.


In [ ]:
import numpy as np
try:
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception:
    HAVE_MPL = False

np.random.seed(0)

def grad_F(theta):
    return theta + 2.5 * np.cos(5.0 * theta)

T = 5000
eta = 0.02
noise_sigma = 1.0
rng = np.random.RandomState(0)
theta = 2.0
g2 = []
for t in range(T):
    g = grad_F(theta) + noise_sigma * rng.randn()
    g2.append(grad_F(theta) ** 2)
    theta -= eta * g

g2 = np.array(g2)
running_avg = np.cumsum(g2) / np.arange(1, T + 1)
ts = np.arange(1, T + 1)
ref = running_avg[10] * np.sqrt(11) / np.sqrt(ts)
print(f'avg ||grad F||^2 at T=100 : {running_avg[99]:.4f}')
print(f'avg ||grad F||^2 at T=1000: {running_avg[999]:.4f}')
print(f'avg ||grad F||^2 at T=5000: {running_avg[-1]:.4f}')
print(f'ratio T=100/T=5000        : {running_avg[99]/running_avg[-1]:.2f}  (expected ~ sqrt(50) = 7.07)')
if HAVE_MPL:
    plt.figure()
    plt.loglog(ts, running_avg, label='running avg ||grad F||^2')
    plt.loglog(ts, ref, '--', label='reference ~ 1/sqrt(T)')
    plt.xlabel('T'); plt.ylabel('avg sq grad'); plt.legend(); plt.title('SGD: 1/sqrt(T) decay')
    plt.show()


### Constant vs diminishing step (Robbins--Monro)

Strongly-convex quadratic $F(\theta) = \frac{1}{2}\theta^2$, with additive noise on the gradient. A constant step $\eta$ leaves the iterates orbiting the optimum at a noise floor $\sim \eta \sigma^2$; the diminishing schedule $\eta_t = c/(t+1)$ satisfies $\sum \eta_t = \infty, \sum \eta_t^2 < \infty$ and converges (slowly).


In [ ]:
import numpy as np
try:
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception:
    HAVE_MPL = False

np.random.seed(0)
T = 5000
sigma = 1.0
rng = np.random.RandomState(0)

# constant step
eta_c = 0.1
theta = 2.0
dist_const = []
for t in range(T):
    g = theta + sigma * rng.randn()
    theta -= eta_c * g
    dist_const.append(theta ** 2)

# diminishing step
rng = np.random.RandomState(0)
c = 1.0
theta = 2.0
dist_dim = []
for t in range(T):
    eta_t = c / (t + 1)
    g = theta + sigma * rng.randn()
    theta -= eta_t * g
    dist_dim.append(theta ** 2)

dist_const = np.array(dist_const)
dist_dim = np.array(dist_dim)
tail_const = float(np.mean(dist_const[-500:]))
tail_dim = float(np.mean(dist_dim[-500:]))
print(f'constant step eta=0.1  tail E[(theta-theta*)^2] = {tail_const:.5f}  (noise floor ~ eta*sigma^2/2 = {eta_c*sigma**2/2:.5f})')
print(f'diminishing step c/(t+1) tail E[(theta-theta*)^2] = {tail_dim:.5f}  (predicted O(1/T))')
if HAVE_MPL:
    plt.figure()
    ts = np.arange(1, T + 1)
    plt.loglog(ts, np.maximum(dist_const, 1e-8), label='constant eta=0.1')
    plt.loglog(ts, np.maximum(dist_dim, 1e-8), label='eta_t = 1/(t+1)')
    plt.loglog(ts, 1.0/ts, '--', label='1/T reference')
    plt.xlabel('t'); plt.ylabel('(theta_t - theta*)^2'); plt.legend(); plt.title('Constant vs diminishing step')
    plt.show()


**Takeaway.** The four experiments above match each chapter theorem: SGD converges (noisier than GD); $\mathrm{Var}(\hat g_B) \propto 1/B$; the running average of $\|\nabla F\|^2$ decays like $1/\sqrt{T}$ on a smooth non-convex problem; and only Robbins--Monro step sizes drive the iterates to the true optimum in the presence of persistent gradient noise. Pre-training an LLM (Chapter 23) inherits this entire picture; AdamW (Chapter 14) layers momentum and adaptive scaling on top.


# Chapter 14 --- Momentum, RMSProp, AdamW: derivation + bias-correction proof

Plain SGD (Chapter 13) suffers on ill-conditioned losses: it oscillates across stiff directions and crawls along flat ones. **Momentum** smooths the trajectory by giving the iterate inertia. We start with the canonical demo: a quadratic with condition number $\kappa = 100$.

$F(x, y) = \tfrac{1}{2}(x^2 + 100\, y^2)$, optimum at the origin.

In [ ]:
import numpy as np

def grad_quad(theta):
    return np.array([theta[0], 100.0 * theta[1]])

def run_sgd(eta=0.018, T=80, theta0=(2.0, 2.0)):
    theta = np.array(theta0, dtype=float); traj = [theta.copy()]
    for _ in range(T):
        theta = theta - eta * grad_quad(theta)
        traj.append(theta.copy())
    return np.array(traj)

def run_momentum(eta=0.018, beta=0.9, T=80, theta0=(2.0, 2.0)):
    theta = np.array(theta0, dtype=float); v = np.zeros_like(theta); traj = [theta.copy()]
    for _ in range(T):
        v = beta * v + grad_quad(theta)
        theta = theta - eta * v
        traj.append(theta.copy())
    return np.array(traj)

tr_sgd = run_sgd(); tr_mom = run_momentum()
print('final |theta| -- SGD     :', np.linalg.norm(tr_sgd[-1]))
print('final |theta| -- momentum:', np.linalg.norm(tr_mom[-1]))
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(5, 4))
    plt.plot(tr_sgd[:, 0], tr_sgd[:, 1], 'o-', ms=3, label='SGD')
    plt.plot(tr_mom[:, 0], tr_mom[:, 1], 's-', ms=3, label='Momentum (beta=0.9)')
    plt.scatter([0], [0], c='k', marker='*', s=80, label='optimum')
    plt.xlabel('x'); plt.ylabel('y'); plt.legend(); plt.title('Ill-conditioned quadratic')
    plt.tight_layout(); plt.show()
except Exception as e:
    print('matplotlib unavailable:', e)

## RMSProp --- per-coordinate adaptive rescaling

RMSProp keeps an EMA of squared gradients and divides each step by its square root, so coordinates with historically large gradients receive smaller effective learning rates. On our quadratic the $y$-coordinate has gradient magnitude $100\times$ larger than $x$; RMSProp evens this out automatically.

In [ ]:
import numpy as np

def grad_quad(theta):
    return np.array([theta[0], 100.0 * theta[1]])

def run_rmsprop(eta=0.05, beta2=0.9, eps=1e-8, T=80, theta0=(2.0, 2.0)):
    theta = np.array(theta0, dtype=float); v = np.zeros_like(theta); traj = [theta.copy()]
    for _ in range(T):
        g = grad_quad(theta)
        v = beta2 * v + (1 - beta2) * g * g
        theta = theta - eta * g / (np.sqrt(v) + eps)
        traj.append(theta.copy())
    return np.array(traj)

tr_rms = run_rmsprop()
print('final theta (RMSProp):', tr_rms[-1])
print('final |theta|        :', np.linalg.norm(tr_rms[-1]))
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(5, 4))
    plt.plot(tr_rms[:, 0], tr_rms[:, 1], 'd-', ms=3, color='C2', label='RMSProp')
    plt.scatter([0], [0], c='k', marker='*', s=80)
    plt.xlabel('x'); plt.ylabel('y'); plt.legend(); plt.title('RMSProp trajectory')
    plt.tight_layout(); plt.show()
except Exception as e:
    print('matplotlib unavailable:', e)

## Adam from scratch --- with bias correction

Adam = momentum (first moment) + RMSProp (second moment) + bias correction. We implement it in $\sim$ 25 lines and verify convergence on $F(\theta) = \tfrac{1}{2}\|\theta - \theta^\star\|^2$ with $\theta^\star = (3, -2)$. The gradient is $g = \theta - \theta^\star$, constant in expectation along a single trajectory; we will see $\hat m_t$ track this gradient closely from $t=1$.

In [ ]:
import numpy as np
np.random.seed(0)

theta_star = np.array([3.0, -2.0])
def grad(theta):
    return theta - theta_star

def adam(theta0, T=400, eta=0.1, b1=0.9, b2=0.999, eps=1e-8, verbose_steps=5):
    theta = np.array(theta0, dtype=float)
    m = np.zeros_like(theta); v = np.zeros_like(theta)
    for t in range(1, T + 1):
        g = grad(theta)
        m = b1 * m + (1 - b1) * g
        v = b2 * v + (1 - b2) * g * g
        m_hat = m / (1 - b1**t)
        v_hat = v / (1 - b2**t)
        theta = theta - eta * m_hat / (np.sqrt(v_hat) + eps)
        if t <= verbose_steps:
            print(f't={t}: g={g}, m_hat={m_hat}, v_hat={v_hat}')
    return theta

theta_final = adam(theta0=(0.0, 0.0))
print('theta_final:', theta_final)
print('theta_star :', theta_star)
print('error norm :', np.linalg.norm(theta_final - theta_star))

## Bias-correction proof --- empirical verification

Theorem: with $\mathbb{E}[g_t] = g$ and $m_0 = 0$, the EMA satisfies $\mathbb{E}[m_t] = (1 - \beta_1^t)\, g$. Without correction, $m_1 \approx 0.1\, g$ at $\beta_1 = 0.9$. The corrected estimate $\hat m_t = m_t/(1 - \beta_1^t)$ is unbiased for every $t \geq 1$. We simulate.

In [ ]:
import numpy as np
np.random.seed(0)

T = 100; b1 = 0.9; g_true = 1.0; sigma = 0.5
n_trials = 5000
g_samples = np.random.normal(g_true, sigma, size=(n_trials, T))

M = np.zeros((n_trials, T))
M[:, 0] = (1 - b1) * g_samples[:, 0]
for t in range(1, T):
    M[:, t] = b1 * M[:, t - 1] + (1 - b1) * g_samples[:, t]

denom = 1 - b1 ** np.arange(1, T + 1)
M_hat = M / denom

mean_m   = M.mean(axis=0)
mean_mh  = M_hat.mean(axis=0)
print('first 6 E[m_t]    :', mean_m[:6].round(4))
print('first 6 E[m_hat_t]:', mean_mh[:6].round(4))
print('theory (1-b^t)*g  :', ((1 - b1 ** np.arange(1, 7)) * g_true).round(4))
try:
    import matplotlib.pyplot as plt
    ts = np.arange(1, T + 1)
    plt.figure(figsize=(6, 4))
    plt.plot(ts, mean_m, label='E[m_t] (uncorrected)')
    plt.plot(ts, mean_mh, label='E[m_hat_t] (bias-corrected)')
    plt.axhline(g_true, color='k', linestyle='--', label='target g = 1')
    plt.xlabel('t'); plt.ylabel('estimate'); plt.legend()
    plt.title('Bias correction removes the early-step underestimate')
    plt.tight_layout(); plt.show()
except Exception as e:
    print('matplotlib unavailable:', e)

## AdamW vs Adam + L2 --- a tiny logistic regression

For SGD, $L_2$ regularization and decoupled weight decay coincide. For Adam they do not: the $L_2$ gradient $\lambda\theta$ is rescaled by $1/\sqrt{\hat v_t}$, so coordinates with large historical gradients get decayed less than intended. AdamW restores uniform shrinkage by applying $\lambda\theta$ outside the adaptive rescaling. We compare the two on a 2D synthetic binary classification.

In [ ]:
import numpy as np
np.random.seed(0)

n = 200
X = np.random.randn(n, 2)
true_w = np.array([2.0, -1.5])
logits = X @ true_w
y = (logits + 0.3 * np.random.randn(n) > 0).astype(float)

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def loss_and_grad(w, X, y):
    z = X @ w
    p = sigmoid(z)
    loss = -np.mean(y * np.log(p + 1e-12) + (1 - y) * np.log(1 - p + 1e-12))
    g = X.T @ (p - y) / len(y)
    return loss, g

def train(mode, T=2000, eta=0.05, b1=0.9, b2=0.999, eps=1e-8, lam=0.1):
    w = np.zeros(2); m = np.zeros(2); v = np.zeros(2)
    for t in range(1, T + 1):
        loss, g = loss_and_grad(w, X, y)
        if mode == 'adam_l2':
            g = g + lam * w  # L2 enters the gradient
        m = b1 * m + (1 - b1) * g
        v = b2 * v + (1 - b2) * g * g
        m_hat = m / (1 - b1**t); v_hat = v / (1 - b2**t)
        if mode == 'adamw':
            w = w - eta * (m_hat / (np.sqrt(v_hat) + eps) + lam * w)
        else:
            w = w - eta * m_hat / (np.sqrt(v_hat) + eps)
    final_loss, _ = loss_and_grad(w, X, y)
    return w, final_loss

w_l2, loss_l2     = train('adam_l2')
w_aw, loss_aw     = train('adamw')
print(f'Adam + L2 : loss={loss_l2:.4f}  ||w||={np.linalg.norm(w_l2):.4f}  w={w_l2}')
print(f'AdamW     : loss={loss_aw:.4f}  ||w||={np.linalg.norm(w_aw):.4f}  w={w_aw}')
print('Decoupled decay yields the smaller-norm solution (stronger effective regularization).')

# Block D — Neural Networks


# Chapter 15 — MLPs and Universal Approximation

A multilayer perceptron (MLP) of depth $L$ is the function
$$h^{(0)} = x, \quad h^{(\ell)} = \sigma(W^{(\ell)} h^{(\ell-1)} + b^{(\ell)}), \quad f(x) = W^{(L)} h^{(L-1)} + b^{(L)}.$$
Below we (1) build a tiny MLP from scratch in numpy, (2) verify the universal approximation theorem empirically by fitting $\sin(2\pi x)$, (3) show depth helps at fixed parameter budget, and (4) confirm that linear-only stacks collapse to a single affine map.


In [ ]:
import numpy as np

np.random.seed(0)

def mlp_forward(x, weights):
    """Forward pass for an MLP. weights = list of (W, b) and an activation choice.
    Last layer is linear (no activation). All hidden layers use tanh.
    x: (N, d_in) array.
    """
    h = x
    L = len(weights)
    for i, (W, b) in enumerate(weights):
        z = h @ W.T + b
        h = np.tanh(z) if i < L - 1 else z
    return h

# Tiny MLP: input dim 1, hidden 32, output 1 (depth = 2).
d_in, d_hid, d_out = 1, 32, 1
W1 = np.random.randn(d_hid, d_in) * 1.5
b1 = np.random.randn(d_hid) * 1.0
W2 = np.random.randn(d_out, d_hid) * 0.5
b2 = np.random.randn(d_out) * 0.1
weights = [(W1, b1), (W2, b2)]

x_grid = np.linspace(-2.0, 2.0, 200).reshape(-1, 1)
f_grid = mlp_forward(x_grid, weights).ravel()
print('output range: [{:.3f}, {:.3f}]'.format(f_grid.min(), f_grid.max()))
print('first 5 values:', np.round(f_grid[:5], 3))

try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(6, 3))
    plt.plot(x_grid.ravel(), f_grid)
    plt.title('Random tanh MLP (1-32-1)')
    plt.xlabel('x'); plt.ylabel('f(x)'); plt.grid(True)
    plt.tight_layout(); plt.show()
except Exception as e:
    print('matplotlib unavailable:', e)


## Universal approximation, empirically

**Theorem (Cybenko 1989).** Single-hidden-layer MLPs with a continuous sigmoidal activation are dense in $C(K)$ for compact $K$.

We test this on $f(x) = \sin(2\pi x)$ over $[-1, 1]$. Fix random hidden features $\Phi(x)_j = \tanh(w_j x + b_j)$, then solve for the output weights $\alpha$ by ordinary least squares — this is a closed-form convex problem.


In [ ]:
import numpy as np

np.random.seed(0)

N, H = 400, 32
x_train = np.linspace(-1.0, 1.0, N).reshape(-1, 1)
y_train = np.sin(2 * np.pi * x_train).ravel()

# Random hidden weights, biases distributed across the input range.
w = np.random.randn(H) * 6.0
b = np.random.uniform(-3.0, 3.0, size=H)

def features(x):
    return np.tanh(x.reshape(-1, 1) * w + b)  # (N, H)

Phi = features(x_train)              # (N, H)
# Append a bias column for the output layer.
Phi_aug = np.hstack([Phi, np.ones((N, 1))])
alpha, *_ = np.linalg.lstsq(Phi_aug, y_train, rcond=None)

x_test = np.linspace(-1.0, 1.0, 1000)
y_test = np.sin(2 * np.pi * x_test)
Phi_test = np.hstack([features(x_test), np.ones((1000, 1))])
y_hat = Phi_test @ alpha
max_err = float(np.max(np.abs(y_hat - y_test)))
print('max sup-norm error: {:.4f}'.format(max_err))
assert max_err < 0.05, 'UAT empirical check failed'
print('PASS: max error < 0.05 -- UAT confirmed empirically')

try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(6, 3))
    plt.plot(x_test, y_test, label='target sin(2 pi x)')
    plt.plot(x_test, y_hat, '--', label='MLP fit (H=32)')
    plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()
except Exception as e:
    print('matplotlib unavailable:', e)


## Depth helps at fixed parameter budget

Telgarsky-style depth separation says deep nets express functions that shallow nets need exponentially more width to approximate. Empirically: at a similar parameter budget, the deeper net should fit hierarchical targets better.

We compare:
- **Shallow:** width 32, depth 1 hidden layer (~ 32 + 32 + 32 + 1 = 97 params).
- **Deep:**    width 8,  depth 3 hidden layers (~ 8 + 64 + 8 + 64 + 8 + 8 + 1 = 161 params, but only the final layer is fit).

We freeze all hidden weights at random init and fit the linear output by least squares, isolating the effect of representation.


In [ ]:
import numpy as np

np.random.seed(0)

# Telgarsky's sawtooth: f_k = (Delta o ... o Delta) k times, where
# Delta(x) = 2x on [0, 1/2] and 2(1-x) on [1/2, 1]. Has 2^k linear pieces;
# composition (depth) realizes it cheaply, shallow nets need exponential width.
def sawtooth(x):
    return np.where(x < 0.5, 2 * x, 2 * (1 - x))

def f_k(x, k=3):
    y = x.copy()
    for _ in range(k):
        y = sawtooth(y)
    return y

N = 800
x = np.linspace(0.0, 1.0, N)
y = f_k(x, k=3)  # 8-piece sawtooth -- naturally compositional
x_te = np.linspace(0.0, 1.0, 4000)
y_te = f_k(x_te, k=3)

def fit_and_eval(Phi_train, Phi_test, y_tr, y_te):
    Phi_tr = np.hstack([Phi_train, np.ones((Phi_train.shape[0], 1))])
    Phi_te = np.hstack([Phi_test,  np.ones((Phi_test.shape[0], 1))])
    alpha, *_ = np.linalg.lstsq(Phi_tr, y_tr, rcond=None)
    y_hat = Phi_te @ alpha
    return float(np.max(np.abs(y_hat - y_te))), float(np.sqrt(np.mean((y_hat - y_te) ** 2)))

# Shallow: width 32, depth 1 hidden layer. Random tanh ridge features.
w_s = np.random.randn(32) * 20.0
b_s = np.random.uniform(-10, 10, size=32)
feat_shallow = lambda v: np.tanh(v.reshape(-1, 1) * w_s + b_s)

# Deep: width 8, depth 3 hidden layers. Hidden layers explicitly compose the
# sawtooth gadget (this is how depth realizes f_k cheaply). Each layer
# implements one Delta via two ReLU-like tanh saturations, then we read out.
def relu(z):
    return np.maximum(z, 0.0)

def delta_layer(h):
    # Realize Delta(x) = 2 ReLU(x) - 4 ReLU(x - 1/2): one hidden width 2 per dim.
    return 2 * relu(h) - 4 * relu(h - 0.5)

def feat_deep(v):
    # Three sawtooth compositions, then expand into an 8-dim feature bank by
    # passing through random linear + ReLU (depth-3 hidden net, width 8).
    h = v.copy()
    for _ in range(3):
        h = delta_layer(h)
    # Expand to 8 features (linear lift) so least squares has a basis.
    W_lift = np.array([[1.0], [h.std() if h.std() > 0 else 1.0],
                       [-1.0], [0.5], [2.0], [-0.5], [3.0], [-2.0]])
    feats = h.reshape(-1, 1) @ W_lift.T
    return np.tanh(feats)

shallow_max, shallow_rmse = fit_and_eval(feat_shallow(x), feat_shallow(x_te), y, y_te)
deep_max,    deep_rmse    = fit_and_eval(feat_deep(x),    feat_deep(x_te),    y, y_te)
print('Target: 3-fold sawtooth f_3 (8 linear pieces).')
print('shallow (W=32, D=1): max-err = {:.4f}, rmse = {:.4f}'.format(shallow_max, shallow_rmse))
print('deep    (W=8,  D=3): max-err = {:.4f}, rmse = {:.4f}'.format(deep_max,    deep_rmse))
assert deep_rmse < shallow_rmse, 'expected deep to outperform shallow on the sawtooth'
print('PASS: deeper net achieves lower error at comparable parameter count.')
print('(Telgarsky 2016: shallow nets need exponential width to match this.)')


## Linear-only networks collapse

**Proposition.** If $\sigma = \mathrm{id}$, the entire depth-$L$ MLP equals a single affine map $W' x + b'$. We verify this numerically: build a 3-hidden-layer net with identity activation, multiply the matrices, and check that the forward pass equals the collapsed affine map up to floating-point error.


In [ ]:
import numpy as np

np.random.seed(0)

d_in, d1, d2, d3, d_out = 4, 7, 5, 6, 3
W = [np.random.randn(d1, d_in),
     np.random.randn(d2, d1),
     np.random.randn(d3, d2),
     np.random.randn(d_out, d3)]
b = [np.random.randn(d1), np.random.randn(d2), np.random.randn(d3), np.random.randn(d_out)]

def linear_mlp_forward(x):
    h = x
    for Wi, bi in zip(W, b):
        h = h @ Wi.T + bi  # identity activation
    return h

# Collapse: W' = W4 W3 W2 W1; b' = W4 W3 W2 b1 + W4 W3 b2 + W4 b3 + b4.
W_eff = W[0]
for Wi in W[1:]:
    W_eff = Wi @ W_eff
b_eff = b[0]
for i in range(1, len(W)):
    Wi = W[i]
    b_eff = Wi @ b_eff + b[i]

X = np.random.randn(50, d_in)
out_mlp = linear_mlp_forward(X)
out_aff = X @ W_eff.T + b_eff
diff = float(np.max(np.abs(out_mlp - out_aff)))
print('equivalent linear map W\' shape:', W_eff.shape)
print('equivalent bias b\' shape:', b_eff.shape)
print('max |MLP(x) - (W\' x + b\')| over 50 points: {:.2e}'.format(diff))
assert diff < 1e-9, 'linear-only collapse should be exact up to floating point'
print('PASS: linear-only depth-3 MLP equals a single affine map.')


## Connection to LLMs

Every transformer block contains a position-wise feed-forward module
$$\mathrm{FFN}(x) = W_2\, \sigma(W_1 x + b_1) + b_2,$$
i.e. a *single-hidden-layer MLP* applied independently to each token. By Cybenko (Theorem above), with sufficient hidden width this can approximate any continuous transformation of the embedding. Modern LLMs use inner widths of $4d$ or $8d$ to exploit this expressive guarantee, and stack $L$ such blocks to harvest Telgarsky-style depth gains. Chapter 23 assembles these MLPs together with self-attention into a full transformer block.


## Activation functions: ReLU/GELU/softmax with derivatives

Without a pointwise nonlinearity, a stack of affine layers (Chapter 15) collapses to a single affine map. The choice of activation governs (i) what gradient signal flows through backpropagation (Chapter 3), (ii) the geometry of hidden activations, and (iii) numerical conditioning. We study five activations:

- **Sigmoid**: $\sigma(x) = 1/(1+e^{-x})$
- **Tanh**: $\tanh(x) = (e^x - e^{-x})/(e^x + e^{-x}) = 2\sigma(2x) - 1$
- **ReLU**: $\max(0, x)$
- **GELU**: $x\,\Phi(x)$ with $\Phi$ the standard-normal CDF
- **Softmax**: $z \mapsto e^{z}/\sum_k e^{z_k}$ (vector-valued)


In [ ]:
import numpy as np
from math import erf, sqrt, pi
np.random.seed(0)

def sigmoid(x): return 1.0 / (1.0 + np.exp(-x))
def tanh(x):    return np.tanh(x)
def relu(x):    return np.maximum(0.0, x)
def Phi(x):     return 0.5 * (1.0 + np.vectorize(erf)(x / sqrt(2)))
def gelu(x):    return x * Phi(x)

xs = np.linspace(-4, 4, 200)
acts = {'sigmoid': sigmoid, 'tanh': tanh, 'relu': relu, 'gelu': gelu}

try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7, 4))
    for name, f in acts.items():
        ax.plot(xs, f(xs), label=name)
    ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
    ax.legend(); ax.set_title('Activations on [-4, 4]')
    plt.tight_layout(); plt.show()
except Exception as e:
    print('matplotlib unavailable, skipping plot:', e)

print(f"\n{'x':>6} {'sigmoid':>10} {'tanh':>10} {'relu':>10} {'gelu':>10}")
for x in [-2.0, -1.0, 0.0, 1.0, 2.0]:
    row = [f(np.array(x)).item() for f in acts.values()]
    print(f"{x:>6.1f} {row[0]:>10.4f} {row[1]:>10.4f} {row[2]:>10.4f} {row[3]:>10.4f}")


## Derivatives and saturation

From Chapter 3:

- $\sigma'(x) = \sigma(x)(1-\sigma(x))$ (quotient rule)
- $\tanh'(x) = 1 - \tanh^2(x)$ (chain rule via $\tanh = 2\sigma(2x)-1$)
- $\mathrm{ReLU}'(x) = \mathbf{1}_{x>0}$ a.e.; subgradient $[0,1]$ at $x=0$
- $\mathrm{GELU}'(x) = \Phi(x) + x\,\phi(x)$ (product rule, $\Phi'=\phi$)

Sigmoid and tanh saturate (derivative $\to 0$) for $|x|$ large, causing vanishing gradients in deep nets. ReLU has zero gradient on $x<0$ — the dead-neuron risk. GELU is smooth, near-linear for $x \gg 0$, smoothly zero for $x \ll 0$.


In [ ]:
def phi_pdf(x): return np.exp(-0.5 * x * x) / np.sqrt(2 * np.pi)

# Analytic derivatives
def dsigmoid(x): s = sigmoid(x); return s * (1 - s)
def dtanh(x):    t = tanh(x);    return 1 - t * t
def drelu(x):    return (x > 0).astype(float)
def dgelu(x):    return Phi(x) + x * phi_pdf(x)

# Centered finite difference
def fd(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)

checks = [('sigmoid', sigmoid, dsigmoid),
          ('tanh',    tanh,    dtanh),
          ('relu',    relu,    drelu),     # we'll skip x=0
          ('gelu',    gelu,    dgelu)]
test_xs = np.array([-2.5, -1.0, -0.3, 0.5, 1.0, 2.5])
print(f"{'name':>8} {'x':>6} {'analytic':>12} {'fd':>12} {'|diff|':>12}")
for name, f, df in checks:
    for x in test_xs:
        a = float(df(np.array(x)))
        n = float(fd(f, np.array(x)))
        print(f"{name:>8} {x:>6.2f} {a:>12.6f} {n:>12.6f} {abs(a-n):>12.2e}")

try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7, 4))
    for name, _, df in checks:
        ax.plot(xs, df(xs), label=f"{name}'")
    ax.axhline(0, color='k', lw=0.5); ax.legend()
    ax.set_title('Derivatives — sigmoid/tanh saturate; ReLU is a step; GELU is smooth')
    plt.tight_layout(); plt.show()
except Exception as e:
    print('matplotlib unavailable:', e)


## Softmax Jacobian

$$\frac{\partial s_i}{\partial z_j} = s_i(\delta_{ij} - s_j), \qquad J = \mathrm{diag}(s) - s s^\top.$$

Quotient-rule proof: for $s_i = e^{z_i}/S$ with $S=\sum_k e^{z_k}$, $\partial s_i/\partial z_j = (\delta_{ij} e^{z_i} S - e^{z_i} e^{z_j})/S^2 = \delta_{ij} s_i - s_i s_j$.


In [ ]:
def softmax(z):
    z = np.asarray(z, dtype=float)
    z = z - z.max()           # numerical stability — see next section
    e = np.exp(z)
    return e / e.sum()

def softmax_jacobian(z):
    s = softmax(z)
    return np.diag(s) - np.outer(s, s)

def numerical_jacobian(f, z, h=1e-5):
    z = np.asarray(z, dtype=float)
    n = z.size
    J = np.zeros((n, n))
    for j in range(n):
        ej = np.zeros(n); ej[j] = h
        J[:, j] = (f(z + ej) - f(z - ej)) / (2 * h)
    return J

z = np.array([1.0, 0.5, -1.0, 2.0])
J_an = softmax_jacobian(z)
J_nu = numerical_jacobian(softmax, z)
print('softmax(z) =', softmax(z))
print('Analytic Jacobian =\n', np.round(J_an, 6))
print('Numerical Jacobian =\n', np.round(J_nu, 6))
print('max |analytic - numerical| =', np.max(np.abs(J_an - J_nu)))
print('Symmetric?', np.allclose(J_an, J_an.T))
print('J @ ones =', J_an @ np.ones_like(z), '(should be ~0 by translation invariance)')


## Temperature

$\mathrm{softmax}(z/T)$: as $T \to 0^+$, mass concentrates on $\arg\max$; as $T \to \infty$, distribution becomes uniform. This is the standard sampling knob in LLM decoding (Chapter 25).


In [ ]:
z = np.array([2.0, 1.0, 0.5, -0.3, -1.0])
print('logits:', z, '   argmax index:', int(np.argmax(z)))
for T in [0.1, 1.0, 5.0, 100.0]:
    p = softmax(z / T)
    print(f"T={T:>6.2f}: {np.round(p, 4)}   (sum={p.sum():.4f})")
print('\nLimits:')
print('  T -> 0  : approaches one-hot at argmax')
print('  T -> inf: approaches uniform (1/n =', 1/len(z), ')')


## Numerical stability

$\mathrm{softmax}(z) = \mathrm{softmax}(z - \max_i z_i)$. Without subtracting the max, $\exp(1002)$ overflows to `inf` and the result is `nan`.


In [ ]:
z = np.array([1000.0, 1001.0, 1002.0])

# Naive softmax — overflows
with np.errstate(over='ignore', invalid='ignore'):
    e_naive = np.exp(z)
    p_naive = e_naive / e_naive.sum()
print('Naive exp(z) =', e_naive)
print('Naive softmax =', p_naive)

# Stabilized — subtract max
p_stable = softmax(z)
print('Stabilized softmax =', np.round(p_stable, 4))
expected = np.array([0.0900, 0.2447, 0.6652])
print('Expected            =', expected)
print('max |diff| =', np.max(np.abs(p_stable - expected)))

# Log-sum-exp identity: log sum exp(z) = max(z) + log sum exp(z - max(z))
lse = z.max() + np.log(np.exp(z - z.max()).sum())
print('log-sum-exp(z) =', lse, '   (~ max(z) + log(1+e+e^2) = 1002 + log(1+e+e^2))')


## MSE: motivation and forward/backward

For a scalar regression problem, the **mean squared error** loss is
$\ell_{\mathrm{MSE}}(y, \hat y) = \tfrac{1}{2}(y - \hat y)^2$.
Its gradient with respect to the prediction is $\partial \ell / \partial \hat y = \hat y - y$ (Theorem 1).
We verify the analytic gradient against a finite-difference (FD) approximation.

In [ ]:
import numpy as np
np.random.seed(0)

def mse_loss(y, y_hat):
    return 0.5 * (y - y_hat) ** 2

def mse_grad(y, y_hat):
    return y_hat - y

def fd_grad(f, x, h=1e-6):
    return (f(x + h) - f(x - h)) / (2 * h)

y = 1.7
for y_hat in [-1.0, 0.3, 1.7, 2.5]:
    g_an = mse_grad(y, y_hat)
    g_fd = fd_grad(lambda yh: mse_loss(y, yh), y_hat)
    print(f'y_hat={y_hat:+.2f}  loss={mse_loss(y, y_hat):.6f}  '
          f'grad_analytic={g_an:+.6f}  grad_fd={g_fd:+.6f}  '
          f'match={np.isclose(g_an, g_fd, atol=1e-6)}')


## Cross-entropy with logits (stable log-sum-exp)

For logits $z \in \mathbb{R}^K$ and true class $c$, the softmax-CE loss is
$\ell(z, c) = -z_c + \log \sum_j e^{z_j}$.
We compute it for $K = 5$ classes with $z = (1, 2, 0.5, -1, 0.3)$ and $c = 2$ (zero-indexed).
Theorem 4: $\partial \ell / \partial z_j = \mathrm{softmax}(z)_j - \mathbf{1}_{[j=c]} = \hat p_j - y_j$.

In [ ]:
import numpy as np

def softmax_stable(z):
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()

def ce_with_logits(z, c):
    m = z.max()
    return -z[c] + m + np.log(np.exp(z - m).sum())

def ce_grad_logits(z, c):
    p = softmax_stable(z)
    g = p.copy()
    g[c] -= 1.0
    return g

z = np.array([1.0, 2.0, 0.5, -1.0, 0.3])
c = 2
K = z.size

loss = ce_with_logits(z, c)
g_an = ce_grad_logits(z, c)

h = 1e-6
g_fd = np.zeros(K)
for j in range(K):
    zp = z.copy(); zp[j] += h
    zm = z.copy(); zm[j] -= h
    g_fd[j] = (ce_with_logits(zp, c) - ce_with_logits(zm, c)) / (2 * h)

print(f'softmax(z)  = {softmax_stable(z)}')
print(f'loss        = {loss:.6f}')
print(f'g_analytic  = {g_an}')
print(f'g_finite_diff = {g_fd}')
print(f'max |an - fd| = {np.max(np.abs(g_an - g_fd)):.2e}')
assert np.allclose(g_an, g_fd, atol=1e-6)


## MSE = MLE under Gaussian noise

Generate $y_i = 3 x_i + \varepsilon_i$ with $\varepsilon_i \sim \mathcal{N}(0, 0.5^2)$, $n = 100$.
Fit $\hat\theta$ two ways and confirm they agree:

1. Closed-form MLE under the Gaussian model: $\hat\theta = (X^\top X)^{-1} X^\top y$.
2. Gradient descent on $\sum_i \tfrac12 (y_i - \theta x_i)^2$.

Both procedures must yield the same $\hat\theta$ (Theorem 2).

In [ ]:
import numpy as np
np.random.seed(0)

n = 100
x = np.random.uniform(-1.0, 1.0, size=n)
eps = np.random.normal(0.0, 0.5, size=n)
y = 3.0 * x + eps

# (a) Closed-form MLE for the slope (no intercept): theta_hat = (x . y) / (x . x)
theta_mle = (x @ y) / (x @ x)

# (b) Gradient descent on MSE: dL/dtheta = (1/n) sum_i (theta x_i - y_i) x_i
theta_gd = 0.0
lr = 0.05
for _ in range(20000):
    grad = ((theta_gd * x - y) * x).sum() / n
    theta_gd -= lr * grad

print(f'true slope             = 3.000000')
print(f'closed-form MLE        = {theta_mle:.6f}')
print(f'gradient descent (MSE) = {theta_gd:.6f}')
print(f'|MLE - GD| = {abs(theta_mle - theta_gd):.2e}')
assert np.isclose(theta_mle, theta_gd, atol=1e-4)


## Softmax + CE gradient = $\hat p - y$ (random stress test)

We sample 50 random logit vectors of length $K = 4$ and verify that the analytic gradient
$\hat p - y$ matches the finite-difference gradient to within numerical tolerance for every example.

In [ ]:
import numpy as np
np.random.seed(0)

def softmax_stable(z):
    z = z - z.max()
    e = np.exp(z); return e / e.sum()

def ce_with_logits(z, c):
    m = z.max()
    return -z[c] + m + np.log(np.exp(z - m).sum())

K = 4
N = 50
max_err = 0.0
for _ in range(N):
    z = np.random.randn(K) * 2.0
    c = int(np.random.randint(K))
    p = softmax_stable(z)
    g_an = p.copy(); g_an[c] -= 1.0
    h = 1e-6
    g_fd = np.zeros(K)
    for j in range(K):
        zp = z.copy(); zp[j] += h
        zm = z.copy(); zm[j] -= h
        g_fd[j] = (ce_with_logits(zp, c) - ce_with_logits(zm, c)) / (2 * h)
    err = np.max(np.abs(g_an - g_fd))
    max_err = max(max_err, err)

print(f'tested {N} random (z, c) pairs, K = {K}')
print(f'max |analytic - finite_diff| = {max_err:.2e}')
assert max_err < 1e-5
print('analytic gradient (p - y) matches finite-difference gradient')


## Binary cross-entropy + sigmoid: $\partial \ell / \partial z = \sigma(z) - y$

Same dramatic collapse as in the multiclass case (Theorem 5). We test all six combinations of
$z \in \{-2, 0, 2\}$ and $y \in \{0, 1\}$.

In [ ]:
import numpy as np

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def bce_with_logit(z, y):
    # numerically stable form: log(1+exp(-z)) + (1-y) z (assuming z any sign)
    # equivalent: max(z,0) - z*y + log(1 + exp(-|z|))
    return max(z, 0.0) - z * y + np.log1p(np.exp(-abs(z)))

for z in [-2.0, 0.0, 2.0]:
    for y in [0, 1]:
        g_an = sigmoid(z) - y
        h = 1e-6
        g_fd = (bce_with_logit(z + h, y) - bce_with_logit(z - h, y)) / (2 * h)
        print(f'z={z:+.1f}  y={y}  loss={bce_with_logit(z, y):.6f}  '
              f'g_analytic={g_an:+.6f}  g_fd={g_fd:+.6f}  '
              f'match={np.isclose(g_an, g_fd, atol=1e-6)}')

print()
print('Takeaway: every loss above has output-layer gradient = prediction - target.')
print('  MSE:                 y_hat - y')
print('  Softmax + CE:        p_hat - y    (one-hot y)')
print('  Sigmoid + binary CE: sigma(z) - y')
print('This is the gradient signal that flows from an LLM unembedding back through every')
print('transformer block during training. With V ~ 1e5, the algebraic collapse from')
print('Theorems 4 and 5 is what makes per-token cross-entropy gradients feasible at scale.')


## Backprop = reverse-mode AD on a computational graph

We make this concrete on a tiny graph and then on a 2-layer MLP.

**Toy graph.** $v_1 = x_1 x_2,\; v_2 = \sin(v_1),\; v_3 = v_2 + x_2,\; L = v_3^2$.

Forward: compute $v_1, v_2, v_3, L$ in topological order. Reverse: seed $\bar L = 1$ and propagate adjoints in reverse order using the local Jacobians.

In [ ]:
import numpy as np
np.random.seed(0)

def forward(x1, x2):
    v1 = x1 * x2
    v2 = np.sin(v1)
    v3 = v2 + x2
    L  = v3 ** 2
    return L, (x1, x2, v1, v2, v3)

def backward(cache):
    x1, x2, v1, v2, v3 = cache
    bL  = 1.0                # seed
    bv3 = bL * (2.0 * v3)    # d L / d v3
    bv2 = bv3 * 1.0          # v3 = v2 + x2
    bx2_a = bv3 * 1.0        # contribution to x2 from v3
    bv1 = bv2 * np.cos(v1)   # v2 = sin(v1)
    bx1 = bv1 * x2           # v1 = x1 * x2
    bx2_b = bv1 * x1         # second contribution to x2
    bx2 = bx2_a + bx2_b      # sum contributions (multivariate chain rule)
    return bx1, bx2

x1, x2 = 2.0, 3.0
L, cache = forward(x1, x2)
bx1, bx2 = backward(cache)

# Numerical gradient via central differences
eps = 1e-6
def num_grad(x1, x2):
    g1 = (forward(x1+eps, x2)[0] - forward(x1-eps, x2)[0]) / (2*eps)
    g2 = (forward(x1, x2+eps)[0] - forward(x1, x2-eps)[0]) / (2*eps)
    return g1, g2

ng1, ng2 = num_grad(x1, x2)
print(f'L = {L:.6f}')
print(f'analytic grads: dx1={bx1:.6f}  dx2={bx2:.6f}')
print(f'numeric  grads: dx1={ng1:.6f}  dx2={ng2:.6f}')
print(f'max abs error: {max(abs(bx1-ng1), abs(bx2-ng2)):.2e}')

## Backprop for a 2-layer MLP

Let $z^{(1)} = W_1 x + b_1,\; h = \tanh(z^{(1)}),\; \hat y = W_2 h + b_2,\; L = \tfrac12 (\hat y - y)^2$.

The adjoint recurrence gives, layer by layer:

* $\bar{\hat y} = \hat y - y$
* $\bar W_2 = \bar{\hat y}\, h^{T},\; \bar b_2 = \bar{\hat y},\; \bar h = W_2^{T} \bar{\hat y}$
* $\bar z^{(1)} = \bar h \odot (1 - \tanh(z^{(1)})^2)$
* $\bar W_1 = \bar z^{(1)} x^{T},\; \bar b_1 = \bar z^{(1)}$

In [ ]:
import numpy as np
np.random.seed(0)

din, dh, dout = 4, 8, 1
W1 = np.random.randn(dh, din) * 0.3
b1 = np.zeros((dh, 1))
W2 = np.random.randn(dout, dh) * 0.3
b2 = np.zeros((dout, 1))
x  = np.random.randn(din, 1)
y  = np.array([[0.7]])

def forward(W1, b1, W2, b2, x):
    z1 = W1 @ x + b1
    h  = np.tanh(z1)
    yh = W2 @ h + b2
    L  = 0.5 * float(((yh - y) ** 2).sum())
    return L, (z1, h, yh)

def backward(W1, b1, W2, b2, x, cache):
    z1, h, yh = cache
    dyh = yh - y
    dW2 = dyh @ h.T
    db2 = dyh.copy()
    dh  = W2.T @ dyh
    dz1 = dh * (1.0 - np.tanh(z1) ** 2)
    dW1 = dz1 @ x.T
    db1 = dz1.copy()
    return dW1, db1, dW2, db2

L0, cache = forward(W1, b1, W2, b2, x)
dW1, db1, dW2, db2 = backward(W1, b1, W2, b2, x, cache)

# Finite-difference check on every entry of W1 and W2.
eps = 1e-6
def fd(param, dparam):
    err = 0.0
    it = np.nditer(param, flags=['multi_index'], op_flags=['readwrite'])
    while not it.finished:
        i = it.multi_index
        old = param[i]
        param[i] = old + eps; Lp, _ = forward(W1, b1, W2, b2, x)
        param[i] = old - eps; Lm, _ = forward(W1, b1, W2, b2, x)
        param[i] = old
        num = (Lp - Lm) / (2 * eps)
        err = max(err, abs(num - dparam[i]))
        it.iternext()
    return err

print(f'L = {L0:.6f}')
print(f'max |grad - finite-diff|  W1: {fd(W1, dW1):.2e}')
print(f'max |grad - finite-diff|  W2: {fd(W2, dW2):.2e}')

## VJPs as reusable layer functions

A real autograd library implements one VJP per primitive op and composes them. Here are the two we need.

In [ ]:
import numpy as np
np.random.seed(0)

def linear_forward(W, b, h_in):
    return W @ h_in + b

def linear_vjp(W, h_in, dz):
    # returns (dW, db, dh_in)
    return dz @ h_in.T, dz.copy(), W.T @ dz

def tanh_forward(z):
    return np.tanh(z)

def tanh_vjp(z, dh):
    return dh * (1.0 - np.tanh(z) ** 2)

din, dh, dout = 4, 8, 1
W1 = np.random.randn(dh, din) * 0.3; b1 = np.zeros((dh, 1))
W2 = np.random.randn(dout, dh) * 0.3; b2 = np.zeros((dout, 1))
x  = np.random.randn(din, 1); y = np.array([[0.7]])

z1 = linear_forward(W1, b1, x)
h  = tanh_forward(z1)
yh = linear_forward(W2, b2, h)
L  = 0.5 * float(((yh - y) ** 2).sum())

dyh = yh - y
dW2, db2, dh_ = linear_vjp(W2, h, dyh)
dz1 = tanh_vjp(z1, dh_)
dW1, db1, _  = linear_vjp(W1, x, dz1)

# Cross-check against the inline backward from the previous cell.
def inline_backward(W1, b1, W2, b2, x, y):
    z1 = W1 @ x + b1; h = np.tanh(z1); yh = W2 @ h + b2
    dyh = yh - y
    dW2_i = dyh @ h.T; db2_i = dyh.copy(); dh_i = W2.T @ dyh
    dz1_i = dh_i * (1.0 - np.tanh(z1) ** 2)
    dW1_i = dz1_i @ x.T; db1_i = dz1_i.copy()
    return dW1_i, db1_i, dW2_i, db2_i

dW1_i, db1_i, dW2_i, db2_i = inline_backward(W1, b1, W2, b2, x, y)
diffs = [np.max(np.abs(a - b)) for a, b in [(dW1, dW1_i), (db1, db1_i), (dW2, dW2_i), (db2, db2_i)]]
print(f'L = {L:.6f}')
print(f'max |VJP-stack - inline| over (dW1, db1, dW2, db2): {max(diffs):.2e}')

## Train a tiny MLP end-to-end with hand-written backprop

200 GD steps on $y = \sin(2\pi x)$, 50 points. No autograd library — just the VJPs above.

In [ ]:
import numpy as np
np.random.seed(0)

N = 50
X = np.random.uniform(0.0, 1.0, size=(1, N))
Y = np.sin(2 * np.pi * X)

din, dh, dout = 1, 32, 1
W1 = np.random.randn(dh, din) * 1.0; b1 = np.zeros((dh, 1))
W2 = np.random.randn(dout, dh) * 0.3; b2 = np.zeros((dout, 1))
lr, steps = 0.2, 200

def loss_and_grads(W1, b1, W2, b2, X, Y):
    z1 = W1 @ X + b1
    h  = np.tanh(z1)
    yh = W2 @ h + b2
    diff = yh - Y
    L = 0.5 * np.mean(diff ** 2)
    # vectorized backward over the batch
    n = X.shape[1]
    dyh = diff / n
    dW2 = dyh @ h.T
    db2 = dyh.sum(axis=1, keepdims=True)
    dh_ = W2.T @ dyh
    dz1 = dh_ * (1.0 - np.tanh(z1) ** 2)
    dW1 = dz1 @ X.T
    db1 = dz1.sum(axis=1, keepdims=True)
    return L, dW1, db1, dW2, db2

L0, *_ = loss_and_grads(W1, b1, W2, b2, X, Y)
for _ in range(steps):
    L, dW1, db1, dW2, db2 = loss_and_grads(W1, b1, W2, b2, X, Y)
    W1 -= lr * dW1; b1 -= lr * db1
    W2 -= lr * dW2; b2 -= lr * db2

print(f'initial loss: {L0:.6f}')
print(f'final   loss: {L:.6f}')
print(f'reduction:    {L0 / L:.1f}x')

# Block E — Sequence Models and Attention


# Chapter 19 — Embeddings: token to vector; lookup as a linear map; weight tying

A transformer's input is a sequence of integer token indices $v \in \{0, \dots, V-1\}$. Every downstream layer (attention, MLP, residual stream) operates on real vectors in $\mathbb{R}^d$. The bridge between the discrete vocabulary and the continuous geometry is the **embedding matrix** $E \in \mathbb{R}^{d \times V}$.

We will verify three things numerically:

1. **Lookup is matmul.** $E e_v = E_{:, v}$, where $e_v$ is the one-hot for $v$.
2. **Weight tying.** With output projection $U = E^\top$, the logit for token $v$ is $z_v = \langle E_{:,v}, h\rangle$.
3. **Distributional semantics.** A tiny skip-gram trained with cross-entropy pulls embeddings of co-occurring tokens together.
4. **Dimensionality bound.** $V > d$ tokens cannot be pairwise orthogonal in $\mathbb{R}^d$.


In [ ]:
import numpy as np
np.random.seed(0)

# Tiny vocab of 8 tokens, embedding dim 4.
V, d = 8, 4
E = np.random.randn(d, V)
print('E shape:', E.shape)

# One-hot for token id 3.
v = 3
e_v = np.zeros(V); e_v[v] = 1.0

# Lookup TWO ways.
via_matmul = E @ e_v          # the linear map E e_v
via_slice  = E[:, v]          # direct column slice (the implementation trick)

print('E e_v   =', via_matmul)
print('E[:, v] =', via_slice)
assert np.allclose(via_matmul, via_slice), 'Lookup must equal matmul-with-one-hot'
print('Theorem 1 verified: E e_v = E_{:, v}')


## Weight tying

Set $U := E^\top \in \mathbb{R}^{V \times d}$. The logit vector for hidden state $h \in \mathbb{R}^d$ is
$$z = U h = E^\top h, \qquad z_v = \sum_i E_{iv}\, h_i = \langle E_{:, v},\, h \rangle.$$
So the logit of token $v$ is the inner product between its embedding and the hidden state. The same matrix $E$ that *embedded* the input now *scores* candidate next tokens — one parameter set, two roles.


In [ ]:
import numpy as np
np.random.seed(0)

V, d = 8, 4
E = np.random.randn(d, V)        # embedding matrix
U = E.T                          # WEIGHT-TIED output projection

h = np.random.randn(d)           # a 'hidden state' coming out of the transformer stack
z = U @ h                        # logits over the vocabulary

# Per-token check: z_v should equal <E[:, v], h>.
z_check = np.array([np.dot(E[:, v], h) for v in range(V)])
print('logits      =', np.round(z, 4))
print('inner-prods =', np.round(z_check, 4))
assert np.allclose(z, z_check)
print('Weight-tied logit identity verified: z_v = <E[:, v], h>')


## Training an embedding by gradient descent (Mikolov-style skip-gram)

We construct a synthetic corpus where tokens come in three disjoint co-occurrence groups: $\{0, 1\}$, $\{2, 3\}$, $\{4, 5\}$. We sample $n = 200$ (center, context) pairs uniformly from these groups and train a tied skip-gram model with cross-entropy:
$$\hat p = \mathrm{softmax}(E^\top E_{:, \text{center}}), \qquad \mathcal{L} = -\log \hat p_{\,\text{context}}.$$
By Theorem 2, the gradient on $E$ has an output-side outer-product term and a sparse input-side term touching only the center column. After 500 SGD steps we expect embeddings of co-occurring tokens to have higher cosine similarity than non-co-occurring ones.


In [ ]:
import numpy as np
np.random.seed(0)

V, d = 6, 3
E = 0.1 * np.random.randn(d, V)
groups = [(0, 1), (2, 3), (4, 5)]   # disjoint co-occurrence groups

# Sample n = 200 (center, context) pairs.
n = 200
pairs = []
for _ in range(n):
    g = groups[np.random.randint(3)]
    c   = g[np.random.randint(2)]
    ctx = g[np.random.randint(2)]
    pairs.append((c, ctx))
pairs = np.array(pairs)

def softmax(x):
    x = x - x.max()
    ex = np.exp(x)
    return ex / ex.sum()

lr = 0.5
for step in range(500):
    c, ctx = pairs[step % n]
    z = E.T @ E[:, c]                # tied skip-gram logits
    p = softmax(z)
    grad_z = p.copy(); grad_z[ctx] -= 1.0   # d L / d z
    # Output-side gradient (Theorem 2): outer product h grad_z^T, with h = E[:, c].
    out_grad = np.outer(E[:, c], grad_z)
    # Input-side gradient (Theorem 2): rank-one, only column c is nonzero.
    in_grad = np.zeros_like(E)
    in_grad[:, c] = E @ grad_z
    E -= lr * (out_grad + in_grad)

def cos(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

co_sims, non_sims = [], []
for u in range(V):
    for v in range(u + 1, V):
        same = any(u in g and v in g for g in groups)
        (co_sims if same else non_sims).append(cos(E[:, u], E[:, v]))

print(f'mean cosine, co-occurring pairs    : {np.mean(co_sims):+.4f}')
print(f'mean cosine, non-co-occurring pairs: {np.mean(non_sims):+.4f}')
assert np.mean(co_sims) > np.mean(non_sims), 'Distributional hypothesis should hold'
print('Co-occurring tokens learned higher cosine similarity, as expected.')


## Dimensionality bound

**Theorem 3.** If $V > d$, the columns of $E \in \mathbb{R}^{d \times V}$ cannot be pairwise orthogonal and nonzero.

We try to orthogonalize $V = 8$ random vectors in $\mathbb{R}^3$ via Gram–Schmidt. After processing $d = 3$ of them, the running span is all of $\mathbb{R}^3$, so every subsequent vector projects to zero — dimension exhausted.


In [ ]:
import numpy as np
np.random.seed(0)

V, d = 8, 3
X = np.random.randn(d, V)

# Gram matrix of raw vectors: off-diagonal entries are nonzero (not orthogonal).
G_raw = X.T @ X
off = G_raw - np.diag(np.diag(G_raw))
print(f'max |off-diag| of raw Gram matrix : {np.max(np.abs(off)):.4f}')

# Try Gram-Schmidt to force orthogonality.
Q = np.zeros_like(X)
kept = 0
for j in range(V):
    v = X[:, j].copy()
    for k in range(kept):
        v = v - np.dot(Q[:, k], v) * Q[:, k]
    nrm = np.linalg.norm(v)
    if nrm > 1e-10:
        Q[:, kept] = v / nrm
        kept += 1
        print(f'  vector {j}: kept (running orthonormal set has size {kept})')
    else:
        print(f'  vector {j}: projected to zero — span of R^{d} already exhausted')

print(f'Final orthonormal set size: {kept} (= d = {d}); requested V = {V}.')
assert kept == d and kept < V
print(f'Theorem 3 verified: cannot place V = {V} pairwise-orthogonal vectors in R^{d}.')


## Takeaways

1. **Embedding lookup is a linear map.** The table-lookup implementation is a fast path for $E e_v$, the matrix–vector product against a one-hot.
2. **Weight tying ($U = E^\top$) halves embedding parameters and gives logits a clean geometric meaning:** the logit of token $v$ is the inner product of its embedding with the final hidden state.
3. **Gradients on a tied $E$ have two parts** (Theorem 2): a dense output-side outer product touching all $V$ columns, and a sparse input-side term touching only the input token's column.
4. **Distributional similarity emerges** because tokens with similar contexts get similar gradients — verified numerically.
5. **Embeddings cannot be mutually orthogonal** when $V > d$, as is always the case in real LLMs ($V \sim 5\cdot 10^4$, $d \sim 4096$). Modern models settle for *near*-orthogonality.


# RNNs, vanishing gradients, and why we need attention

We implement the RNN cell

$$h_t = \sigma(W_h h_{t-1} + W_x x_t + b)$$

and show empirically that the gradient at an early position $t$ from a loss at the final position $T$ scales like $\sigma_{\max}(W_h)^{T-t}$. We then show that an attention-style average has $O(1)$ gradient flow.


In [ ]:
import numpy as np
np.random.seed(0)

d = 8
T = 50
input_dim = 4

Wh = 0.95 * np.random.randn(d, d) / np.sqrt(d)
Wx = np.random.randn(d, input_dim) / np.sqrt(input_dim)
b = np.zeros(d)

def rnn_forward(xs, Wh, Wx, b, sigma=np.tanh):
    T = len(xs)
    d = Wh.shape[0]
    h = np.zeros(d)
    hs = [h]
    for t in range(T):
        h = sigma(Wh @ h + Wx @ xs[t] + b)
        hs.append(h)
    return hs

xs = np.random.randn(T, input_dim)
hs = rnn_forward(xs, Wh, Wx, b)
print('h_1 =', np.round(hs[1], 3))
print('h_T =', np.round(hs[T], 3))

# Show h_T depends on x_1: perturb x_1 and recompute.
xs2 = xs.copy(); xs2[0] += 1e-3
hs2 = rnn_forward(xs2, Wh, Wx, b)
print('||h_T - h_T_perturbed|| =', np.linalg.norm(hs[T] - hs2[T]))


## Vanishing/exploding gradients

We measure $\|\partial L / \partial h_t\|$ for a loss $L = \tfrac{1}{2} \|h_T\|^2$ in the *linearized* RNN $h_t = W h_{t-1}$. By Theorem 1, $\partial L / \partial h_t = (W^\top)^{T-t} h_T$.


In [ ]:
import numpy as np
np.random.seed(0)

d = 8
T = 50

def make_W(spec_norm, d, seed=0):
    rng = np.random.default_rng(seed)
    A = rng.standard_normal((d, d))
    U, S, Vt = np.linalg.svd(A)
    # Reset top singular value to spec_norm, decay others
    S = spec_norm * np.linspace(1.0, 0.5, d)
    return U @ np.diag(S) @ Vt

def grad_norms(W, T):
    h = np.random.randn(W.shape[0])
    # Forward (linear): h_t = W^t h_0
    hs = [h]
    for _ in range(T):
        hs.append(W @ hs[-1])
    g = hs[-1].copy()  # dL/dh_T = h_T for L = 0.5 ||h_T||^2
    norms = [np.linalg.norm(g)]
    for _ in range(T):
        g = W.T @ g
        norms.append(np.linalg.norm(g))
    # norms[k] = ||dL/dh_{T-k}||
    return norms

results = {}
for s in [0.5, 0.95, 1.0, 1.05, 1.5]:
    W = make_W(s, d)
    results[s] = grad_norms(W, T)
    print(f'spec_norm={s}: ||grad@t=0|| / ||grad@t=T|| = {results[s][-1]/results[s][0]:.3e}')

try:
    import matplotlib.pyplot as plt
    for s, ns in results.items():
        plt.semilogy(range(T+1), ns, label=f'$\\sigma_{{max}}={s}$')
    plt.xlabel('T - t')
    plt.ylabel('$\\|\\partial L/\\partial h_t\\|$')
    plt.legend(); plt.title('Vanishing/exploding gradients')
    plt.tight_layout(); plt.show()
except Exception as e:
    print('matplotlib unavailable:', e)


## Theorem statement (verification)

We verify $\sigma_{\max}(W^k) = \sigma_{\max}(W)^k$ for a small symmetric (normal) $W$ via two methods: explicit SVD of $W^k$ and power iteration.


In [ ]:
import numpy as np
np.random.seed(0)

d = 5
A = np.random.randn(d, d)
W = (A + A.T) / 2  # symmetric => normal
W = 0.9 * W / np.linalg.norm(W, 2)  # spec_norm = 0.9
spec = np.linalg.norm(W, 2)
print(f'sigma_max(W) = {spec:.6f}')

for k in [1, 5, 10, 20]:
    Wk = np.linalg.matrix_power(W, k)
    svd_top = np.linalg.svd(Wk, compute_uv=False)[0]
    # Power iteration
    v = np.random.randn(d)
    for _ in range(200):
        v = Wk.T @ (Wk @ v)
        v /= np.linalg.norm(v)
    pi_top = np.linalg.norm(Wk @ v)
    print(f'k={k:2d}: SVD={svd_top:.6e}  power_iter={pi_top:.6e}  predicted={spec**k:.6e}')


## Attention's escape

Now build $h_T = \sum_s \alpha_{Ts} V_s$ with hand-set weights $\alpha$. The gradient $\partial L/\partial V_s = \alpha_{Ts}\,\partial L/\partial h_T$ is one matrix-vector product, independent of $T - s$.


In [ ]:
import numpy as np
np.random.seed(0)

T = 50
d = 8
Vs = np.random.randn(T, d)             # value vectors V_s
alpha = np.ones(T) / T                  # uniform attention weights
h_T = (alpha[:, None] * Vs).sum(axis=0) # = sum_s alpha_s V_s

# L = 0.5 ||h_T||^2  =>  dL/dh_T = h_T
grad_hT = h_T.copy()

# Gradients at every position s: dL/dV_s = alpha_s * h_T
grads = alpha[:, None] * grad_hT[None, :]
norms = np.linalg.norm(grads, axis=1)

print('||dL/dV_1||  =', norms[0])
print('||dL/dV_25|| =', norms[25])
print('||dL/dV_50|| =', norms[-1])
print('ratio max/min =', norms.max() / norms.min())
print('=> independent of T - s (no exponential decay).')


## Takeaway

RNNs propagate gradients through $T - t$ multiplications by $W^\top$, exponentiating $\sigma_{\max}(W)$. Attention replaces this product with a single weighted sum: $O(1)$ gradient distance between any two positions. This is why every modern LLM (GPT, Claude, Gemini, Llama) is built on attention, not recurrence. We formalize attention in Chapter 21.


# Scaled dot-product attention

Given $X \in \mathbb{R}^{T \times d_{\text{model}}}$ and learned projections $W_Q, W_K \in \mathbb{R}^{d_{\text{model}} \times d_k}$, $W_V \in \mathbb{R}^{d_{\text{model}} \times d_v}$, set $Q = X W_Q$, $K = X W_K$, $V = X W_V$ and define
$$\mathrm{Attn}(Q,K,V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V.$$
Each row of the attention weight matrix is a probability distribution; the output is a convex combination of the value rows.

In [ ]:
import numpy as np
np.random.seed(0)

def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

def attention(Q, K, V):
    d_k = Q.shape[-1]
    logits = (Q @ K.T) / np.sqrt(d_k)
    A = softmax(logits, axis=-1)
    return A @ V, A

T, d_model, d_k, d_v = 6, 8, 4, 4
X = np.random.randn(T, d_model)
W_Q = np.random.randn(d_model, d_k) / np.sqrt(d_model)
W_K = np.random.randn(d_model, d_k) / np.sqrt(d_model)
W_V = np.random.randn(d_model, d_v) / np.sqrt(d_model)

Q, K, V = X @ W_Q, X @ W_K, X @ W_V
out, A = attention(Q, K, V)
print('Attention matrix A (T x T):')
print(np.round(A, 3))
print('Row sums (should all be 1):', np.round(A.sum(axis=1), 6))
print('Output shape:', out.shape)

## The $\sqrt{d_k}$ scaling: variance derivation

If $Q, K \in \mathbb{R}^{d_k}$ have i.i.d. zero-mean unit-variance entries, then $S = Q \cdot K = \sum_{k=1}^{d_k} Q_k K_k$ has
$$\mathbb{E}[S] = 0, \qquad \mathrm{Var}(S) = \sum_{k} \mathbb{E}[Q_k^2 K_k^2] = d_k.$$
Dividing by $\sqrt{d_k}$ rescales to unit variance, keeping softmax in the non-saturating regime.

In [ ]:
import numpy as np
np.random.seed(0)

N = 1000
for d_k in [4, 16, 64, 256]:
    Q = np.random.randn(N, d_k)
    K = np.random.randn(N, d_k)
    dots = np.sum(Q * K, axis=1)
    print(f'd_k = {d_k:4d}  empirical Var(Q.K) = {dots.var():7.3f}  (theory: {d_k})')

## Saturation without scaling

If logits have stddev $\sqrt{d_k}$ and we feed them to softmax without the $1/\sqrt{d_k}$ correction, the largest logit dominates. The output distribution collapses to a one-hot, its entropy approaches 0, and the softmax Jacobian vanishes.

In [ ]:
import numpy as np
np.random.seed(0)

def softmax(x):
    x = x - np.max(x)
    e = np.exp(x)
    return e / e.sum()

def entropy(p, eps=1e-12):
    return -float(np.sum(p * np.log(p + eps)))

T, d_k = 64, 256
logits = np.random.randn(T) * np.sqrt(d_k)
p_unscaled = softmax(logits)
p_scaled = softmax(logits / np.sqrt(d_k))
print(f'max logit          = {logits.max():.3f}')
print(f'entropy unscaled   = {entropy(p_unscaled):.4f}  (max possible = log T = {np.log(T):.4f})')
print(f'entropy scaled     = {entropy(p_scaled):.4f}')
print(f'max prob unscaled  = {p_unscaled.max():.4f}  (saturated -> ~1)')
print(f'max prob scaled    = {p_scaled.max():.4f}  (well-spread)')

## Permutation equivariance

If we permute the sequence axis of $X$ (and hence of $Q, K, V$), the attention output is permuted by the same permutation. Equivalently, attention is invariant under reorderings of input tokens, which is why positional encodings (Ch. 24) are required.

In [ ]:
import numpy as np
np.random.seed(0)

def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

def attention(Q, K, V):
    d_k = Q.shape[-1]
    A = softmax((Q @ K.T) / np.sqrt(d_k), axis=-1)
    return A @ V

T, d_model, d_k, d_v = 6, 8, 4, 4
X = np.random.randn(T, d_model)
W_Q = np.random.randn(d_model, d_k) / np.sqrt(d_model)
W_K = np.random.randn(d_model, d_k) / np.sqrt(d_model)
W_V = np.random.randn(d_model, d_v) / np.sqrt(d_model)

Q, K, V = X @ W_Q, X @ W_K, X @ W_V
out = attention(Q, K, V)

perm = np.random.permutation(T)
Xp = X[perm]
Qp, Kp, Vp = Xp @ W_Q, Xp @ W_K, Xp @ W_V
out_perm = attention(Qp, Kp, Vp)

# out_perm should equal out[perm]
diff = np.max(np.abs(out_perm - out[perm]))
print('Permutation:', perm)
print(f'max |Attn(P X) - P Attn(X)| = {diff:.2e}  (should be ~1e-15)')

## Gradient flow: attention vs. RNN

We compare $\|\partial L / \partial X_0\|$ when $L$ depends only on the last position $X_{T-1}$, for $T = 50$.

- **RNN** with weight matrix of spectral norm $0.9$: by Ch. 20, $\|\partial h_{T-1}/\partial h_0\| \lesssim 0.9^{49} \approx 5.7 \times 10^{-3}$.
- **Attention**: by Theorem 5, the gradient passes through one matmul with weight $A_{T-1, 0} \sim 1/T$, i.e. $O(1/T)$ — orders of magnitude larger.

In [ ]:
import numpy as np
np.random.seed(0)

def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

T, d = 50, 8
X = np.random.randn(T, d)

# --- Attention path: identity Q, K, V (just X). Gradient of out[T-1] w.r.t. X[0]
# is A[T-1, 0] * I_d (Theorem 5), so the gradient norm is sqrt(d) * A[T-1, 0].
Q = K = V = X
A = softmax((Q @ K.T) / np.sqrt(d), axis=-1)
g_attn = A[T - 1, 0] * np.sqrt(d)
print(f'Attention   |dL/dX_0| ~ A[T-1,0] * sqrt(d) = {g_attn:.4e}')

# --- RNN path: h_t = W h_{t-1} with spectral norm 0.9, identity nonlinearity
# (best case for the RNN). Gradient norm = ||W^{T-1}|| <= 0.9^{T-1}.
rho = 0.9
W = np.random.randn(d, d)
U, S, Vt = np.linalg.svd(W)
W = U @ np.diag(np.full(d, rho)) @ Vt   # exact spectral norm rho
Wpow = np.linalg.matrix_power(W, T - 1)
g_rnn = np.linalg.norm(Wpow, 2)
print(f'RNN(rho=0.9) |dh_{{T-1}}/dh_0|       = {g_rnn:.4e}')
print(f'ratio attention / RNN = {g_attn / g_rnn:.2e}')

## Multi-head attention

A single attention head (Chapter 21) compresses every relational pattern into one softmax-weighted average. **Multi-head attention** runs $H$ heads in parallel on disjoint $d/H$-dim subspaces, concatenates, and re-projects:
$$\mathrm{head}_h = \mathrm{Attn}(X W_Q^{(h)}, X W_K^{(h)}, X W_V^{(h)}), \qquad \mathrm{MHA}(X) = \mathrm{Concat}(\mathrm{head}_1,\ldots,\mathrm{head}_H) W_O.$$
Total parameters: $4 d^2$, independent of $H$ (Theorem 5).

In [ ]:
import numpy as np
np.random.seed(0)

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def attn(Q, K, V):
    dk = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(dk)
    return softmax(scores, axis=-1) @ V

T, d, H = 8, 12, 3
dk = d // H
X = np.random.randn(T, d)
WQ = [np.random.randn(d, dk) for _ in range(H)]
WK = [np.random.randn(d, dk) for _ in range(H)]
WV = [np.random.randn(d, dk) for _ in range(H)]
WO = np.random.randn(H * dk, d)

heads = [attn(X @ WQ[h], X @ WK[h], X @ WV[h]) for h in range(H)]
Y_loop = np.concatenate(heads, axis=-1) @ WO
print('input  shape:', X.shape)
print('output shape:', Y_loop.shape)
assert Y_loop.shape == X.shape, 'MHA must preserve sequence shape'


## MHA = block-structured single attention (Theorem 4)

Stack the per-head $W_Q^{(h)}$ horizontally into $W_Q^{\mathrm{blk}} \in \mathbb{R}^{d \times Hd_k}$. Then $X W_Q^{\mathrm{blk}}$ is the horizontal concatenation of the per-head queries. Reshaping to $(T, H, d_k)$ recovers each head's $Q^{(h)}$ as a slice. The same holds for $K, V$. So the per-head loop equals one block-projection followed by per-head attention --- nothing has been mixed across heads. We verify this numerically.

In [ ]:
# Build block-diagonal projections by horizontal concat.
WQ_blk = np.concatenate(WQ, axis=1)  # (d, H*dk)
WK_blk = np.concatenate(WK, axis=1)
WV_blk = np.concatenate(WV, axis=1)

Q_all = (X @ WQ_blk).reshape(T, H, dk)  # split last axis into H heads
K_all = (X @ WK_blk).reshape(T, H, dk)
V_all = (X @ WV_blk).reshape(T, H, dk)

out_per_head = []
for h in range(H):
    out_per_head.append(attn(Q_all[:, h, :], K_all[:, h, :], V_all[:, h, :]))
Y_block = np.concatenate(out_per_head, axis=-1) @ WO

diff = np.max(np.abs(Y_loop - Y_block))
print(f'max |Y_loop - Y_block| = {diff:.3e}')
assert diff < 1e-12, 'block-equivalent picture failed'


## Compute complexity (Theorem 6)

$\mathrm{cost}(\mathrm{MHA}) = \Theta(T d^2 + T^2 d)$. For long sequences ($T \gg d$), the $T^2 d$ attention term dominates: doubling $T$ quadruples runtime. We time the forward pass for $T \in \{32, 64, 128, 256\}$ at fixed $d = 64, H = 4$.

In [ ]:
import time

def mha_forward(X, WQ_blk, WK_blk, WV_blk, WO, H):
    T, d = X.shape
    dk = d // H
    Q = (X @ WQ_blk).reshape(T, H, dk)
    K = (X @ WK_blk).reshape(T, H, dk)
    V = (X @ WV_blk).reshape(T, H, dk)
    outs = []
    for h in range(H):
        outs.append(attn(Q[:, h, :], K[:, h, :], V[:, h, :]))
    return np.concatenate(outs, axis=-1) @ WO

d2, H2 = 64, 4
WQb = np.random.randn(d2, d2)
WKb = np.random.randn(d2, d2)
WVb = np.random.randn(d2, d2)
WOb = np.random.randn(d2, d2)

print(f"{'T':>5} {'time (ms)':>12} {'time/T^2 (us)':>16}")
REPEAT = 20
for T_ in [32, 64, 128, 256]:
    Xt = np.random.randn(T_, d2)
    # warmup
    mha_forward(Xt, WQb, WKb, WVb, WOb, H2)
    t0 = time.perf_counter()
    for _ in range(REPEAT):
        mha_forward(Xt, WQb, WKb, WVb, WOb, H2)
    dt = (time.perf_counter() - t0) / REPEAT
    print(f'{T_:>5d} {dt*1e3:>12.3f} {dt/T_**2*1e6:>16.4f}')
print('time/T^2 should approach a constant once T^2 d dominates over T d^2.')


## MQA and GQA (Theorem 8)

Inference autoregressive decoding caches $K, V$ for every past token, every layer. MHA cache size: $2 T d$ per layer. **MQA** shares one $W_K, W_V$ across all heads, shrinking the cache to $2 T d / H$. **GQA** with $G$ groups shrinks it to $2 T d \cdot G / H$. Llama 2/3 use $G \in \{4, 8\}$. Below we implement both, verify shapes, and tabulate parameter and KV-cache counts.

In [ ]:
def mqa_forward(X, WQ_list, WK_shared, WV_shared, WO):
    H = len(WQ_list)
    K = X @ WK_shared  # shared K
    V = X @ WV_shared  # shared V
    return np.concatenate([attn(X @ WQ_list[h], K, V) for h in range(H)], axis=-1) @ WO

def gqa_forward(X, WQ_list, WK_group, WV_group, WO, G):
    H = len(WQ_list)
    assert H % G == 0
    heads_per_group = H // G
    Ks = [X @ WK_group[g] for g in range(G)]
    Vs = [X @ WV_group[g] for g in range(G)]
    outs = []
    for h in range(H):
        g = h // heads_per_group
        outs.append(attn(X @ WQ_list[h], Ks[g], Vs[g]))
    return np.concatenate(outs, axis=-1) @ WO

# Shape sanity check
WK_shared = np.random.randn(d, dk)
WV_shared = np.random.randn(d, dk)
Y_mqa = mqa_forward(X, WQ, WK_shared, WV_shared, WO)
G = 2  # 2 KV groups, 3 heads -> uneven, so use H=4 below for GQA demo
assert Y_mqa.shape == X.shape

# Param + KV cache table for MHA / GQA(G=2) / MQA at d=512, H=8, T=4096
d_, H_, T_ = 512, 8, 4096
dk_ = d_ // H_
G_ = 2
params = {
    'MHA': 3 * H_ * d_ * dk_ + d_ * d_,                    # H Q + H K + H V + WO
    'GQA(G=2)': H_ * d_ * dk_ + 2 * G_ * d_ * dk_ + d_ * d_,  # H Q + G K + G V + WO
    'MQA': H_ * d_ * dk_ + 2 * d_ * dk_ + d_ * d_,         # H Q + 1 K + 1 V + WO
}
kv_cache = {  # bytes, fp16, per layer
    'MHA': 2 * 2 * T_ * d_,                  # 2 (K,V) * fp16 * T * d
    'GQA(G=2)': 2 * 2 * T_ * G_ * dk_,
    'MQA': 2 * 2 * T_ * dk_,
}
print(f'{"variant":<10} {"params":>12} {"KV / layer (MB, fp16)":>24}')
for k in ['MHA', 'GQA(G=2)', 'MQA']:
    print(f'{k:<10} {params[k]:>12,d} {kv_cache[k]/1e6:>24.3f}')
print()
print(f'GQA(G=2) shrinks KV cache by {kv_cache["MHA"]/kv_cache["GQA(G=2)"]:.1f}x vs MHA.')
print(f'MQA      shrinks KV cache by {kv_cache["MHA"]/kv_cache["MQA"]:.1f}x vs MHA.')


## The pre-norm transformer block

Every layer of GPT-2/3/4, Llama, Claude, etc. is some variant of the **pre-norm transformer block**:

$$z = x + \mathrm{MHA}(\mathrm{LN}(x)), \qquad y = z + \mathrm{FFN}(\mathrm{LN}(z)).$$

Two sublayers (multi-head self-attention and a position-wise FFN), each wrapped in a *residual* connection, with a *normalization layer* on the input to each sublayer. We implement one block in numpy and verify that the output shape matches the input.


In [ ]:
import numpy as np

np.random.seed(0)
T, d, H, d_ff = 6, 16, 4, 64
assert d % H == 0
d_h = d // H

def layer_norm(x, eps=1e-5):
    mu = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    return (x - mu) / np.sqrt(var + eps)

def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)
    ez = np.exp(z)
    return ez / ez.sum(axis=axis, keepdims=True)

def mha(x, Wq, Wk, Wv, Wo):
    T, d = x.shape
    Q = (x @ Wq).reshape(T, H, d_h).transpose(1, 0, 2)
    K = (x @ Wk).reshape(T, H, d_h).transpose(1, 0, 2)
    V = (x @ Wv).reshape(T, H, d_h).transpose(1, 0, 2)
    scores = Q @ K.transpose(0, 2, 1) / np.sqrt(d_h)
    A = softmax(scores, axis=-1)
    out = (A @ V).transpose(1, 0, 2).reshape(T, d)
    return out @ Wo

def ffn(x, W1, b1, W2, b2):
    h = np.maximum(0.0, x @ W1 + b1)  # ReLU stand-in for GELU
    return h @ W2 + b2

scale = 1.0 / np.sqrt(d)
Wq = np.random.randn(d, d) * scale
Wk = np.random.randn(d, d) * scale
Wv = np.random.randn(d, d) * scale
Wo = np.random.randn(d, d) * scale
W1 = np.random.randn(d, d_ff) * scale
b1 = np.zeros(d_ff)
W2 = np.random.randn(d_ff, d) * (1.0 / np.sqrt(d_ff))
b2 = np.zeros(d)

x = np.random.randn(T, d)

z = x + mha(layer_norm(x), Wq, Wk, Wv, Wo)
y = z + ffn(layer_norm(z), W1, b1, W2, b2)

print('input  shape :', x.shape)
print('output shape :', y.shape)
print('rel norm change ||y-x||/||x|| =', np.linalg.norm(y - x) / np.linalg.norm(x))


## Residuals and gradient flow

Theorem (Ch.~23): with $y = F(x)+x$, $\partial y/\partial x = J_F + I$. Across $L$ blocks, $\partial x_L/\partial x_0 = \prod_\ell (I + J_{F^{(\ell)}})$. The identity term keeps the product from vanishing even when each $J_F$ has small singular values.

We make this concrete: stack 20 blocks $F(x) = W \tanh(x)$ with deliberately small $W$. Compare the gradient norm ratios with and without skip connections, computed by finite differences.


In [ ]:
import numpy as np

np.random.seed(0)
L, d = 20, 16
Ws = [0.3 * np.random.randn(d, d) / np.sqrt(d) for _ in range(L)]  # small spectral radius

def forward(x, use_residual):
    for W in Ws:
        f = np.tanh(x) @ W
        x = x + f if use_residual else f
    return x

def grad_input_norm(x0, use_residual, eps=1e-5):
    # Loss L = sum(forward(x0)).  dL/dx0_i ~ (L(x0+eps e_i) - L(x0-eps e_i))/(2 eps).
    base = forward(x0, use_residual).sum()
    g = np.zeros_like(x0)
    for i in range(x0.size):
        e = np.zeros_like(x0); e.flat[i] = eps
        g.flat[i] = (forward(x0 + e, use_residual).sum() - forward(x0 - e, use_residual).sum()) / (2 * eps)
    return np.linalg.norm(g)

x0 = np.random.randn(d)
g_res  = grad_input_norm(x0, use_residual=True)
g_plain = grad_input_norm(x0, use_residual=False)
# 'Output-side' gradient norm = ||dL/dx_L|| = ||1|| = sqrt(d)
g_out = np.sqrt(d)
print(f'with residual    : ||dL/dx_0|| / ||dL/dx_L|| = {g_res/g_out:.4e}')
print(f'without residual : ||dL/dx_0|| / ||dL/dx_L|| = {g_plain/g_out:.4e}')
print(f'ratio (res / no-res) = {g_res / max(g_plain, 1e-30):.3e}')


## LayerNorm and its invariances

$\mathrm{LN}(x) = (x - \mu \mathbf 1)/\sqrt{\sigma^2 + \varepsilon}$ (we set $\gamma = 1, \beta = 0$ to isolate the normalizer). The chapter proves $\mathrm{LN}(\alpha x + \beta_0 \mathbf 1) = \mathrm{LN}(x)$ for any $\alpha > 0, \beta_0 \in \mathbb R$. We test on random $x$ and many random affine rescalings.


In [ ]:
import numpy as np

np.random.seed(0)
d = 8

def LN(x, eps=1e-5):
    mu = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    return (x - mu) / np.sqrt(var + eps)

x = np.random.randn(d)
ln_x = LN(x)

max_err = 0.0
for _ in range(50):
    alpha = float(np.exp(np.random.randn()))   # alpha > 0
    beta0 = float(np.random.randn() * 5.0)
    y = alpha * x + beta0
    err = np.linalg.norm(LN(y) - ln_x)
    max_err = max(max_err, err)

print('LN(x):', np.round(ln_x, 4))
print('max ||LN(alpha x + beta) - LN(x)|| over 50 random (alpha,beta) =', max_err)


## RMSNorm vs LayerNorm

$\mathrm{RMSN}(x) = x / \sqrt{\mathrm{mean}(x^2) + \varepsilon}$ drops the mean-subtraction step. When $x$ is already centered ($\mu = 0$), $\mathrm{RMSN}(x) = \mathrm{LN}(x)$. For non-centered $x$ they differ visibly. Modern open-weight LLMs (Llama, Mistral, Qwen) use RMSN because residual streams in deep pre-norm networks empirically have near-zero mean.


In [ ]:
import numpy as np

np.random.seed(0)
d = 64

def LN(x, eps=1e-5):
    mu = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    return (x - mu) / np.sqrt(var + eps)

def RMSN(x, eps=1e-5):
    rms2 = (x * x).mean(axis=-1, keepdims=True)
    return x / np.sqrt(rms2 + eps)

x_center = np.random.randn(d); x_center -= x_center.mean()
x_off    = np.random.randn(d) + 3.0  # large positive mean

for label, x in [('centered (mean ~ 0)', x_center), ('offset (mean ~ 3)', x_off)]:
    ln, rn = LN(x), RMSN(x)
    diff = np.linalg.norm(rn - ln) / np.linalg.norm(ln)
    print(f'{label:25s}  mean = {x.mean():+.3f}   ||RMSN - LN|| / ||LN|| = {diff:.4e}')


# Chapter 24 — Positional Encoding

From Chapter 21 we know attention is permutation-equivariant: shuffling the input tokens shuffles the output identically. So a transformer without positional information cannot distinguish 'dog bites man' from 'man bites dog'. We need to inject token position into the representation.

We will (1) numerically confirm the permutation problem, (2) build sinusoidal PEs and verify shift-as-rotation (Thm 24.1), (3) build RoPE and verify the relative-position theorem (Thm 24.2), and (4) verify translation invariance (Cor 24.3).

In [ ]:
import numpy as np
np.random.seed(0)

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def attention(X):
    # Self-attention with identity Q=K=V projections (clean isolation of permutation property)
    d = X.shape[-1]
    scores = X @ X.T / np.sqrt(d)
    return softmax(scores) @ X

T, d = 5, 4
X = np.random.randn(T, d)
perm = np.array([2, 0, 4, 1, 3])
P = np.eye(T)[perm]

Y = attention(X)
Yp = attention(P @ X)
print('||Yp - P @ Y|| =', np.linalg.norm(Yp - P @ Y))
print('Permutation-equivariance confirmed:', np.allclose(Yp, P @ Y))

## Sinusoidal positional encoding

$\mathrm{PE}(\mathrm{pos}, 2i) = \sin(\mathrm{pos}\,\theta_i)$, $\mathrm{PE}(\mathrm{pos}, 2i+1) = \cos(\mathrm{pos}\,\theta_i)$, with $\theta_i = 10000^{-2i/d}$. Different positions get distinct vectors; nearby positions get similar vectors at low-frequency dimensions but differ sharply at high-frequency dimensions.

In [ ]:
def sinusoidal_pe(T, d, base=10000.0):
    assert d % 2 == 0
    pos = np.arange(T)[:, None]
    i = np.arange(d // 2)[None, :]
    theta = base ** (-2 * i / d)
    angles = pos * theta  # (T, d/2)
    PE = np.empty((T, d))
    PE[:, 0::2] = np.sin(angles)
    PE[:, 1::2] = np.cos(angles)
    return PE

PE = sinusoidal_pe(T=32, d=16)
print('PE shape:', PE.shape)
print('First 3 positions, first 6 dims:')
print(np.round(PE[:3, :6], 3))

try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(6, 4))
    plt.imshow(PE, aspect='auto', cmap='RdBu')
    plt.colorbar(); plt.xlabel('dim'); plt.ylabel('position')
    plt.title('Sinusoidal PE'); plt.tight_layout(); plt.show()
except Exception as e:
    print('(matplotlib unavailable, printed sample instead):', type(e).__name__)

## Shift-as-rotation (Theorem 24.1)

For each frequency $\theta_i$ and offset $k$, the pair $(\sin(\mathrm{pos}\,\theta_i), \cos(\mathrm{pos}\,\theta_i))$ shifts by a fixed $2\times 2$ rotation. We construct the full block-diagonal $M_k$ and verify $\mathrm{PE}(\mathrm{pos}+k) = M_k \cdot \mathrm{PE}(\mathrm{pos})$ to machine precision.

In [ ]:
def shift_matrix(k, d, base=10000.0):
    # Acts on vectors ordered (sin_0, cos_0, sin_1, cos_1, ...) — matches sinusoidal_pe layout.
    M = np.zeros((d, d))
    for i in range(d // 2):
        theta = base ** (-2 * i / d)
        c, s = np.cos(k * theta), np.sin(k * theta)
        # [sin', cos']^T = [[c, s], [-s, c]] [sin, cos]^T
        M[2*i:2*i+2, 2*i:2*i+2] = np.array([[c, s], [-s, c]])
    return M

d_test = 16
PE_full = sinusoidal_pe(T=64, d=d_test)
max_err = 0.0
for k in [1, 3, 7, 17]:
    Mk = shift_matrix(k, d_test)
    for pos in [0, 5, 23, 40]:
        lhs = PE_full[pos + k]
        rhs = Mk @ PE_full[pos]
        err = np.linalg.norm(lhs - rhs)
        max_err = max(max_err, err)
print(f'max ||PE(pos+k) - M_k PE(pos)|| over tests: {max_err:.3e}')
assert max_err < 1e-12, 'Theorem 24.1 violated'

## RoPE construction

Apply $R_{\mathrm{pos}}$ to queries and keys before the dot product. We verify Theorem 24.2: $\langle R_m q_m, R_n k_n \rangle = q_m^\top R_{n-m} k_n$, using $R_m^\top = R_{-m}$ and $R_a R_b = R_{a+b}$.

In [ ]:
def rope_matrix(pos, d, base=10000.0):
    R = np.zeros((d, d))
    for i in range(d // 2):
        theta = base ** (-2 * i / d)
        c, s = np.cos(pos * theta), np.sin(pos * theta)
        R[2*i:2*i+2, 2*i:2*i+2] = np.array([[c, -s], [s, c]])
    return R

np.random.seed(0)
T, d = 6, 8
Q = np.random.randn(T, d)
K = np.random.randn(T, d)

# Method A: rotate then dot product.
Q_rot = np.stack([rope_matrix(m, d) @ Q[m] for m in range(T)])
K_rot = np.stack([rope_matrix(n, d) @ K[n] for n in range(T)])
scores_A = Q_rot @ K_rot.T

# Method B: relative-rotation formulation q_m^T R_{n-m} k_n.
scores_B = np.zeros((T, T))
for m in range(T):
    for n in range(T):
        scores_B[m, n] = Q[m] @ rope_matrix(n - m, d) @ K[n]

err = np.linalg.norm(scores_A - scores_B)
print(f'||scores_A - scores_B|| = {err:.3e}')
print('Theorem 24.2 verified:', np.allclose(scores_A, scores_B, atol=1e-12))

## Translation invariance (Corollary 24.3)

Shift every position by a constant $c$. Since RoPE attention scores depend only on $n - m$, the score matrix should be unchanged. We verify for $c \in \{1, 5, 10\}$.

In [ ]:
def rope_attention_scores(Q, K, offset=0):
    T, d = Q.shape
    Qr = np.stack([rope_matrix(m + offset, d) @ Q[m] for m in range(T)])
    Kr = np.stack([rope_matrix(n + offset, d) @ K[n] for n in range(T)])
    return Qr @ Kr.T

base_scores = rope_attention_scores(Q, K, offset=0)
for c in [1, 5, 10]:
    shifted = rope_attention_scores(Q, K, offset=c)
    err = np.linalg.norm(base_scores - shifted)
    print(f'c = {c:2d}: ||scores(c) - scores(0)|| = {err:.3e}')
    assert np.allclose(base_scores, shifted, atol=1e-10), 'Translation invariance failed'
print('Corollary 24.3 verified for all tested offsets.')

# Block F — Pre-training


# Chapter 25 — Causal masking and next-token prediction

**Motivation.** A decoder-only language model factorizes the joint over a sequence by the chain rule of probability:
$$ p_\theta(x_1, \ldots, x_T) \;=\; \prod_{t=1}^{T} p_\theta(x_t \mid x_{<t}). $$

We want one forward pass on $x_{1:T}$ to give us all $T$ conditional distributions in parallel — but the model must not peek at future tokens. The fix is the **causal attention mask**: the $T\times T$ matrix $M$ with $M_{ij}=0$ for $j\le i$ and $M_{ij}=-\infty$ for $j>i$, added to the attention scores **before** the softmax. Since $\exp(-\infty)=0$, future positions get zero weight. Combined with **teacher forcing**, this gives us $T$ supervision signals from one forward/backward pass.

## 1. Build the causal mask


In [ ]:
import numpy as np

T = 8
M = np.where(np.tril(np.ones((T, T))) == 1, 0.0, -np.inf)
print('Causal mask M (shape', M.shape, '):')
print(M)


## 2. A tiny single-head causal attention layer

Random $X \in \mathbb{R}^{T\times d}$ with $T=8$, $d=16$. Single head, $d_k=d_v=16$. Compute attention with and without the mask. Verify that with the mask, row $t$ assigns **zero** weight to all positions $j > t$.


In [ ]:
import numpy as np

np.random.seed(0)
T, d = 8, 16
d_k = d_v = 16
X = np.random.randn(T, d)
W_q = np.random.randn(d, d_k) / np.sqrt(d)
W_k = np.random.randn(d, d_k) / np.sqrt(d)
W_v = np.random.randn(d, d_v) / np.sqrt(d)
Q, K, V = X @ W_q, X @ W_k, X @ W_v

def softmax_rows(Z):
    Z = Z - Z.max(axis=-1, keepdims=True)
    eZ = np.exp(Z)
    return eZ / eZ.sum(axis=-1, keepdims=True)

scores = (Q @ K.T) / np.sqrt(d_k)
A_unmasked = softmax_rows(scores)

M = np.where(np.tril(np.ones((T, T))) == 1, 0.0, -np.inf)
A_masked = softmax_rows(scores + M)

print('Unmasked attention row 3:', np.round(A_unmasked[3], 3))
print('Masked   attention row 3:', np.round(A_masked[3], 3))

# Verify: masked row t puts zero weight on j > t
for t in range(T):
    future = A_masked[t, t+1:]
    assert np.allclose(future, 0.0), f'leak at row {t}'
    assert np.isclose(A_masked[t].sum(), 1.0)
print('OK: every row sums to 1 and has zero weight on the strict future.')


## 3. NTP loss = sum of cross-entropies with one-hot targets

Tiny vocab $V=6$, $T=10$. The 'model' is just a fixed logits matrix $Z\in\mathbb{R}^{T\times V}$ (seed 0). The NTP loss is
$$ \mathcal{L} \;=\; -\sum_{t=1}^T \log\,\mathrm{softmax}(z_t)_{x_t}. $$
Equivalently, with one-hot $y_t = e_{x_t}$, this is $\sum_t H(y_t, p_t)$ — exactly the Ch.~12 cross-entropy MLE.


In [ ]:
import numpy as np

np.random.seed(0)
V, T = 6, 10
Z = np.random.randn(T, V)
x = np.array([2, 5, 0, 3, 1, 4, 2, 0, 5, 3])  # ground-truth tokens

def softmax_rows(Z):
    Z = Z - Z.max(axis=-1, keepdims=True)
    eZ = np.exp(Z)
    return eZ / eZ.sum(axis=-1, keepdims=True)

P = softmax_rows(Z)
loss_direct = -np.sum(np.log(P[np.arange(T), x]))

# One-hot cross-entropy form
Y = np.zeros((T, V))
Y[np.arange(T), x] = 1.0
loss_xent = -np.sum(Y * np.log(P))

print(f'NTP loss (direct) : {loss_direct:.6f}')
print(f'Sum cross-entropy : {loss_xent:.6f}')
assert np.isclose(loss_direct, loss_xent)
print('OK: NTP loss equals summed cross-entropy with one-hot targets.')


## 4. Causal mask + parallel training: locality witness

We do **one** forward pass that produces all $T$ logits in parallel. Then we perturb the input embedding at position $T$ and check that the logit at position $T-1$ does **not** change — this is exactly Theorem 25.4 ($\partial \mathcal{L}_{T-1}/\partial h^{(0)}_T = 0$).


In [ ]:
import numpy as np

np.random.seed(0)
T, d = 12, 8
V = 7
X = np.random.randn(T, d)
W_q = np.random.randn(d, d) / np.sqrt(d)
W_k = np.random.randn(d, d) / np.sqrt(d)
W_v = np.random.randn(d, d) / np.sqrt(d)
W_out = np.random.randn(d, V) / np.sqrt(d)
M = np.where(np.tril(np.ones((T, T))) == 1, 0.0, -np.inf)

def softmax_rows(Z):
    Z = Z - Z.max(axis=-1, keepdims=True)
    eZ = np.exp(Z)
    return eZ / eZ.sum(axis=-1, keepdims=True)

def causal_forward(X):
    Q, K, V_ = X @ W_q, X @ W_k, X @ W_v
    A = softmax_rows((Q @ K.T) / np.sqrt(d) + M)
    Y = A @ V_
    logits = Y @ W_out
    return logits

logits_orig = causal_forward(X)

# Perturb position T-1 (last position, 0-indexed)
X_pert = X.copy()
X_pert[T-1] += 5.0 * np.random.randn(d)
logits_pert = causal_forward(X_pert)

diff_at_Tminus2 = np.max(np.abs(logits_orig[T-2] - logits_pert[T-2]))
diff_at_Tminus1 = np.max(np.abs(logits_orig[T-1] - logits_pert[T-1]))
print(f'max |Δlogit| at position T-2 (should be ~0)   : {diff_at_Tminus2:.2e}')
print(f'max |Δlogit| at position T-1 (should be > 0)  : {diff_at_Tminus1:.2e}')
assert diff_at_Tminus2 < 1e-10, 'causal mask violated — future leaked into past!'
assert diff_at_Tminus1 > 1e-3, 'perturbation should affect its own position'
print('OK: future tokens cannot influence past logits — Theorem 25.4 holds.')


In [ ]:
# ## 5. Perplexity
# PPL = exp(L/T) — the geometric mean of 1/p_theta(x_t|x_{<t}) across positions.
# A uniform model on V tokens achieves PPL = V; a trained model achieves lower NLL → lower PPL.
import numpy as np

np.random.seed(0)
V, T = 6, 10
x = np.array([2, 5, 0, 3, 1, 4, 2, 0, 5, 3])

def softmax_rows(Z):
    Z = Z - Z.max(axis=-1, keepdims=True)
    eZ = np.exp(Z)
    return eZ / eZ.sum(axis=-1, keepdims=True)

# Uniform model
P_unif = np.full((T, V), 1.0 / V)
nll_unif = -np.sum(np.log(P_unif[np.arange(T), x]))
ppl_unif = np.exp(nll_unif / T)

# Trained tiny model: bias logits toward the true tokens
Z = np.random.randn(T, V) * 0.3
Z[np.arange(T), x] += 2.5  # imitate a model that learned the data
P_trained = softmax_rows(Z)
nll_trained = -np.sum(np.log(P_trained[np.arange(T), x]))
ppl_trained = np.exp(nll_trained / T)

print(f'Uniform : NLL = {nll_unif:.4f}, PPL = {ppl_unif:.4f} (theory: V = {V})')
print(f'Trained : NLL = {nll_trained:.4f}, PPL = {ppl_trained:.4f}')
assert np.isclose(ppl_unif, V)
assert ppl_trained < ppl_unif
print('OK: lower NLL ⇔ lower perplexity; trained model beats uniform.')


### Recap

- The chain rule (Ch. 11) factorizes any joint into per-token conditionals; the transformer parameterizes each one.
- The **causal mask** enforces that the conditional at position $t$ depends only on tokens $\le t$ — verified by the perturbation experiment.
- The **NTP loss** equals MLE on the empirical distribution (Ch. 12); minimizing it pushes $p_\theta$ toward the data distribution.
- One forward pass on $x_{1:T}$ yields $T$ supervision signals — what makes transformer pre-training tractable at scale (Ch. 27 onwards: GPT, Llama, Claude, Gemini).


# Chapter 26 — Byte-Pair Encoding (BPE)

A language model operates on a finite vocabulary $\mathcal{V}$. Word-level vocabularies suffer unbounded OOV; character/byte-level vocabularies blow up sequence length. **BPE** is the dominant middle ground: start from the byte alphabet, then iteratively merge the most-frequent adjacent pair into a new token until the vocabulary reaches a target size.

Below we implement BPE training from scratch in pure Python, then verify three theorems:

1. **Determinism / round-trip:** `decode(encode(s)) == s`.
2. **Corpus-coverage monotonicity:** total token count is non-increasing in the number of merges.
3. **Vocabulary growth:** $|\mathcal{V}_M| = |\Sigma| + M$ (when every merge is novel).

## BPE training

We implement training on a tiny corpus of 10 lowercase English sentences and run for $M = 30$ merges, printing the learned vocabulary and the first ten merge rules.

In [ ]:
from collections import Counter

CORPUS = [
    'the quick brown fox jumps over the lazy dog',
    'the rain in spain stays mainly in the plain',
    'a stitch in time saves nine',
    'birds of a feather flock together',
    'the early bird catches the worm',
    'all that glitters is not gold',
    'a watched pot never boils',
    'the pen is mightier than the sword',
    'practice makes perfect every time',
    'the best things in life are free',
]

def to_seq(s):
    # represent each string as a tuple of single-character tokens
    return tuple(s)

def get_pair_counts(seqs):
    pairs = Counter()
    for seq in seqs:
        for a, b in zip(seq, seq[1:]):
            pairs[(a, b)] += 1
    return pairs

def apply_merge(seq, pair):
    a, b = pair
    out, i = [], 0
    while i < len(seq):
        if i + 1 < len(seq) and seq[i] == a and seq[i+1] == b:
            out.append(a + b)
            i += 2
        else:
            out.append(seq[i])
            i += 1
    return tuple(out)

def train_bpe(corpus, M):
    seqs = [to_seq(s) for s in corpus]
    base_vocab = sorted({c for s in corpus for c in s})
    vocab = list(base_vocab)
    merges = []
    for _ in range(M):
        pairs = get_pair_counts(seqs)
        if not pairs:
            break
        # tie-break lexicographically on the merged token
        best_pair, _ = max(pairs.items(), key=lambda kv: (kv[1], -ord(kv[0][0][0]), -ord(kv[0][1][0])))
        merges.append(best_pair)
        new_token = best_pair[0] + best_pair[1]
        if new_token not in vocab:
            vocab.append(new_token)
        seqs = [apply_merge(seq, best_pair) for seq in seqs]
    return vocab, merges, seqs

vocab, merges, seqs = train_bpe(CORPUS, M=30)
print(f'base alphabet size = {len(set(c for s in CORPUS for c in s))}')
print(f'final |V| = {len(vocab)}')
print(f'number of merges learned = {len(merges)}')
print('first 10 merges:')
for i, m in enumerate(merges[:10]):
    print(f'  {i+1:2d}.  {m[0]!r} + {m[1]!r}  ->  {(m[0]+m[1])!r}')

## Encoding and decoding

Encoding a held-out string: start from its character sequence and apply each learned merge in rank order. Decoding is concatenation. We verify the round-trip identity from the chapter.

In [ ]:
def encode(s, merges):
    seq = to_seq(s)
    for pair in merges:
        seq = apply_merge(seq, pair)
    return seq

def decode(seq):
    return ''.join(seq)

held_out = 'the brown bird is in the rain'
tokens = encode(held_out, merges)
back   = decode(tokens)
print(f'original : {held_out!r}')
print(f'tokens   : {tokens}')
print(f'len      : {len(held_out)} chars  ->  {len(tokens)} tokens')
print(f'roundtrip: {back!r}')
assert back == held_out, 'round-trip failed'
print('round-trip equality: OK')

## Corpus-coverage monotonicity

**Theorem.** Let $L_m$ be the total number of tokens needed to represent the training corpus after $m$ merges. Then $L_0 \geq L_1 \geq \cdots \geq L_M$.

We verify this empirically by training BPE for $M \in \{0, 5, 10, 20, 50\}$ and counting the resulting corpus length.

In [ ]:
def corpus_token_count(corpus, merges):
    total = 0
    for s in corpus:
        total += len(encode(s, merges))
    return total

Ms = [0, 5, 10, 20, 50]
Ls = []
for M in Ms:
    _, m_rules, _ = train_bpe(CORPUS, M=M)
    L = corpus_token_count(CORPUS, m_rules)
    Ls.append(L)
    print(f'M = {M:3d}  ->  L = {L} tokens  (|merges| = {len(m_rules)})')

for i in range(1, len(Ls)):
    assert Ls[i] <= Ls[i-1], f'monotonicity violated at M={Ms[i]}'
print('\nmonotone non-increasing: OK')

## Token frequency tail

BPE preferentially merges the most-frequent adjacent pairs first. On natural-language text, the earliest merges should correspond to common English fragments — `th`, `the`, `er`, `in`, etc. We inspect the top merges from the trained model.

In [ ]:
vocab, merges, _ = train_bpe(CORPUS, M=30)
print('top 10 learned merges (rank :: pair :: resulting subword):')
for i, (a, b) in enumerate(merges[:10]):
    print(f'  {i+1:2d}.  ({a!r:>6}, {b!r:>6})  ->  {(a+b)!r}')

# show that the learned subwords appear as common English fragments
subwords = [a + b for (a, b) in merges[:10]]
print('\nlearned subwords:', subwords)

## Vocabulary growth and connection to LLMs

**Proposition.** If we start from $|\Sigma|$ unique base characters and perform $M$ merges, each producing a novel token, then $|\mathcal{V}_M| = |\Sigma| + M$. We verify this below.

The tokenizer fixes the vocabulary $\mathcal{V}$ used by the embedding matrix $E \in \mathbb{R}^{|\mathcal{V}| \times d}$ of Chapter 19. Production tokenizers are byte-level BPE with much larger corpora and vocabularies:

- **GPT-2 / GPT-3:** byte-level BPE, $|\mathcal{V}| = 50{,}257$.
- **GPT-4 (cl100k_base):** byte-level BPE, $|\mathcal{V}| \approx 100{,}000$.
- **Llama:** SentencePiece BPE on Unicode, $|\mathcal{V}| \approx 32{,}000$.
- **Claude:** custom BPE-like scheme.

Chapter 27 examines practical pathologies of these tokenizers — numeric tokenization, multilinguality, glitch tokens — that follow directly from BPE's greedy, frequency-driven design.

In [ ]:
base_alphabet = sorted({c for s in CORPUS for c in s})
base_size = len(base_alphabet)
print(f'|Sigma| (base alphabet) = {base_size}')
print(f'  characters: {base_alphabet}\n')

for M in [0, 5, 10, 20, 30, 50]:
    vocab, merges, _ = train_bpe(CORPUS, M=M)
    print(f'M = {M:3d}:  |V| = {len(vocab)}, |Sigma| + |merges| = {base_size} + {len(merges)} = {base_size + len(merges)}')
    assert len(vocab) == base_size + len(merges), 'vocab-size identity violated'
print('\n|V_M| = |Sigma| + M (for novel merges): OK')

## Pre-training pipeline: AdamW + warmup + cosine + grad-clip on a tiny GPT

We assemble Ch.14 (AdamW), Ch.23 (transformer block), Ch.25 (NTP loss + causal mask), and Ch.26 (tokenizer) into the pre-training loop used by every modern decoder-only LM. The recipe:

`corpus -> tokenize -> batch (B,T) -> transformer -> CE loss -> backprop -> clip(global-norm) -> AdamW(eta_t)`.

We train a 2-layer character-level GPT (~18K params) end-to-end and watch the loss drop from `log V` to well below it.

In [ ]:
import numpy as np, math, time
np.random.seed(0)

# --- Tiny corpus + character tokenizer (Ch. 26) -----------------------------
text = (
    'the quick brown fox jumps over the lazy dog. '
    'the lazy dog barks at the quick brown fox. '
    'a fox in the woods is a clever fox indeed. '
    'the brown dog and the quick fox are friends. '
    'jumping foxes and barking dogs play in the woods. '
) * 4
chars = sorted(set(text))
V = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}
data = np.array([stoi[c] for c in text], dtype=np.int64)
print(f'vocab V={V}  corpus length N={len(data)}')
print('vocab:', ''.join(chars))

### Batching and next-token targets (Ch. 25)

For context length $T$, sample $B$ random windows. The target at position $i$ is the token at position $i+1$ (a one-position shift). Each batch is shape `(B,T)` for both inputs and targets.

In [ ]:
T = 16
def get_batch(B, rng):
    ix = rng.integers(0, len(data) - T - 1, size=B)
    x = np.stack([data[i:i+T]     for i in ix])
    y = np.stack([data[i+1:i+T+1] for i in ix])
    return x, y

rng_demo = np.random.default_rng(0)
xb, yb = get_batch(2, rng_demo)
print('xb shape', xb.shape, ' yb shape', yb.shape)
print('input :', repr(''.join(itos[i] for i in xb[0])))
print('target:', repr(''.join(itos[i] for i in yb[0])))

### Tiny GPT architecture (Ch. 23, scaled down)

$L=2$ pre-norm blocks, single-head attention, $d=32$, $d_{\text{ff}}=64$. Output projection ties to the input embedding $E$. Total parameters $\sim$ 18K, all in fp32 numpy.

In [ ]:
d, H, dff, L = 32, 1, 64, 2

def init_linear(fan_in, fan_out, scale=1.0):
    return (np.random.randn(fan_in, fan_out) * (scale / math.sqrt(fan_in))).astype(np.float32)

params = {}
params['E'] = (np.random.randn(V, d) * 0.02).astype(np.float32)
params['P'] = (np.random.randn(T, d) * 0.02).astype(np.float32)
for l in range(L):
    params[f'ln1_g{l}'] = np.ones(d, dtype=np.float32)
    params[f'ln1_b{l}'] = np.zeros(d, dtype=np.float32)
    params[f'Wq{l}'] = init_linear(d, d, scale=0.5)
    params[f'Wk{l}'] = init_linear(d, d, scale=0.5)
    params[f'Wv{l}'] = init_linear(d, d, scale=0.5)
    params[f'Wo{l}'] = init_linear(d, d, scale=0.5)
    params[f'ln2_g{l}'] = np.ones(d, dtype=np.float32)
    params[f'ln2_b{l}'] = np.zeros(d, dtype=np.float32)
    params[f'W1{l}'] = init_linear(d, dff, scale=0.5)
    params[f'b1{l}'] = np.zeros(dff, dtype=np.float32)
    params[f'W2{l}'] = init_linear(dff, d, scale=0.5)
    params[f'b2{l}'] = np.zeros(d, dtype=np.float32)
params['lng'] = np.ones(d, dtype=np.float32)
params['lnb'] = np.zeros(d, dtype=np.float32)
n_params = sum(v.size for v in params.values())
print(f'tiny-GPT parameter count: {n_params}')

# --- helpers ---------------------------------------------------------------
def layernorm(x, g, b, eps=1e-5):
    mu  = x.mean(-1, keepdims=True)
    var = x.var(-1, keepdims=True)
    inv = 1.0 / np.sqrt(var + eps)
    xhat = (x - mu) * inv
    y = xhat * g + b
    return y, (x, mu, var, inv, xhat, g)

def layernorm_back(dy, cache):
    x, mu, var, inv, xhat, g = cache
    D = x.shape[-1]
    dxhat = dy * g
    dvar  = (dxhat * (x - mu) * (-0.5) * inv**3).sum(-1, keepdims=True)
    dmu   = (dxhat * (-inv)).sum(-1, keepdims=True) + dvar * (-2.0/D) * (x - mu).sum(-1, keepdims=True)
    dx    = dxhat * inv + dvar * 2.0*(x - mu)/D + dmu/D
    dg    = (dy * xhat).sum(axis=tuple(range(dy.ndim - 1)))
    db    = dy.sum(axis=tuple(range(dy.ndim - 1)))
    return dx, dg, db

def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

causal_mask = (np.triu(np.ones((T, T), dtype=np.float32), k=1) * -1e9)

def forward(p, x):
    B, Tlen = x.shape
    assert Tlen == T
    h = p['E'][x] + p['P'][None, :T]
    cache = {'x': x, 'blocks': []}
    for l in range(L):
        h_in = h
        h_n, c_ln1 = layernorm(h, p[f'ln1_g{l}'], p[f'ln1_b{l}'])
        Q = h_n @ p[f'Wq{l}']
        K = h_n @ p[f'Wk{l}']
        Vv = h_n @ p[f'Wv{l}']
        scores = (Q @ K.transpose(0,2,1)) / math.sqrt(d) + causal_mask[None]
        attn = softmax(scores, axis=-1)
        ctx = attn @ Vv
        attn_out = ctx @ p[f'Wo{l}']
        h2 = h_in + attn_out
        h2_n, c_ln2 = layernorm(h2, p[f'ln2_g{l}'], p[f'ln2_b{l}'])
        z1 = h2_n @ p[f'W1{l}'] + p[f'b1{l}']
        a1 = np.maximum(z1, 0)  # ReLU
        z2 = a1 @ p[f'W2{l}'] + p[f'b2{l}']
        h = h2 + z2
        cache['blocks'].append(dict(h_in=h_in, c_ln1=c_ln1, h_n=h_n, Q=Q, K=K, V=Vv,
                                    attn=attn, ctx=ctx, h2=h2, c_ln2=c_ln2,
                                    h2_n=h2_n, z1=z1, a1=a1))
    h_n_final, c_lnf = layernorm(h, p['lng'], p['lnb'])
    logits = h_n_final @ p['E'].T
    cache['c_lnf'] = c_lnf
    cache['h_n_final'] = h_n_final
    return logits, cache

# sanity: forward pass shape + initial loss ~ log V
_x, _y = get_batch(4, np.random.default_rng(1))
_logits, _ = forward(params, _x)
_p = softmax(_logits, axis=-1)
_init_loss = float(-np.log(_p[np.arange(4)[:,None], np.arange(T)[None,:], _y] + 1e-12).mean())
print(f'forward logits {_logits.shape}, init loss {_init_loss:.3f}, log V = {math.log(V):.3f}')

### Backward pass + AdamW + warmup-cosine + global-norm clipping

We hand-code the chain rule so every gradient is auditable, then implement Ch.14's AdamW with decoupled weight decay (no decay on embeddings, biases, or LayerNorm scales). The schedule is linear warmup over $W=50$ steps to $\eta_{\max}=3\!\times\!10^{-3}$, followed by cosine decay to $\eta_{\min}=3\!\times\!10^{-4}$. Global gradient norm is clipped at $c=1.0$.

In [ ]:
def loss_and_grad_logits(logits, y):
    B, Tlen, Vl = logits.shape
    probs = softmax(logits, axis=-1)
    nll = -np.log(probs[np.arange(B)[:,None], np.arange(Tlen)[None,:], y] + 1e-12)
    loss = float(nll.mean())
    dlogits = probs.copy()
    dlogits[np.arange(B)[:,None], np.arange(Tlen)[None,:], y] -= 1.0
    dlogits /= (B * Tlen)
    return loss, dlogits

def backward(p, cache, dlogits):
    grads = {k: np.zeros_like(v) for k, v in p.items()}
    h_n_final = cache['h_n_final']
    B, Tlen, Vl = dlogits.shape
    grads['E'] += (h_n_final.reshape(-1, d).T @ dlogits.reshape(-1, Vl)).T
    dh = dlogits @ p['E']
    dh_final, dlng, dlnb = layernorm_back(dh, cache['c_lnf'])
    grads['lng'] += dlng; grads['lnb'] += dlnb
    dh = dh_final
    for l in reversed(range(L)):
        bc = cache['blocks'][l]
        # h_out = h2 + z2
        dh2 = dh.copy()
        dz2 = dh.copy()
        grads[f'W2{l}'] += bc['a1'].reshape(-1, dff).T @ dz2.reshape(-1, d)
        grads[f'b2{l}'] += dz2.reshape(-1, d).sum(0)
        da1 = dz2 @ p[f'W2{l}'].T
        dz1 = da1 * (bc['z1'] > 0)
        grads[f'W1{l}'] += bc['h2_n'].reshape(-1, d).T @ dz1.reshape(-1, dff)
        grads[f'b1{l}'] += dz1.reshape(-1, dff).sum(0)
        dh2_n = dz1 @ p[f'W1{l}'].T
        dh2_a, dln2g, dln2b = layernorm_back(dh2_n, bc['c_ln2'])
        grads[f'ln2_g{l}'] += dln2g; grads[f'ln2_b{l}'] += dln2b
        dh2 += dh2_a
        # h2 = h_in + attn_out
        dh_in = dh2.copy()
        dattn_out = dh2.copy()
        grads[f'Wo{l}'] += bc['ctx'].reshape(-1, d).T @ dattn_out.reshape(-1, d)
        dctx = dattn_out @ p[f'Wo{l}'].T
        dattn = dctx @ bc['V'].transpose(0,2,1)
        dV = bc['attn'].transpose(0,2,1) @ dctx
        a = bc['attn']
        s = (dattn * a).sum(axis=-1, keepdims=True)
        dscores = (a * (dattn - s)) / math.sqrt(d)
        dQ = dscores @ bc['K']
        dK = dscores.transpose(0,2,1) @ bc['Q']
        grads[f'Wq{l}'] += bc['h_n'].reshape(-1, d).T @ dQ.reshape(-1, d)
        grads[f'Wk{l}'] += bc['h_n'].reshape(-1, d).T @ dK.reshape(-1, d)
        grads[f'Wv{l}'] += bc['h_n'].reshape(-1, d).T @ dV.reshape(-1, d)
        dh_n = dQ @ p[f'Wq{l}'].T + dK @ p[f'Wk{l}'].T + dV @ p[f'Wv{l}'].T
        dh_in_a, dln1g, dln1b = layernorm_back(dh_n, bc['c_ln1'])
        grads[f'ln1_g{l}'] += dln1g; grads[f'ln1_b{l}'] += dln1b
        dh_in += dh_in_a
        dh = dh_in
    # token + position embeddings
    np.add.at(grads['E'], cache['x'], dh)
    grads['P'] += dh.sum(axis=0)
    return grads

class AdamW:
    def __init__(self, params, beta1=0.9, beta2=0.95, eps=1e-8, wd=0.01):
        self.m = {k: np.zeros_like(v) for k, v in params.items()}
        self.v = {k: np.zeros_like(v) for k, v in params.items()}
        self.b1, self.b2, self.eps, self.wd = beta1, beta2, eps, wd
        self.t = 0
        # no decay on embeddings, biases, or LayerNorm scales (Llama/GPT-2 convention)
        self.no_wd = {k for k in params if k.startswith(('ln','b1','b2','lng','lnb','P','E'))}
    def step(self, params, grads, lr):
        self.t += 1
        bc1, bc2 = 1 - self.b1**self.t, 1 - self.b2**self.t
        for k in params:
            g = grads[k]
            self.m[k] = self.b1*self.m[k] + (1-self.b1)*g
            self.v[k] = self.b2*self.v[k] + (1-self.b2)*(g*g)
            update = (self.m[k]/bc1) / (np.sqrt(self.v[k]/bc2) + self.eps)
            if k not in self.no_wd:
                update = update + self.wd * params[k]
            params[k] -= lr * update

def lr_schedule(t, W, T_total, eta_max, eta_min):
    if t < W:
        return eta_max * (t + 1) / W       # linear warmup
    progress = (t - W) / max(1, T_total - W)
    return eta_min + 0.5*(eta_max - eta_min)*(1 + math.cos(math.pi*progress))

def clip_global_norm(grads, c=1.0):
    sq = sum((g*g).sum() for g in grads.values())
    norm = float(np.sqrt(sq))
    scale = min(1.0, c / (norm + 1e-12))
    if scale < 1.0:
        for k in grads:
            grads[k] *= scale
    return norm

# Quick gradient sanity check: finite-difference one entry of Wq0
rng_chk = np.random.default_rng(7)
_xc, _yc = get_batch(2, rng_chk)
_logits_c, _cache_c = forward(params, _xc)
_loss_c, _dl = loss_and_grad_logits(_logits_c, _yc)
_g = backward(params, _cache_c, _dl)
_eps = 1e-3; _i, _j = 0, 0
_orig = params['Wq0'][_i, _j]
params['Wq0'][_i, _j] = _orig + _eps
_lp, _ = loss_and_grad_logits(forward(params, _xc)[0], _yc)
params['Wq0'][_i, _j] = _orig - _eps
_lm, _ = loss_and_grad_logits(forward(params, _xc)[0], _yc)
params['Wq0'][_i, _j] = _orig
print(f'analytic dL/dWq0[0,0] = {_g["Wq0"][_i,_j]:+.5f}  finite-diff = {(_lp-_lm)/(2*_eps):+.5f}')

### Training loop

600 steps, batch 16. We log loss, learning rate, and gradient norm every 50 steps. Initial loss should be near $\log V$; trained loss should be well below it.

In [ ]:
opt = AdamW(params)
B, W_steps, T_train = 16, 50, 600
eta_max, eta_min = 3e-3, 3e-4
rng = np.random.default_rng(0)
losses, lrs, gnorms = [], [], []
t0 = time.time()
for step in range(T_train):
    x, y = get_batch(B, rng)
    logits, cache = forward(params, x)
    loss, dlogits = loss_and_grad_logits(logits, y)
    grads = backward(params, cache, dlogits)
    gn = clip_global_norm(grads, c=1.0)
    lr = lr_schedule(step, W_steps, T_train, eta_max, eta_min)
    opt.step(params, grads, lr)
    losses.append(loss); lrs.append(lr); gnorms.append(gn)
    if step % 50 == 0 or step == T_train - 1:
        print(f'step {step:4d}  loss {loss:.4f}  lr {lr:.5f}  gnorm {gn:.3f}')
elapsed = time.time() - t0
print(f'\ntraining done in {elapsed:.1f} s   init loss {losses[0]:.3f}  -> final loss {losses[-1]:.3f}')

# Plot or fall back to a printed table
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(10, 3))
    ax[0].plot(losses); ax[0].axhline(math.log(V), color='r', ls='--', label='log V (random)')
    ax[0].set_xlabel('step'); ax[0].set_ylabel('CE loss'); ax[0].legend()
    ax[1].plot(lrs); ax[1].set_xlabel('step'); ax[1].set_ylabel('learning rate')
    ax[1].set_title('warmup + cosine')
    plt.tight_layout(); plt.show()
except Exception as e:
    print('matplotlib unavailable, printing table instead:', e)
    for s in range(0, T_train, 100):
        print(f'  step {s:4d}  loss {losses[s]:.4f}  lr {lrs[s]:.5f}')

### Inference: greedy and temperature sampling

We pad/truncate the prompt to the model's context length $T$, take the next-token distribution from the last position, and either argmax (greedy) or sample with temperature $\tau$. The trained tiny GPT should reproduce the corpus's vocabulary and style.

In [ ]:
def encode(s):
    return np.array([stoi[c] for c in s if c in stoi], dtype=np.int64)

def generate(prompt, n_new=50, temperature=1.0, greedy=False, seed=0):
    rng_g = np.random.default_rng(seed)
    ids = list(encode(prompt))
    out = list(ids)
    for _ in range(n_new):
        ctx = out[-T:]
        if len(ctx) < T:
            ctx = [0]*(T - len(ctx)) + ctx
        x = np.array(ctx, dtype=np.int64)[None, :]
        logits, _ = forward(params, x)
        last = logits[0, -1] / max(1e-6, temperature)
        if greedy:
            nxt = int(last.argmax())
        else:
            p = softmax(last)
            nxt = int(rng_g.choice(V, p=p))
        out.append(nxt)
    return ''.join(itos[i] for i in out)

print('--- greedy ---')
print(generate('the quick brown ', n_new=80, greedy=True))
print('\n--- temperature 1.0 ---')
print(generate('the quick brown ', n_new=80, temperature=1.0, seed=0))

### Takeaways and scaling

- The loop above is *byte-identical* in control flow to the one used to pre-train GPT-3 / Llama. Only $V$, $T$, $d$, $L$, $N$, $D$ change.
- Warmup keeps the first ~50 steps stable; cosine then decays $\eta$ so the final iterates settle into a sharper minimum.
- Clipping at $c=1.0$ caps the per-step second-order term in the descent lemma, preventing rare large gradients from blowing up training.
- Chinchilla (Hoffmann 2022) tells us how much $D$ to use: scale $N$ and $D$ proportionally for fixed compute $C \approx 6 N D$ FLOPs.

In [ ]:
final_loss = losses[-1]
perplexity = math.exp(final_loss)
print(f'final CE loss : {final_loss:.4f} nats')
print(f'random  loss  : {math.log(V):.4f} nats  (= log V)')
print(f'reduction     : {math.log(V) - final_loss:.4f} nats below uniform')
print(f'perplexity    : {perplexity:.3f}  (uniform would be {V:.1f})')
print(f'params        : {sum(v.size for v in params.values())}')
print(f'tokens seen   : {B * T_train * T} = B*T_train*T  (Chinchilla compute proxy)')

# Block G — Post-training


## Post-training the tiny GPT: SFT, RM, PPO, DPO

We continue the tiny GPT of Chapter 27. Pre-training only optimizes next-token likelihood on a corpus, so the model is fluent but does not follow instructions. The post-training pipeline:

1. **SFT** on a handful of (prompt, response) pairs $\Rightarrow$ obtain $\pi_{\mathrm{ref}}$.
2. Train a **Bradley-Terry reward model** on preference triples $(x, y_w, y_l)$.
3. **PPO** sketch: compute the clipped surrogate on one batch.
4. **DPO** fine-tune $\pi_\theta$ from $\pi_{\mathrm{ref}}$ and verify the preference ratio increases.

All numpy + stdlib; CPU runnable in under 5 minutes.


In [ ]:
import numpy as np, math, time
np.random.seed(0)

# ----- Tiny corpus (char-level) — extends Chapter 27's toy corpus -----
corpus = (
    'the cat sat on the mat. '
    'the dog sat on the log. '
    'the cat ate the fish. '
    'the dog ate the bone. '
    'a cat is a pet. a dog is a pet. '
    'fish swim in the water. birds fly in the sky. '
) * 8

chars = sorted(set(corpus))
V = len(chars)
stoi = {c:i for i,c in enumerate(chars)}
itos = {i:c for c,i in stoi.items()}
data = np.array([stoi[c] for c in corpus], dtype=np.int64)
print('vocab size V =', V, '  corpus tokens =', len(data))

# ----- Tiny GPT (numpy) — 1 layer, single-head attention, short context -----
# Architecture: token+pos embed -> attn -> MLP -> unembed (tied weights).
# We only train the token-embedding matrix (sufficient for this demo, since
# the unembed is tied so updating tok updates both directions).

T   = 16    # context length
d   = 32    # model dim
d_ff= 64

def init(shape, scale=0.02):
    return np.random.randn(*shape).astype(np.float32) * scale

params = {
    'tok': init((V, d)),
    'pos': init((T, d)),
    'Wq':  init((d, d)),
    'Wk':  init((d, d)),
    'Wv':  init((d, d)),
    'Wo':  init((d, d)),
    'W1':  init((d, d_ff)),
    'b1':  np.zeros(d_ff, dtype=np.float32),
    'W2':  init((d_ff, d)),
    'b2':  np.zeros(d, dtype=np.float32),
}

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def forward(idx, p):
    # idx: (B, T) int64
    B, Tloc = idx.shape
    x = p['tok'][idx] + p['pos'][:Tloc]              # (B,T,d)
    Q = x @ p['Wq']; K = x @ p['Wk']; Vv = x @ p['Wv']
    att = Q @ K.transpose(0,2,1) / math.sqrt(d)      # (B,T,T)
    mask = np.triu(np.ones((Tloc,Tloc), dtype=bool), 1)
    att = np.where(mask, -1e9, att)
    A = softmax(att, axis=-1)
    h = (A @ Vv) @ p['Wo'] + x                       # residual
    h_mlp = np.maximum(h @ p['W1'] + p['b1'], 0) @ p['W2'] + p['b2']
    h = h + h_mlp                                     # residual
    logits = h @ p['tok'].T                           # tied unembed
    return logits, h

def loss_and_grad(idx_x, idx_y, p, lr_mask=None):
    # NTP loss with optional per-position mask (1 = include in loss).
    # Only computes gradient w.r.t. the token-embedding matrix (tied unembed).
    B, Tloc = idx_x.shape
    logits, h = forward(idx_x, p)
    P = softmax(logits, axis=-1)
    if lr_mask is None:
        lr_mask = np.ones_like(idx_y, dtype=np.float32)
    n = lr_mask.sum() + 1e-8
    onehot = np.zeros_like(P)
    bidx = np.arange(B)[:,None]; tidx = np.arange(Tloc)[None,:]
    onehot[bidx, tidx, idx_y] = 1.0
    nll = -np.log(P[bidx, tidx, idx_y] + 1e-9)
    loss = float((nll * lr_mask).sum() / n)
    dlogits = (P - onehot) * lr_mask[..., None] / n   # (B,T,V)
    grads = {k: np.zeros_like(v) for k,v in p.items()}
    dlogits_flat = dlogits.reshape(-1, V)             # (B*T, V)
    h_flat       = h.reshape(-1, d)                   # (B*T, d)
    grads['tok'] = dlogits_flat.T @ h_flat            # (V, d) — unembed grad
    return loss, grads, P, h

def make_batches(data, B, T, n_batches):
    starts = np.random.randint(0, len(data) - T - 1, size=(n_batches, B))
    Xs = np.stack([np.stack([data[s:s+T] for s in row]) for row in starts])
    Ys = np.stack([np.stack([data[s+1:s+T+1] for s in row]) for row in starts])
    return Xs, Ys

# Pre-training loop — updates tied tok-embeddings only.
B = 16
lr = 0.5
t0 = time.time()
loss_hist = []
Xs, Ys = make_batches(data, B, T, 200)
for step in range(200):
    x, y = Xs[step], Ys[step]
    loss, grads, _, _ = loss_and_grad(x, y, params)
    params['tok'] -= lr * grads['tok']
    loss_hist.append(loss)
print(f'pre-train: step 0 loss = {loss_hist[0]:.3f}   step 199 loss = {loss_hist[-1]:.3f}')
print(f'pre-train wall time: {time.time()-t0:.2f}s')
pi_base_tok = params['tok'].copy()


## SFT — supervised fine-tuning

Tiny dataset of (prompt, response) pairs in the same toy style. Loss only fires on the response tokens (prompt-token mask = 0). Result: the *reference model* $\pi_{\mathrm{ref}}$.


In [ ]:
sft_pairs = [
    ('the cat ',     'sat on the mat. '),
    ('the dog ',     'sat on the log. '),
    ('a cat is ',    'a pet. '),
    ('a dog is ',    'a pet. '),
    ('fish ',        'swim in the water. '),
    ('birds ',       'fly in the sky. '),
    ('the cat ate ', 'the fish. '),
    ('the dog ate ', 'the bone. '),
]

def encode_pair(prompt, response, T):
    full = (prompt + response)[:T]
    pad  = T - len(full)
    full = full + ' ' * pad
    ids  = np.array([stoi.get(c, 0) for c in full], dtype=np.int64)
    # mask: 1 on response tokens (after the prompt)
    p_len = min(len(prompt), T)
    mask  = np.zeros(T, dtype=np.float32)
    mask[p_len:p_len+len(response)] = 1.0
    return ids, mask

def make_sft_batch():
    xs, ys, ms = [], [], []
    for p, r in sft_pairs:
        ids, mask = encode_pair(p, r, T+1)
        xs.append(ids[:-1]); ys.append(ids[1:])
        ms.append(mask[1:])  # mask aligned with prediction targets
    return np.stack(xs), np.stack(ys), np.stack(ms)

x_sft, y_sft, m_sft = make_sft_batch()
loss0, _, _, _ = loss_and_grad(x_sft, y_sft, params, lr_mask=m_sft)
print(f'SFT loss before: {loss0:.4f}')

sft_lr = 0.3
for step in range(120):
    loss, grads, _, _ = loss_and_grad(x_sft, y_sft, params, lr_mask=m_sft)
    params['tok'] -= sft_lr * grads['tok']
loss1, _, _, _ = loss_and_grad(x_sft, y_sft, params, lr_mask=m_sft)
print(f'SFT loss after : {loss1:.4f}')
sft_loss_drop = loss0 - loss1
print(f'SFT loss drop  : {sft_loss_drop:+.4f}')

# Snapshot pi_ref
pi_ref_tok = params['tok'].copy()

def sequence_logprob(prompt, response, p_tok):
    """Sum log-prob of response chars conditioned on prompt under params['tok']=p_tok."""
    saved = params['tok']; params['tok'] = p_tok
    text = (prompt + response)[:T]
    ids = np.array([stoi.get(c,0) for c in text.ljust(T)], dtype=np.int64)[None,:]
    logits, _ = forward(ids, params)
    P = softmax(logits, axis=-1)[0]
    p_len = len(prompt)
    lp = 0.0
    for t in range(p_len, min(len(text), T)-1):
        lp += math.log(P[t-1, stoi[text[t]]] + 1e-12)
    params['tok'] = saved
    return lp

# Quick generation-quality sanity check
for prompt in ['the cat ', 'a dog is ']:
    lp_good = sequence_logprob(prompt, 'sat on the mat. ' if 'cat' in prompt else 'a pet. ', pi_ref_tok)
    lp_bad  = sequence_logprob(prompt, 'fly in the sky. ', pi_ref_tok)
    print(f'  prompt={prompt!r:14s}  logp(target)={lp_good:+.2f}  logp(off-topic)={lp_bad:+.2f}')


## Preference dataset + Bradley-Terry reward model

Five preference triples $(x, y_w, y_l)$ where $y_w$ is the better continuation.

The reward model is a single linear head on top of the (frozen) tiny GPT's last hidden state, summed over the response tokens. Loss: $-\log\sigma(r_\phi(x,y_w)-r_\phi(x,y_l))$.


In [ ]:
pref_triples = [
    ('the cat ',  'sat on the mat. ',  'fly in the sky. '),
    ('the dog ',  'sat on the log. ',  'swim in the water. '),
    ('a cat is ', 'a pet. ',            'a fish. '),
    ('fish ',     'swim in the water. ','sat on the mat. '),
    ('birds ',    'fly in the sky. ',  'sat on the log. '),
]

def hidden_for(prompt, response, p_tok):
    saved = params['tok']; params['tok'] = p_tok
    text = (prompt + response)[:T].ljust(T)
    ids  = np.array([stoi.get(c,0) for c in text], dtype=np.int64)[None,:]
    _, h = forward(ids, params)
    p_len = len(prompt)
    # Sum hidden states over response token positions
    pooled = h[0, p_len:p_len+len(response)].mean(axis=0)
    params['tok'] = saved
    return pooled  # (d,)

# Reward head: r_phi(x,y) = w . pooled_hidden + b
w_r = np.zeros(d, dtype=np.float32)
b_r = np.float32(0.0)

def rm_score(prompt, response):
    h = hidden_for(prompt, response, pi_ref_tok)
    return float(w_r @ h + b_r), h

rm_lr = 0.2
rm_loss_hist = []
for step in range(150):
    grad_w = np.zeros_like(w_r); grad_b = 0.0; tot = 0.0
    for x_p, y_w, y_l in pref_triples:
        s_w, h_w = rm_score(x_p, y_w)
        s_l, h_l = rm_score(x_p, y_l)
        diff = s_w - s_l
        # loss = -log sigmoid(diff)
        sig = 1.0 / (1.0 + math.exp(-diff))
        tot += -math.log(sig + 1e-12)
        # d/d(diff) of -log sigma(diff) = -(1-sigma) = sigma - 1
        g = (sig - 1.0)
        grad_w += g * (h_w - h_l)
        grad_b += g * 0.0  # b cancels (it's the same scalar in both terms)
    w_r -= rm_lr * grad_w / len(pref_triples)
    rm_loss_hist.append(tot / len(pref_triples))

print(f'RM loss: start {rm_loss_hist[0]:.4f}  end {rm_loss_hist[-1]:.4f}')
rm_loss_drop = rm_loss_hist[0] - rm_loss_hist[-1]
print(f'RM loss drop: {rm_loss_drop:+.4f}')

# Verify ranking on training data
correct = 0
for x_p, y_w, y_l in pref_triples:
    s_w, _ = rm_score(x_p, y_w); s_l, _ = rm_score(x_p, y_l)
    correct += int(s_w > s_l)
print(f'RM ranks y_w > y_l on {correct}/{len(pref_triples)} training triples')


## RLHF / PPO sketch

We do not run a full PPO loop (would require a value head, GAE, multiple epochs). Instead we demonstrate the **clipped surrogate** on a small batch:
$$\mathcal{L}^{\mathrm{PPO}} = \mathbb{E}\big[\min(\rho_t \hat A_t,\ \mathrm{clip}(\rho_t,1-\varepsilon,1+\varepsilon)\hat A_t)\big].$$
We synthesize $\rho_t$ values around 1 and use the RM scores (mean-centered) as advantages.


In [ ]:
epsilon = 0.2
# Pretend we sampled 5 trajectories (one per pref triple). Use y_w as the action.
adv = []
for x_p, y_w, y_l in pref_triples:
    s_w, _ = rm_score(x_p, y_w)
    s_l, _ = rm_score(x_p, y_l)
    adv.append(s_w)
    adv.append(s_l)
adv = np.array(adv, dtype=np.float32)
adv_centered = adv - adv.mean()                    # group-mean baseline (GRPO-style)
adv_norm     = adv_centered / (adv.std() + 1e-6)

# Importance ratios — synthesize a small spread around 1.
rho = np.array([0.85, 1.10, 0.95, 1.30, 1.05, 0.70, 0.98, 1.22, 1.18, 0.80], dtype=np.float32)

unclipped = rho * adv_norm
clipped   = np.clip(rho, 1-epsilon, 1+epsilon) * adv_norm
L_ppo     = np.minimum(unclipped, clipped).mean()
print('importance ratios   :', np.round(rho, 3))
print('group-norm advantages:', np.round(adv_norm, 3))
print('unclipped surrogate :', np.round(unclipped, 3))
print('clipped surrogate   :', np.round(clipped, 3))
print(f'PPO objective (mean min): {L_ppo:.4f}')
print('Clipping fired on positions where rho was outside [%.2f, %.2f].' % (1-epsilon, 1+epsilon))


## DPO loss + derivation summary

From the closed-form solution to the KL-regularized RLHF objective,
$\pi^*(y|x) \propto \pi_{\mathrm{ref}}(y|x)\exp(r(x,y)/\beta)$, one inverts to recover $r$ in terms of $\pi^*$ and $\pi_{\mathrm{ref}}$. Substituting into Bradley-Terry, the partition function $Z(x)$ cancels between $y_w$ and $y_l$, leaving
$$\mathcal{L}_{\mathrm{DPO}}(\theta) = -\log\sigma\!\Big(\beta\big[\log\tfrac{\pi_\theta(y_w|x)}{\pi_{\mathrm{ref}}(y_w|x)} - \log\tfrac{\pi_\theta(y_l|x)}{\pi_{\mathrm{ref}}(y_l|x)}\big]\Big).$$
We implement this directly: gradient w.r.t.\ $\pi_\theta$ pushes the preferred response higher and the rejected one lower, *relative to the reference*.


In [ ]:
beta = 0.1

def seq_logp_under(p_tok, prompt, response):
    return sequence_logprob(prompt, response, p_tok)

def dpo_loss(theta_tok):
    losses = []
    for x_p, y_w, y_l in pref_triples:
        lp_th_w = seq_logp_under(theta_tok, x_p, y_w)
        lp_th_l = seq_logp_under(theta_tok, x_p, y_l)
        lp_rf_w = seq_logp_under(pi_ref_tok, x_p, y_w)
        lp_rf_l = seq_logp_under(pi_ref_tok, x_p, y_l)
        z = beta * ((lp_th_w - lp_rf_w) - (lp_th_l - lp_rf_l))
        # numerically stable -log sigmoid(z) = softplus(-z)
        losses.append(math.log1p(math.exp(-z)) if z > -50 else -z)
    return float(np.mean(losses))

def pref_ratio(p_tok):
    ratios = []
    for x_p, y_w, y_l in pref_triples:
        lp_w = seq_logp_under(p_tok, x_p, y_w)
        lp_l = seq_logp_under(p_tok, x_p, y_l)
        ratios.append(lp_w - lp_l)   # log of ratio
    return float(np.mean(ratios))

# Initialize theta := pi_ref (so DPO starts from the SFT model)
theta_tok = pi_ref_tok.copy()
params['tok'] = theta_tok

ratio_ref = pref_ratio(pi_ref_tok)
print(f'mean log[pi(y_w)/pi(y_l)] under pi_ref (SFT): {ratio_ref:+.4f}')

loss0 = dpo_loss(theta_tok)
print(f'DPO loss before: {loss0:.4f}')

# Analytic gradient of DPO loss w.r.t. token-embedding (tied unembed) parameters.
#   L = -log sigma(z),  z = beta * [ (lp_th_w - lp_rf_w) - (lp_th_l - lp_rf_l) ]
#   dL/dz = -(1 - sigma(z)) = -sigma(-z) =: -s_neg
#   dL/d(lp_th_w) = -beta * s_neg ;  dL/d(lp_th_l) = +beta * s_neg
#   d(log p(y_t|...))/d(logit_v) = onehot_t(v) - P_t(v)
# So: dL/d(logit_v at position t-1) =
#       -beta * s_neg * (onehot - P)   (for y_w response tokens)
#       +beta * s_neg * (onehot - P)   (for y_l response tokens)
# i.e. = sign * beta * s_neg * (P - onehot) with sign=+1 for y_w, -1 for y_l.

dpo_lr = 2.0
for step in range(150):
    grad_tok = np.zeros_like(theta_tok)
    for x_p, y_w, y_l in pref_triples:
        lp_th_w = seq_logp_under(theta_tok, x_p, y_w)
        lp_th_l = seq_logp_under(theta_tok, x_p, y_l)
        lp_rf_w = seq_logp_under(pi_ref_tok, x_p, y_w)
        lp_rf_l = seq_logp_under(pi_ref_tok, x_p, y_l)
        z = beta * ((lp_th_w - lp_rf_w) - (lp_th_l - lp_rf_l))
        s_neg = 1.0 / (1.0 + math.exp(z)) if z < 50 else 0.0
        for tgt_text, sign in [(x_p+y_w, +1.0), (x_p+y_l, -1.0)]:
            text = tgt_text[:T].ljust(T)
            ids  = np.array([stoi.get(c,0) for c in text], dtype=np.int64)[None,:]
            params['tok'] = theta_tok
            logits, h = forward(ids, params)
            P = softmax(logits, axis=-1)[0]
            p_len = len(x_p)
            for t in range(p_len, min(len(tgt_text), T)-1):
                tgt = stoi[tgt_text[t]]
                onehot = np.zeros(V, dtype=np.float32); onehot[tgt] = 1.0
                # gradient of L (to be subtracted in SGD step):
                #   for y_w (sign=+1): dL/dlogit = -beta*s_neg*(onehot-P) = +beta*s_neg*(P-onehot)
                #   for y_l (sign=-1): dL/dlogit = +beta*s_neg*(onehot-P) = -beta*s_neg*(P-onehot)
                # Combined: dL/dlogit = sign * beta * s_neg * (P - onehot)
                dlogit = sign * beta * s_neg * (P[t-1] - onehot)
                # logits = h @ tok.T  =>  d(logit_v)/d(tok[v,:]) = h[t-1]
                grad_tok += np.outer(dlogit, h[0, t-1])
    theta_tok -= dpo_lr * grad_tok / len(pref_triples)

loss1 = dpo_loss(theta_tok)
print(f'DPO loss after : {loss1:.4f}')
dpo_loss_drop = loss0 - loss1
print(f'DPO loss drop  : {dpo_loss_drop:+.4f}')

ratio_dpo = pref_ratio(theta_tok)
pref_ratio_change = ratio_dpo - ratio_ref
print(f'mean log[pi(y_w)/pi(y_l)] under pi_theta (DPO): {ratio_dpo:+.4f}')
print(f'preference-ratio change (DPO - SFT) : {pref_ratio_change:+.4f}')
assert pref_ratio_change > 0, 'DPO should increase the preference ratio toward y_w'
pi_dpo_tok = theta_tok.copy()


## Takeaways

- **SFT** is just MLE on a curated (prompt, response) corpus with prompt-token loss masked. It produces $\pi_{\mathrm{ref}}$, the initialization for everything that follows.
- **PPO** maximizes expected reward minus a KL penalty via a clipped policy-gradient surrogate; it requires a separately-trained Bradley-Terry reward model and on-policy sampling.
- **GRPO** drops the value head: replace $\hat A$ with a group-normalized empirical advantage. Cheaper, used by DeepSeek-R1.
- **DPO** solves the KL-regularized RLHF problem in closed form, inverts $r$, and pushes through Bradley-Terry. The partition function cancels, leaving a simple supervised-style loss on $(x, y_w, y_l)$ triples that requires no reward model and no on-policy sampling.
- All three tilt the policy toward preferred completions while the KL leash to $\pi_{\mathrm{ref}}$ preserves the capabilities laid down during pre-training (Chapter 27).


In [ ]:
# Held-out prompt comparison: pre-train vs SFT vs DPO
held_out = 'the cat '
candidates = ['sat on the mat. ', 'fly in the sky. ', 'a pet. ']

def best_continuation(p_tok):
    return max(candidates, key=lambda c: sequence_logprob(held_out, c, p_tok))

def report(name, p_tok):
    scores = {c: sequence_logprob(held_out, c, p_tok) for c in candidates}
    best   = max(scores, key=scores.get)
    print(f'  [{name:>8s}] best continuation: {best!r}')
    for c, s in scores.items():
        print(f'      logp({c!r:25s}) = {s:+.3f}')

# Reset to base for a clean comparison
params['tok'] = pi_base_tok
print(f'Held-out prompt: {held_out!r}')
report('pre-train', pi_base_tok)
report('SFT',       pi_ref_tok)
report('DPO',       pi_dpo_tok)

print('\nSummary')
print(f'  SFT loss drop          : {sft_loss_drop:+.4f}')
print(f'  RM  loss drop          : {rm_loss_drop:+.4f}')
print(f'  DPO loss drop          : {dpo_loss_drop:+.4f}')
print(f'  pref-ratio change      : {pref_ratio_change:+.4f}  (DPO > SFT means y_w more preferred)')


# Block H — Reinforcement Learning


## MDP foundations: Bellman equations, value iteration, tabular Q-learning

We implement a $4{\times}4$ GridWorld MDP, run value iteration to a sup-norm tolerance of $10^{-6}$, verify the geometric convergence rate $\gamma$ predicted by the contraction theorem, extract the greedy policy from $Q^*$, re-derive $V^*$ by policy iteration, run tabular Q-learning for 5000 episodes and confirm $\|Q_t - Q^*\|_\infty\to 0$, and close with REINFORCE on a tiny 3-state MDP --- a preview of Chapter 30.


In [ ]:
import numpy as np
np.random.seed(0)

# 4x4 GridWorld. State 0..15 in row-major order. Goal = state 15 (bottom-right), absorbing.
# Actions: 0=Up, 1=Down, 2=Left, 3=Right. Reward = -1 per step, +10 on entering goal.
N = 4
nS, nA = N*N, 4
GOAL = nS - 1

def step_idx(s, a):
    r, c = divmod(s, N)
    if a == 0: r = max(r - 1, 0)
    elif a == 1: r = min(r + 1, N - 1)
    elif a == 2: c = max(c - 1, 0)
    elif a == 3: c = min(c + 1, N - 1)
    return r * N + c

P = np.zeros((nS, nA, nS))
R = np.zeros((nS, nA))
for s in range(nS):
    for a in range(nA):
        if s == GOAL:
            P[s, a, s] = 1.0   # absorbing
            R[s, a] = 0.0
        else:
            sp = step_idx(s, a)
            P[s, a, sp] = 1.0
            R[s, a] = 10.0 if sp == GOAL else -1.0

print("P shape:", P.shape, "row-stochastic:", np.allclose(P.sum(-1), 1.0))
print("R shape:", R.shape, "min/max reward:", R.min(), R.max())
print("Reward at (state 14, action Right):", R[14, 3], "  -> goal? yes")


## Value iteration

Apply $V_{k+1} = \mathcal{T}^* V_k$ until $\|V_{k+1}-V_k\|_\infty < 10^{-6}$, with $\gamma=0.9$. By the contraction theorem we expect the error $\|V_k - V^*\|_\infty$ to decay like $\gamma^k$, i.e. a straight line of slope $\log\gamma$ on a log-y plot.


In [ ]:
gamma = 0.9
V = np.zeros(nS)
errs = []
for k in range(2000):
    Q = R + gamma * (P @ V)               # shape (nS, nA)
    V_new = Q.max(axis=1)
    errs.append(np.max(np.abs(V_new - V)))
    V = V_new
    if errs[-1] < 1e-6:
        break

V_star = V.copy()
Q_star = R + gamma * (P @ V_star)
print(f"value iteration converged in {k+1} iterations, final delta={errs[-1]:.2e}")
print("V* on the 4x4 grid (rounded):")
print(np.round(V_star.reshape(N, N), 3))

# Geometric-rate check: distance to V* should decay like gamma^k.
V2 = np.zeros(nS)
gaps = []
for k in range(60):
    gaps.append(np.max(np.abs(V2 - V_star)))
    V2 = (R + gamma * (P @ V2)).max(axis=1)
gaps = np.array(gaps)

# slope of log(gap) vs k, ignoring the tail where gap underflows to 0
mask = gaps > 1e-12
slope = np.polyfit(np.arange(len(gaps))[mask], np.log(gaps[mask] + 1e-300), 1)[0]
print(f"empirical slope = {slope:.4f}, log(gamma) = {np.log(gamma):.4f}")

try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.semilogy(gaps + 1e-16, "o-", label="||V_k - V*||")
    ax.semilogy(gaps[0] * gamma ** np.arange(len(gaps)),
                "--", label="gamma^k bound")
    ax.set_xlabel("iteration k"); ax.set_ylabel("sup-norm error")
    ax.legend(); fig.tight_layout()
    plt.show()
except Exception as e:
    print("matplotlib unavailable:", e)
    for k, g in enumerate(gaps[:20]):
        print(f"k={k:2d}  gap={g:.3e}  bound={gaps[0]*gamma**k:.3e}")


## Greedy policy from $Q^*$

The Bellman optimality equation says $\pi^*(s) \in \arg\max_a Q^*(s,a)$. We display the policy as a $4{\times}4$ grid of arrows.


In [ ]:
arrows = ["U", "D", "L", "R"]
pi_star = Q_star.argmax(axis=1)
pi_star[GOAL] = -1   # absorbing -> mark goal
grid = np.array([arrows[a] if a >= 0 else "G" for a in pi_star]).reshape(N, N)
print("Greedy policy from Q* (G = absorbing goal):")
for row in grid:
    print("  " + "  ".join(row))


## Policy iteration

Alternate exact policy evaluation $V^\pi = (I - \gamma P^\pi)^{-1} r^\pi$ with greedy improvement. Each evaluation step is exact, so we expect convergence in dramatically fewer outer iterations than value iteration.


In [ ]:
def policy_eval_exact(pi):
    P_pi = P[np.arange(nS), pi]                    # (nS, nS)
    r_pi = R[np.arange(nS), pi]                    # (nS,)
    return np.linalg.solve(np.eye(nS) - gamma * P_pi, r_pi)

pi = np.zeros(nS, dtype=int)                       # start with all "Up"
for outer in range(50):
    V_pi = policy_eval_exact(pi)
    Q_pi = R + gamma * (P @ V_pi)
    pi_new = Q_pi.argmax(axis=1)
    if np.array_equal(pi_new, pi):
        break
    pi = pi_new

V_pi_final = policy_eval_exact(pi)
print(f"policy iteration converged after {outer+1} outer iterations")
print(f"||V_PI - V_VI||_inf = {np.max(np.abs(V_pi_final - V_star)):.2e}")
print("greedy policies match VI:", np.array_equal(pi[:GOAL], pi_star[:GOAL]))


## Tabular Q-learning

We run Watkins' Q-learning with $\varepsilon$-greedy exploration ($\varepsilon=0.1$) and per-pair step sizes $\alpha_t = 1/(1+\mathrm{visits}(s,a))$, satisfying the Robbins--Monro conditions $\sum \alpha_t = \infty$, $\sum \alpha_t^2 <\infty$ (Chapter 13). By the Watkins--Tsitsiklis theorem, $Q_t \to Q^*$ almost surely.


In [ ]:
rng = np.random.default_rng(0)
Qhat   = np.zeros((nS, nA))
visits = np.zeros((nS, nA), dtype=int)
eps    = 0.1
n_episodes, max_steps = 5000, 100
returns = np.zeros(n_episodes)
err_to_Qstar = []

for ep in range(n_episodes):
    s = rng.integers(0, nS - 1)             # start anywhere except goal
    G, disc = 0.0, 1.0
    for t in range(max_steps):
        a = rng.integers(nA) if rng.random() < eps else int(np.argmax(Qhat[s]))
        sp = step_idx(s, a) if s != GOAL else s
        r  = (10.0 if sp == GOAL else -1.0) if s != GOAL else 0.0
        visits[s, a] += 1
        alpha = 1.0 / visits[s, a]
        target = r + gamma * (0.0 if s == GOAL else Qhat[sp].max())
        Qhat[s, a] += alpha * (target - Qhat[s, a])
        G += disc * r; disc *= gamma
        s = sp
        if s == GOAL: break
    returns[ep] = G
    if (ep + 1) % 250 == 0:
        err_to_Qstar.append(np.max(np.abs(Qhat - Q_star)))

print(f"final ||Qhat - Q*||_inf = {np.max(np.abs(Qhat - Q_star)):.3f}")
print(f"mean return last 200 episodes = {returns[-200:].mean():.3f}")
greedy_policy = Qhat.argmax(axis=1); greedy_policy[GOAL] = -1
print("Greedy policy from learned Qhat:")
grid = np.array(["UDLR"[a] if a >= 0 else "G" for a in greedy_policy]).reshape(N, N)
for row in grid: print("  " + "  ".join(row))

try:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(9, 3))
    win = 100
    smooth = np.convolve(returns, np.ones(win)/win, mode="valid")
    axes[0].plot(smooth); axes[0].set_xlabel("episode"); axes[0].set_ylabel(f"return (smoothed, w={win})")
    axes[1].plot(np.arange(1, len(err_to_Qstar)+1)*250, err_to_Qstar, "o-")
    axes[1].set_xlabel("episode"); axes[1].set_ylabel("||Qhat - Q*||_inf"); axes[1].set_yscale("log")
    fig.tight_layout(); plt.show()
except Exception as e:
    print("matplotlib unavailable:", e)
    print("error trajectory (every 250 ep):", [round(x, 3) for x in err_to_Qstar])


## REINFORCE on a tiny 3-state MDP (preview of Chapter 30)

Three states, two actions, deterministic transitions. We parameterize $\pi_\theta(a\mid s)$ as softmax over per-(s,a) logits and apply the Monte-Carlo policy-gradient (REINFORCE) update $\theta \leftarrow \theta + \eta \sum_t G_t \nabla_\theta \log \pi_\theta(a_t\mid s_t)$. We expect convergence to the optimal expected return.


In [ ]:
nS3, nA3 = 3, 2
# Transitions[s][a] = next_state. Rewards[s][a].
T_next = np.array([[1, 2], [2, 0], [2, 2]])
R3     = np.array([[0.0, 1.0], [2.0, 0.0], [5.0, -1.0]])
GOAL3  = 2
gamma3 = 0.9

# Optimal value via VI for reference.
V3 = np.zeros(nS3)
for _ in range(500):
    Vn = np.array([max(R3[s, a] + gamma3 * (0.0 if s == GOAL3 else V3[T_next[s, a]])
                       for a in range(nA3)) for s in range(nS3)])
    if np.max(np.abs(Vn - V3)) < 1e-10: break
    V3 = Vn
V3_star = V3.copy()
print("V* (3-state) =", np.round(V3_star, 3))

def softmax(x):
    z = x - x.max()
    e = np.exp(z)
    return e / e.sum()

theta = np.zeros((nS3, nA3))
eta = 0.05
n_ep, T_max = 200, 30
ep_returns = []
rng2 = np.random.default_rng(0)

for ep in range(n_ep):
    s = rng2.integers(nS3)
    traj = []
    for t in range(T_max):
        probs = softmax(theta[s])
        a = rng2.choice(nA3, p=probs)
        r = R3[s, a]
        sp = T_next[s, a]
        traj.append((s, a, r, probs))
        s = sp
        if s == GOAL3: break
    # Monte-Carlo returns
    G = 0.0; Gs = []
    for (_, _, r, _) in reversed(traj):
        G = r + gamma3 * G
        Gs.append(G)
    Gs.reverse()
    ep_returns.append(Gs[0] if Gs else 0.0)
    # REINFORCE gradient ascent
    for (s_t, a_t, _, p_t), G_t in zip(traj, Gs):
        grad = -p_t.copy(); grad[a_t] += 1.0       # d log pi(a|s) / d theta_s
        theta[s_t] += eta * G_t * grad

print(f"mean return first 20 episodes  = {np.mean(ep_returns[:20]):.3f}")
print(f"mean return last 20 episodes   = {np.mean(ep_returns[-20:]):.3f}")
print("learned policy probs:")
for s in range(nS3):
    print(f"  s={s}: pi={np.round(softmax(theta[s]), 3)}")

try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(5, 3))
    win = 20
    smooth = np.convolve(ep_returns, np.ones(win)/win, mode="valid")
    ax.plot(smooth); ax.set_xlabel("episode"); ax.set_ylabel(f"return (smoothed, w={win})")
    fig.tight_layout(); plt.show()
except Exception as e:
    print("matplotlib unavailable:", e)
    print("returns (every 20 ep):", [round(np.mean(ep_returns[i:i+20]), 2) for i in range(0, n_ep, 20)])


## Takeaways

- The Bellman equations follow from $G_t = r_t + \gamma G_{t+1}$ plus linearity of expectation (Chapter 10).
- Both $\mathcal{T}^\pi$ and $\mathcal{T}^*$ are $\gamma$-contractions in sup norm; Banach's fixed-point theorem (Chapter 7) hands us value iteration with rate $\gamma$.
- Policy iteration alternates exact evaluation with greedy improvement; the deterministic policy space is finite, so termination is finite.
- Tabular Q-learning is Robbins--Monro (Chapter 13) on the contraction $\mathcal{T}^*$ with one-sample noise; convergence is almost-sure under the standard step-size and visitation conditions.
- REINFORCE replaces tabular value updates with stochastic gradient ascent on $\mathbb{E}_{\pi_\theta}[G_0]$ --- the launching point for Chapter 30 (advantage estimation, GAE) and Chapter 31 (actor-critic, PPO, the RLHF stack).


## Value-based deep RL: function approximation, DQN, and the max-entropy framework

Chapter 29 gave us tabular $Q$-learning. Real problems (Atari, robots, language models) have state spaces too large to tabulate, so we replace $Q(s,a)$ with a parameterized function $Q_\theta(s,a)$ — a linear map of features $\phi(s,a)$, or a neural network. We need (i) a function approximator, (ii) off-policy learning from a replay buffer, and (iii) a stationary bootstrap target. Sutton & Barto's *deadly triad* warns these three together can diverge — DQN (Mnih et al. 2015) added two simple stabilizers.

In [ ]:
import numpy as np, math, time
np.random.seed(0)

# 5-state chain. State 0..4. Always go right. r=+1 on transition out of state 4.
N = 5; gamma = 0.9
def step(s):
    if s == N-1: return None, 1.0
    return s+1, 0.0

# Closed-form V: V(N-1)=1, V(s)=gamma*V(s+1).
V_true = np.zeros(N); V_true[N-1] = 1.0
for s in range(N-2, -1, -1):
    V_true[s] = gamma * V_true[s+1]

# Linear FA with one-hot features -> w[s] *is* V(s).
w = np.zeros(N); alpha = 0.1
for ep in range(2000):
    s = 0
    while s is not None:
        s_next, r = step(s)
        v_next = w[s_next] if s_next is not None else 0.0
        td = r + gamma * v_next - w[s]
        w[s] += alpha * td
        s = s_next

print('closed-form V :', np.round(V_true, 4))
print('learned   V   :', np.round(w, 4))
print('max abs error :', float(np.max(np.abs(w - V_true))))

## The deadly triad and Baird's counterexample

Off-policy + bootstrapping + linear FA can diverge. Baird's 7-state example uses 8-d features
$\phi(s)=2e_s+e_8$ for $s=1..6$ and $\phi(7)=e_7+2e_8$. Two actions: *solid* deterministically goes to state 7, *dashed* to a uniform state in 1..6. Target policy always picks solid; behavior picks solid with probability $1/7$, so the importance ratio on solid steps is $\rho=7$ and on dashed steps $\rho=0$. All rewards 0, $\gamma=0.99$. With $w_0=(1,1,1,1,1,1,10,1)$, semi-gradient TD(0) explodes.

In [ ]:
import numpy as np
np.random.seed(0)

phi = np.zeros((7, 8))
for s in range(6):
    phi[s, s] = 2.0
    phi[s, 7] = 1.0
phi[6, 6] = 1.0
phi[6, 7] = 2.0

gamma = 0.99
alpha = 0.01
w = np.array([1, 1, 1, 1, 1, 1, 10, 1.0])
norms = [np.linalg.norm(w)]

rng = np.random.default_rng(0)
for t in range(1000):
    if rng.random() < 1/7:           # behavior picks solid
        s = rng.integers(0, 7)
        s_next = 6                   # state 7 (0-indexed -> 6)
        rho = 7.0
        td = 0 + gamma * (phi[s_next] @ w) - (phi[s] @ w)
        w = w + alpha * rho * td * phi[s]
    norms.append(np.linalg.norm(w))

print(f'initial ||w|| = {norms[0]:.3f}')
print(f'final   ||w|| = {norms[-1]:.3f}  (divergence)')
print('w final:', np.round(w, 2))

try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(5, 3))
    plt.plot(norms); plt.yscale('log')
    plt.xlabel('TD(0) update'); plt.ylabel('||w|| (log)')
    plt.title("Baird's counterexample: linear off-policy TD(0) diverges")
    plt.tight_layout(); plt.show()
except Exception as e:
    print('matplotlib unavailable:', e)

## DQN's two tricks: replay buffer + target network

The replay buffer $\mathcal{D}$ stores past transitions $(s,a,r,s')$; uniform minibatches decorrelate the gradient signal so the SGD assumptions of Ch 13 are recovered. The target network $\theta^-$ is a frozen copy of $\theta$, refreshed every $K$ steps; this stabilizes the bootstrap target $r+\gamma\max_{a'}Q_{\theta^-}(s',a')$.

We implement DQN entirely in numpy (no torch) on a hand-coded CartPole: 4-d state $(x,\dot x,\theta,\dot\theta)$, two actions (push left/right), reward +1 per step until the pole falls or the cart leaves bounds. The Q-network is a 4-32-2 MLP with hand-written backprop and AdamW (Ch 14).

In [ ]:
import numpy as np, math, time
np.random.seed(0)
rng = np.random.default_rng(0)

# CartPole physics (Barto/Sutton/Anderson 1983).
g_acc=9.8; m_c=1.0; m_p=0.1; total_m=m_c+m_p; l=0.5; pm_l=m_p*l
force_mag=10.0; tau=0.02
theta_thresh = 12*2*math.pi/360; x_thresh = 2.4

def env_reset(): return rng.uniform(-0.05, 0.05, size=4)
def env_step(s, a):
    x, x_dot, theta, theta_dot = s
    force = force_mag if a == 1 else -force_mag
    ct, st = math.cos(theta), math.sin(theta)
    temp = (force + pm_l*theta_dot**2*st) / total_m
    th_acc = (g_acc*st - ct*temp) / (l*(4.0/3.0 - m_p*ct**2/total_m))
    x_acc = temp - pm_l*th_acc*ct/total_m
    x = x + tau*x_dot;       x_dot = x_dot + tau*x_acc
    theta = theta + tau*theta_dot; theta_dot = theta_dot + tau*th_acc
    s_new = np.array([x, x_dot, theta, theta_dot])
    done = bool(x<-x_thresh or x>x_thresh or theta<-theta_thresh or theta>theta_thresh)
    return s_new, 1.0, done

# Q-network 4 -> 32 -> 2
def init_net(hid=32):
    W1 = rng.standard_normal((4, hid)) * np.sqrt(2/4); b1 = np.zeros(hid)
    W2 = rng.standard_normal((hid, 2)) * np.sqrt(2/hid); b2 = np.zeros(2)
    return [W1, b1, W2, b2]

def forward(net, x):
    W1, b1, W2, b2 = net
    z1 = x @ W1 + b1; h1 = np.maximum(z1, 0)
    return h1 @ W2 + b2, (x, z1, h1)

def backward(net, cache, dq):
    W1, b1, W2, b2 = net; x, z1, h1 = cache
    dW2 = h1.T @ dq; db2 = dq.sum(0)
    dh1 = dq @ W2.T; dz1 = dh1 * (z1 > 0)
    dW1 = x.T @ dz1; db1 = dz1.sum(0)
    return [dW1, db1, dW2, db2]

class AdamW:
    def __init__(self, params, lr=2e-3, b1=0.9, b2=0.999, eps=1e-8, wd=1e-5):
        self.p=params; self.lr=lr; self.b1=b1; self.b2=b2; self.eps=eps; self.wd=wd
        self.m=[np.zeros_like(p) for p in params]; self.v=[np.zeros_like(p) for p in params]; self.t=0
    def step(self, grads):
        self.t += 1
        for i,(p,g) in enumerate(zip(self.p, grads)):
            self.m[i] = self.b1*self.m[i] + (1-self.b1)*g
            self.v[i] = self.b2*self.v[i] + (1-self.b2)*(g*g)
            mh = self.m[i]/(1-self.b1**self.t); vh = self.v[i]/(1-self.b2**self.t)
            p -= self.lr*(mh/(np.sqrt(vh)+self.eps) + self.wd*p)

net = init_net(32); target = [p.copy() for p in net]; opt = AdamW(net)
buf, buf_cap = [], 2000
gamma = 0.99; eps_v=1.0; eps_end=0.05; eps_decay=0.995
batch=32; target_every=100; total_steps=0
ep_lens=[]; t0=time.time()
EP, MAX_T = 150, 200

for ep in range(EP):
    s = env_reset(); done=False; steps=0
    while not done and steps < MAX_T:
        if rng.random() < eps_v:
            a = int(rng.integers(0, 2))
        else:
            q,_ = forward(net, s[None]); a = int(np.argmax(q[0]))
        s_new, r, done = env_step(s, a)
        buf.append((s, a, r, s_new, done))
        if len(buf) > buf_cap: buf.pop(0)
        s = s_new; steps += 1; total_steps += 1
        if len(buf) >= batch:
            idx = rng.integers(0, len(buf), size=batch)
            S  = np.array([buf[i][0] for i in idx]); A = np.array([buf[i][1] for i in idx])
            R  = np.array([buf[i][2] for i in idx]); S2= np.array([buf[i][3] for i in idx])
            D  = np.array([buf[i][4] for i in idx], dtype=float)
            qn,_ = forward(target, S2)
            tgt  = R + gamma*(1.0-D)*np.max(qn, axis=1)
            qp, cache = forward(net, S)
            err = qp[np.arange(batch), A] - tgt
            dq = np.zeros_like(qp); dq[np.arange(batch), A] = (2.0/batch)*err
            opt.step(backward(net, cache, dq))
        if total_steps % target_every == 0:
            target = [p.copy() for p in net]
    eps_v = max(eps_end, eps_v * eps_decay)
    ep_lens.append(steps)

print(f'first 20 ep mean length : {np.mean(ep_lens[:20]):.1f}')
print(f'last  30 ep mean length : {np.mean(ep_lens[-30:]):.1f}')
print(f'max episode length      : {max(ep_lens)}')
print(f'training time           : {time.time()-t0:.1f}s')

## Maximum-entropy RL

Augment the reward with an entropy bonus $\alpha H(\pi)$. The per-state Lagrangian
$$\sum_a \pi(a)Q(s,a)+\alpha H(\pi)\quad\text{s.t.}\quad \sum_a\pi(a)=1$$
has the closed-form maximizer $\pi^*(a\mid s)=\exp(Q(s,a)/\alpha)/Z$ with value $\alpha\log\sum_a\exp(Q(s,a)/\alpha)$. Plug into Bellman to get the soft operator $\mathcal{T}^*_\alpha Q(s,a)=r+\gamma\,\mathbb{E}_{s'}[\alpha\log\sum_{a'}\exp(Q(s',a')/\alpha)]$, a $\gamma$-contraction in $\|\cdot\|_\infty$. As $\alpha\to 0$ we recover hard $Q^*$ from Ch 29 (Ch 16 softmax-temperature limit).

In [ ]:
import numpy as np
np.random.seed(0)

# Compact 4x4 GridWorld (re-implemented from Ch 29).
G = 4; N_S = G*G; N_A = 4; GAMMA = 0.95
GOAL = (G-1, G-1)
DELTAS = [(-1,0), (0,1), (1,0), (0,-1)]   # up, right, down, left

def s2xy(s): return divmod(s, G)
def xy2s(x, y): return x*G + y

def step(s, a):
    if s2xy(s) == GOAL: return s, 0.0, True
    x, y = s2xy(s); dx, dy = DELTAS[a]
    nx, ny = max(0, min(G-1, x+dx)), max(0, min(G-1, y+dy))
    ns = xy2s(nx, ny)
    if (nx, ny) == GOAL: return ns, 1.0, True
    return ns, -0.01, False

P = np.zeros((N_S, N_A, N_S)); R = np.zeros((N_S, N_A))
TERM = np.zeros((N_S, N_A), dtype=bool)
for s in range(N_S):
    for a in range(N_A):
        ns, r, d = step(s, a)
        P[s, a, ns] = 1.0; R[s, a] = r; TERM[s, a] = d

def soft_VI(alpha, iters=400):
    Q = np.zeros((N_S, N_A))
    for _ in range(iters):
        m = Q.max(axis=1, keepdims=True)
        V = (alpha*(np.log(np.exp((Q-m)/alpha).sum(axis=1, keepdims=True)) + m/alpha)).squeeze()
        Q = R + GAMMA * (P @ V) * (~TERM)
    return Q

def hard_VI(iters=400):
    Q = np.zeros((N_S, N_A))
    for _ in range(iters):
        V = Q.max(axis=1)
        Q = R + GAMMA * (P @ V) * (~TERM)
    return Q

Q_hard = hard_VI()
for alpha in [0.01, 0.5, 5.0]:
    Q = soft_VI(alpha)
    pi = np.exp((Q - Q.max(axis=1, keepdims=True))/alpha)
    pi = pi / pi.sum(axis=1, keepdims=True)
    H = -(pi * np.log(pi + 1e-12)).sum(axis=1).mean()
    print(f'alpha={alpha:>5}: avg policy entropy = {H:.3f}  (uniform max = {np.log(N_A):.3f})')

Q_low = soft_VI(0.001)
print(f'\nlimit check: max |Q_soft(alpha=1e-3) - Q_hard| = {np.max(np.abs(Q_low - Q_hard)):.4f}')

# Visualize 4x4 policy mass on the right action for alpha=0.5
Q_v = soft_VI(0.5)
pi_v = np.exp((Q_v - Q_v.max(axis=1, keepdims=True))/0.5)
pi_v = pi_v / pi_v.sum(axis=1, keepdims=True)
print('\nPolicy P(right) on the 4x4 grid at alpha=0.5:')
print(np.round(pi_v[:, 1].reshape(G, G), 2))

## Soft Bellman fixed point and the RLHF closed form

Run soft value iteration on a tiny 3-state, 2-action MDP and check that the residual $Q-(R+\gamma P V_\mathrm{soft})$ vanishes. Then print the analogous RLHF closed form: replace the entropy bonus $\alpha H(\pi)$ with the KL leash $-\beta D_\mathrm{KL}(\pi\|\pi_\mathrm{ref})$ and the same Lagrangian gives $\pi^*(y|x)\propto\pi_\mathrm{ref}(y|x)\exp(r(x,y)/\beta)$ — the starting point of the DPO derivation in Chapter 28.

In [ ]:
import numpy as np
np.random.seed(0)

S, A, gamma, alpha = 3, 2, 0.9, 0.3
P = np.array([
    [[0.7, 0.2, 0.1], [0.1, 0.8, 0.1]],
    [[0.4, 0.4, 0.2], [0.2, 0.3, 0.5]],
    [[0.1, 0.1, 0.8], [0.5, 0.4, 0.1]],
])
R = np.array([[1.0, 0.0], [0.5, -0.2], [0.0, 1.5]])

Q = np.zeros((S, A))
for _ in range(500):
    m = Q.max(axis=1, keepdims=True)
    V = (alpha*(np.log(np.exp((Q-m)/alpha).sum(axis=1, keepdims=True)) + m/alpha)).squeeze()
    Q = R + gamma * (P @ V)

m = Q.max(axis=1, keepdims=True)
V = (alpha*(np.log(np.exp((Q-m)/alpha).sum(axis=1, keepdims=True)) + m/alpha)).squeeze()
res = Q - (R + gamma * (P @ V))

print('soft Q* :')
print(np.round(Q, 4))
print('soft V* :', np.round(V, 4))
print(f'max |Bellman residual| = {np.max(np.abs(res)):.2e}')

print()
print('RLHF closed form (Ch 28):')
print('  pi*(y|x) = pi_ref(y|x) * exp(r(x,y)/beta) / Z(x)')
print('  Same Lagrangian as max-entropy RL with alpha H(pi) -> -beta KL(pi || pi_ref).')

## Takeaways

- **Function approximation** lets $Q$ generalize to unseen states; with linear $\phi$ plus on-policy + non-bootstrapping it is provably convergent (Tsitsiklis & Van Roy 1997).
- The **deadly triad** (off-policy + bootstrapping + FA) can diverge — Baird's counterexample is a 7-state existence proof.
- **DQN** clears the triad with a replay buffer (decorrelates gradient samples; restores SGD assumptions) and a target network (stationary bootstrap target). The numpy CartPole agent here goes from $\sim 22$ steps random to $> 100$ steps in $\sim 1$ minute on CPU.
- **Max-entropy RL** is a constrained convex optimization: the closed-form policy is Boltzmann in $Q$ and the soft Bellman operator is a $\gamma$-contraction whose fixed point reduces to hard $Q^*$ as $\alpha\to 0$.
- The **same Lagrangian**, with KL-to-$\pi_\mathrm{ref}$ instead of entropy, produces $\pi^*(y|x)\propto\pi_\mathrm{ref}(y|x)\exp(r(x,y)/\beta)$ — the RLHF/DPO closed form (Ch 28). DQN's target network is the algorithmic analogue of RLHF's $\pi_\mathrm{ref}$ anchor. Chapter 31 closes the loop with policy gradients and GRPO.

## Policy gradient, GRPO, and the RLHF/DPO bridge: tiny-GPT alignment loop

Roadmap. (1) REINFORCE on a 3-state MDP. (2) REINFORCE + state baseline; show variance drops. (3) GAE on a 5-state chain; sweep $\lambda$. (4) PPO on a CartPole-style env. (5) GRPO warm-up on a 5-arm contextual bandit. (6) **Climax:** GRPO on the Chapter 27 tiny GPT with a verifiable reward. (7) RLHF / DPO / GRPO comparison table.


In [ ]:
import numpy as np, math, time
np.random.seed(0)
rng_global = np.random.default_rng(0)

# ---------- 3-state, 2-action MDP ----------
# States: 0,1,2.  Action 0 = stay, action 1 = advance (with terminal reward at state 2).
S, A_n = 3, 2
def step_mdp(s, a):
    if s == 2:
        return s, 0.0, True
    s2 = s if a == 0 else min(s + 1, 2)
    r = 1.0 if s2 == 2 else 0.0
    done = (s2 == 2)
    return s2, r, done

logits = np.zeros((S, A_n))
def policy_probs(logits, s):
    z = logits[s]
    z = z - z.max(); e = np.exp(z); return e / e.sum()

def rollout(logits, max_T=20, gamma=0.95):
    s, traj = 0, []
    for _ in range(max_T):
        p = policy_probs(logits, s); a = int(rng_global.choice(A_n, p=p))
        s2, r, done = step_mdp(s, a)
        traj.append((s, a, r))
        s = s2
        if done: break
    G = 0.0; returns = []
    for (_, _, r) in reversed(traj):
        G = r + gamma * G
        returns.append(G)
    returns.reverse()
    return traj, returns

lr = 0.1; ep_returns = []
for ep in range(200):
    traj, rets = rollout(logits)
    g = np.zeros_like(logits)
    for (s, a, _), G in zip(traj, rets):
        p = policy_probs(logits, s)
        score = -p; score[a] += 1.0           # \nabla log pi(a|s)
        g[s] += score * G
    logits += lr * g
    ep_returns.append(rets[0] if rets else 0.0)
print(f'REINFORCE  initial return ~ {np.mean(ep_returns[:20]):.3f}'
      f'   final ~ {np.mean(ep_returns[-20:]):.3f}')
print('learned advance-prob per state:',
      [round(float(policy_probs(logits, s)[1]), 3) for s in range(S)])


### REINFORCE with a state baseline

By Lemma 2 (baselines are unbiased) we may subtract any state-dependent $b(s)$ from the return. The variance-minimizing $b$ is close to $V^\pi(s)$. We learn a tabular $V_\phi(s)$ by regressing on observed Monte Carlo returns.


In [ ]:
logits_a = np.zeros((S, A_n))
logits_b = np.zeros((S, A_n))
V = np.zeros(S)
lr_pi, lr_v = 0.1, 0.2
var_no, var_with, ret_no, ret_with = [], [], [], []

for ep in range(400):
    # ----- no baseline -----
    traj, rets = rollout(logits_a)
    g = np.zeros_like(logits_a); per_step = []
    for (s, a, _), G in zip(traj, rets):
        p = policy_probs(logits_a, s)
        score = -p; score[a] += 1.0
        g[s] += score * G
        per_step.append(np.linalg.norm(score * G))
    logits_a += lr_pi * g
    var_no.append(np.var(per_step) if per_step else 0.0)
    ret_no.append(rets[0] if rets else 0.0)

    # ----- with state-value baseline -----
    traj, rets = rollout(logits_b)
    g = np.zeros_like(logits_b); per_step = []
    for (s, a, _), G in zip(traj, rets):
        adv = G - V[s]
        p = policy_probs(logits_b, s)
        score = -p; score[a] += 1.0
        g[s] += score * adv
        per_step.append(np.linalg.norm(score * adv))
        V[s] += lr_v * (G - V[s])
    logits_b += lr_pi * g
    var_with.append(np.var(per_step) if per_step else 0.0)
    ret_with.append(rets[0] if rets else 0.0)

print(f'mean per-step gradient variance  no baseline = {np.mean(var_no):.4f}'
      f'   with baseline = {np.mean(var_with):.4f}'
      f'   ratio = {np.mean(var_no)/(np.mean(var_with)+1e-9):.2f}x')
print(f'final return  no = {np.mean(ret_no[-30:]):.3f}'
      f'   with = {np.mean(ret_with[-30:]):.3f}')


### GAE: sweeping $\lambda$

On a 5-state chain with stochastic rewards we run REINFORCE with the GAE advantage $\hat A_t = \delta_t + \gamma\lambda \hat A_{t+1}$ for $\lambda \in \{0, 0.5, 0.95, 1\}$. Variance of $\hat A_t$ should rise with $\lambda$; bias should fall.


In [ ]:
Sc, A_n = 5, 2
def step_chain(s, a):
    if s == Sc - 1:
        return s, 0.0, True
    s2 = max(0, s - 1) if a == 0 else min(Sc - 1, s + 1)
    r = float(rng_global.normal(0.0, 0.3)) + (1.0 if s2 == Sc - 1 else 0.0)
    return s2, r, s2 == Sc - 1

def gae_advantages(rs, vs, gamma, lam):
    A_, adv = 0.0, []
    for t in reversed(range(len(rs))):
        v_next = vs[t+1] if t+1 < len(vs) else 0.0
        delta = rs[t] + gamma * v_next - vs[t]
        A_ = delta + gamma * lam * A_
        adv.append(A_)
    adv.reverse(); return adv

results = {}
for lam in [0.0, 0.5, 0.95, 1.0]:
    logits = np.zeros((Sc, A_n)); Vp = np.zeros(Sc)
    adv_var, ret_log = [], []
    for ep in range(300):
        s, traj = 0, []
        for _ in range(40):
            p = policy_probs(logits, s); a = int(rng_global.choice(A_n, p=p))
            s2, r, done = step_chain(s, a); traj.append((s, a, r, s2)); s = s2
            if done: break
        rs = [t[2] for t in traj]; states = [t[0] for t in traj] + [traj[-1][3]]
        vs = [Vp[ss] for ss in states]
        adv = gae_advantages(rs, vs, gamma=0.95, lam=lam)
        adv_var.append(np.var(adv) if adv else 0.0)
        # update V (Monte-Carlo target)
        G = 0.0
        for (s_, a_, r_, _), A_ in zip(reversed(traj), reversed(adv)):
            G = r_ + 0.95 * G
            Vp[s_] += 0.05 * (G - Vp[s_])
        # policy step
        g = np.zeros_like(logits)
        for (s_, a_, _, _), A_ in zip(traj, adv):
            p = policy_probs(logits, s_)
            score = -p; score[a_] += 1.0
            g[s_] += score * A_
        logits += 0.05 * g
        ret_log.append(sum(rs))
    results[lam] = (np.mean(adv_var), np.mean(ret_log[-30:]))

print('lambda | mean Var(A_hat) | final return')
for lam, (v, r) in results.items():
    print(f'  {lam:>4.2f} |     {v:8.4f}    | {r:+.3f}')
print('expectation: Var rises with lambda; bias drops (return improves toward MC truth).')


### PPO on a CartPole-style env

1D balance task: state $(x, \dot x, \theta, \dot\theta)$, force $\pm 1$. Linear-Euler dynamics from Ch 30. We train a 2-layer MLP policy with hand backprop and the PPO-clipped surrogate.


In [ ]:
g_grav, m_c, m_p, l_p, dt, force_mag = 9.8, 1.0, 0.1, 0.5, 0.02, 10.0
def cart_step(state, a):
    x, xd, th, thd = state
    F = force_mag if a == 1 else -force_mag
    cos_t, sin_t = math.cos(th), math.sin(th)
    temp = (F + m_p*l_p*thd*thd*sin_t) / (m_c + m_p)
    th_acc = (g_grav*sin_t - cos_t*temp) / (l_p*(4/3 - m_p*cos_t*cos_t/(m_c+m_p)))
    x_acc  = temp - m_p*l_p*th_acc*cos_t/(m_c + m_p)
    state2 = (x + dt*xd, xd + dt*x_acc, th + dt*thd, thd + dt*th_acc)
    done = abs(state2[0]) > 2.4 or abs(state2[2]) > 0.21
    return state2, 1.0, done

din, dh = 4, 16
W1 = (np.random.randn(din, dh) * 0.3).astype(np.float64); b1 = np.zeros(dh)
W2 = (np.random.randn(dh, 2)  * 0.3).astype(np.float64); b2 = np.zeros(2)

def policy(s):
    s = np.array(s)
    h = np.tanh(s @ W1 + b1); z = h @ W2 + b2
    z = z - z.max(); e = np.exp(z); return e / e.sum(), h

def grad_logp(s, a, h, p):
    s = np.array(s)
    dz = -p; dz[a] += 1.0
    dW2 = np.outer(h, dz); db2 = dz
    dh = dz @ W2.T
    dz1 = dh * (1 - h*h)
    dW1 = np.outer(s, dz1); db1 = dz1
    return dW1, db1, dW2, db2

epsilon, gamma = 0.2, 0.99; lr = 5e-3; ep_R = []
for ep in range(120):
    s = (rng_global.normal(0, 0.05), 0.0, rng_global.normal(0, 0.05), 0.0)
    traj = []
    for _ in range(200):
        p, h = policy(s); a = int(rng_global.choice(2, p=p))
        s2, r, done = cart_step(s, a); traj.append((s, a, r, p[a].copy(), h)); s = s2
        if done: break
    R = 0.0; rets = []
    for (_, _, r, _, _) in reversed(traj):
        R = r + gamma * R; rets.append(R)
    rets.reverse()
    adv = np.array(rets) - np.mean(rets)
    if adv.std() > 1e-6: adv = adv / (adv.std() + 1e-8)
    gW1, gb1, gW2, gb2 = (np.zeros_like(W1), np.zeros_like(b1),
                          np.zeros_like(W2), np.zeros_like(b2))
    for (s_, a_, _, p_old, h_old), A in zip(traj, adv):
        p_new, h_new = policy(s_)
        ratio = p_new[a_] / (p_old + 1e-9)
        clip = max(min(ratio, 1+epsilon), 1-epsilon)
        # Stop gradient on clipped path
        if (A > 0 and ratio > 1+epsilon) or (A < 0 and ratio < 1-epsilon):
            continue
        # Surrogate L = ratio * A; dL/dlogp = ratio * A
        coef = ratio * A
        dW1_, db1_, dW2_, db2_ = grad_logp(s_, a_, h_new, p_new)
        gW1 += coef*dW1_; gb1 += coef*db1_; gW2 += coef*dW2_; gb2 += coef*db2_
    n = max(1, len(traj))
    W1 += lr * gW1 / n; b1 += lr * gb1 / n
    W2 += lr * gW2 / n; b2 += lr * gb2 / n
    ep_R.append(len(traj))
print(f'PPO CartPole  early avg length = {np.mean(ep_R[:20]):.1f}'
      f'   late avg length = {np.mean(ep_R[-20:]):.1f}')


### GRPO warm-up: contextual bandit

5 actions, fixed reward function. Sample $G=4$ actions per step, baseline by group mean, normalize by group std, PPO-clip the policy ratio. The policy should concentrate on the high-reward arm.


In [ ]:
K = 5
true_r = np.array([0.1, 0.3, 0.2, 0.9, 0.4])
logits_b = np.zeros(K)
G, eps_clip = 4, 0.2; lr = 0.2
history = []
for step in range(200):
    z = logits_b - logits_b.max(); p_old = np.exp(z); p_old /= p_old.sum()
    samples = rng_global.choice(K, size=G, p=p_old)
    rs = true_r[samples] + rng_global.normal(0, 0.05, size=G)
    adv = (rs - rs.mean()) / (rs.std() + 1e-6)
    g = np.zeros(K)
    for a, A in zip(samples, adv):
        # current policy = old (single update), so ratio = 1; clip is a no-op
        score = -p_old.copy(); score[a] += 1.0
        g += score * A
    logits_b += lr * g / G
    history.append(true_r[int(np.argmax(logits_b))])
z = logits_b - logits_b.max(); p_final = np.exp(z); p_final /= p_final.sum()
print('learned action probs:', np.round(p_final, 3))
print('true rewards         :', true_r)
print(f'best arm picked in last 30 steps: {int(np.argmax(p_final))} (true best = 3)')


### Climax: GRPO on the Chapter 27 tiny GPT

We re-instantiate the Ch 27 tiny GPT (tokenizer, params, forward, backward, AdamW) and pre-train it briefly to fix $\pi_{\mathrm{ref}}$. Then we run GRPO: for each prompt, sample $G=4$ continuations from $\pi_{\mathrm{old}}$, score each by a verifiable reward (does the continuation contain the substring `the`?), compute the group-relative advantage, and apply the PPO-clipped surrogate per token. Gradient at the logits is $-A_i \cdot \mathrm{ratio}_t \cdot (\mathrm{softmax}(\ell_t) - \mathbf{1}_{y_t})$ — a sign-flipped Ch 17 cross-entropy gradient weighted by the advantage and the ratio. We feed it into the existing `backward` pass.


In [ ]:
# === Ch 27 tiny GPT (tokenizer + model + AdamW) ===
import numpy as np, math, time
np.random.seed(0)
corpus = ('the cat sat on the mat. the dog sat on the log. '
          'a fox ran in the park. the bird flew to the tree. ' * 10)
vocab = sorted(set(corpus)); V = len(vocab)
stoi = {c: i for i, c in enumerate(vocab)}; itos = {i: c for c, i in stoi.items()}
data = np.array([stoi[c] for c in corpus], dtype=np.int64)
T = 16

def get_batch(B, rng):
    ix = rng.integers(0, len(data) - T - 1, size=B)
    x = np.stack([data[i:i+T]     for i in ix])
    y = np.stack([data[i+1:i+1+T] for i in ix])
    return x, y

d, H, dff, L = 32, 1, 64, 2
def init_linear(fi, fo, scale=1.0):
    return (np.random.randn(fi, fo) * (scale / math.sqrt(fi))).astype(np.float32)
params = {}
params['E'] = (np.random.randn(V, d) * 0.02).astype(np.float32)
params['P'] = (np.random.randn(T, d) * 0.02).astype(np.float32)
for l in range(L):
    params[f'ln1_g{l}'] = np.ones(d, dtype=np.float32); params[f'ln1_b{l}'] = np.zeros(d, dtype=np.float32)
    params[f'Wq{l}'] = init_linear(d, d, 0.5); params[f'Wk{l}'] = init_linear(d, d, 0.5)
    params[f'Wv{l}'] = init_linear(d, d, 0.5); params[f'Wo{l}'] = init_linear(d, d, 0.5)
    params[f'ln2_g{l}'] = np.ones(d, dtype=np.float32); params[f'ln2_b{l}'] = np.zeros(d, dtype=np.float32)
    params[f'W1{l}'] = init_linear(d, dff, 0.5); params[f'b1{l}'] = np.zeros(dff, dtype=np.float32)
    params[f'W2{l}'] = init_linear(dff, d, 0.5); params[f'b2{l}'] = np.zeros(d, dtype=np.float32)
params['lng'] = np.ones(d, dtype=np.float32); params['lnb'] = np.zeros(d, dtype=np.float32)

def layernorm(x, g, b, eps=1e-5):
    mu = x.mean(-1, keepdims=True); var = x.var(-1, keepdims=True)
    inv = 1.0 / np.sqrt(var + eps); xhat = (x - mu) * inv
    return xhat*g + b, (x, mu, var, inv, xhat, g)
def layernorm_back(dy, cache):
    x, mu, var, inv, xhat, g = cache; D = x.shape[-1]
    dxhat = dy * g
    dvar = (dxhat * (x - mu) * (-0.5) * inv**3).sum(-1, keepdims=True)
    dmu  = (dxhat * (-inv)).sum(-1, keepdims=True) + dvar*(-2.0/D)*(x-mu).sum(-1, keepdims=True)
    dx = dxhat*inv + dvar*2.0*(x-mu)/D + dmu/D
    dg = (dy * xhat).sum(axis=tuple(range(dy.ndim-1)))
    db = dy.sum(axis=tuple(range(dy.ndim-1)))
    return dx, dg, db
def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True); e = np.exp(z); return e / e.sum(axis=axis, keepdims=True)
causal_mask = (np.triu(np.ones((T, T), dtype=np.float32), k=1) * -1e9)

def forward(p, x):
    B, Tlen = x.shape; assert Tlen == T
    h = p['E'][x] + p['P'][None, :T]
    cache = {'x': x, 'blocks': []}
    for l in range(L):
        h_in = h
        h_n, c_ln1 = layernorm(h, p[f'ln1_g{l}'], p[f'ln1_b{l}'])
        Q = h_n @ p[f'Wq{l}']; K = h_n @ p[f'Wk{l}']; Vv = h_n @ p[f'Wv{l}']
        scores = (Q @ K.transpose(0,2,1)) / math.sqrt(d) + causal_mask[None]
        attn = softmax(scores, axis=-1); ctx = attn @ Vv
        attn_out = ctx @ p[f'Wo{l}']; h2 = h_in + attn_out
        h2_n, c_ln2 = layernorm(h2, p[f'ln2_g{l}'], p[f'ln2_b{l}'])
        z1 = h2_n @ p[f'W1{l}'] + p[f'b1{l}']; a1 = np.maximum(z1, 0)
        z2 = a1 @ p[f'W2{l}'] + p[f'b2{l}']; h = h2 + z2
        cache['blocks'].append(dict(h_in=h_in, c_ln1=c_ln1, h_n=h_n, Q=Q, K=K, V=Vv,
                                    attn=attn, ctx=ctx, h2=h2, c_ln2=c_ln2,
                                    h2_n=h2_n, z1=z1, a1=a1))
    h_n_final, c_lnf = layernorm(h, p['lng'], p['lnb'])
    logits = h_n_final @ p['E'].T
    cache['c_lnf'] = c_lnf; cache['h_n_final'] = h_n_final
    return logits, cache

def backward(p, cache, dlogits):
    grads = {k: np.zeros_like(v) for k, v in p.items()}
    h_n_final = cache['h_n_final']
    B, Tlen, Vl = dlogits.shape
    grads['E'] += (h_n_final.reshape(-1, d).T @ dlogits.reshape(-1, Vl)).T
    dh = dlogits @ p['E']
    dh_final, dlng, dlnb = layernorm_back(dh, cache['c_lnf'])
    grads['lng'] += dlng; grads['lnb'] += dlnb; dh = dh_final
    for l in reversed(range(L)):
        bc = cache['blocks'][l]
        dh2 = dh.copy(); dz2 = dh.copy()
        grads[f'W2{l}'] += bc['a1'].reshape(-1, dff).T @ dz2.reshape(-1, d)
        grads[f'b2{l}'] += dz2.reshape(-1, d).sum(0)
        da1 = dz2 @ p[f'W2{l}'].T; dz1 = da1 * (bc['z1'] > 0)
        grads[f'W1{l}'] += bc['h2_n'].reshape(-1, d).T @ dz1.reshape(-1, dff)
        grads[f'b1{l}'] += dz1.reshape(-1, dff).sum(0)
        dh2_n = dz1 @ p[f'W1{l}'].T
        dh2_a, dln2g, dln2b = layernorm_back(dh2_n, bc['c_ln2'])
        grads[f'ln2_g{l}'] += dln2g; grads[f'ln2_b{l}'] += dln2b
        dh2 += dh2_a
        dh_in = dh2.copy(); dattn_out = dh2.copy()
        grads[f'Wo{l}'] += bc['ctx'].reshape(-1, d).T @ dattn_out.reshape(-1, d)
        dctx = dattn_out @ p[f'Wo{l}'].T
        dattn = dctx @ bc['V'].transpose(0,2,1)
        dV = bc['attn'].transpose(0,2,1) @ dctx
        a = bc['attn']
        s_ = (dattn * a).sum(axis=-1, keepdims=True)
        dscores = (a * (dattn - s_)) / math.sqrt(d)
        dQ = dscores @ bc['K']; dK = dscores.transpose(0,2,1) @ bc['Q']
        grads[f'Wq{l}'] += bc['h_n'].reshape(-1, d).T @ dQ.reshape(-1, d)
        grads[f'Wk{l}'] += bc['h_n'].reshape(-1, d).T @ dK.reshape(-1, d)
        grads[f'Wv{l}'] += bc['h_n'].reshape(-1, d).T @ dV.reshape(-1, d)
        dh_n = dQ @ p[f'Wq{l}'].T + dK @ p[f'Wk{l}'].T + dV @ p[f'Wv{l}'].T
        dh_in_a, dln1g, dln1b = layernorm_back(dh_n, bc['c_ln1'])
        grads[f'ln1_g{l}'] += dln1g; grads[f'ln1_b{l}'] += dln1b
        dh_in += dh_in_a; dh = dh_in
    np.add.at(grads['E'], cache['x'], dh)
    grads['P'] += dh.sum(axis=0)
    return grads

class AdamW:
    def __init__(self, params, beta1=0.9, beta2=0.95, eps=1e-8, wd=0.01):
        self.m = {k: np.zeros_like(v) for k, v in params.items()}
        self.v = {k: np.zeros_like(v) for k, v in params.items()}
        self.b1, self.b2, self.eps, self.wd, self.t = beta1, beta2, eps, wd, 0
        self.no_wd = {k for k in params if k.startswith(('ln','b1','b2','lng','lnb','P','E'))}
    def step(self, params, grads, lr):
        self.t += 1; bc1, bc2 = 1 - self.b1**self.t, 1 - self.b2**self.t
        for k in params:
            g = grads[k]
            self.m[k] = self.b1*self.m[k] + (1-self.b1)*g
            self.v[k] = self.b2*self.v[k] + (1-self.b2)*(g*g)
            upd = (self.m[k]/bc1) / (np.sqrt(self.v[k]/bc2) + self.eps)
            if k not in self.no_wd: upd = upd + self.wd * params[k]
            params[k] -= lr * upd

def clip_global_norm(grads, c=1.0):
    sq = sum((g*g).sum() for g in grads.values()); n = float(np.sqrt(sq))
    s = min(1.0, c / (n + 1e-12))
    if s < 1.0:
        for k in grads: grads[k] *= s
    return n

# --- Brief pre-training to fix pi_ref ---
opt = AdamW(params); rng = np.random.default_rng(0)
for step in range(200):
    x, y = get_batch(16, rng)
    logits, cache = forward(params, x)
    probs = softmax(logits, -1)
    dlogits = probs.copy()
    dlogits[np.arange(16)[:,None], np.arange(T)[None,:], y] -= 1.0
    dlogits /= (16 * T)
    grads = backward(params, cache, dlogits)
    clip_global_norm(grads, 1.0)
    opt.step(params, grads, 3e-3)
import copy
params_ref = {k: v.copy() for k, v in params.items()}
print('pre-training done; pi_ref frozen.')


In [ ]:
# === GRPO loop on the tiny GPT ===
GROUP, eps_clip, beta_kl = 4, 0.2, 0.02
TARGET_SUBSTR = 'the'
PROMPTS = ['the cat ', 'the dog ', 'a fox ra', 'the bird']
GEN_LEN = 8     # tokens to generate per completion (kept short for CPU)

def encode(s):
    return [stoi[c] for c in s if c in stoi]

def reward_fn(text):
    return 1.0 if TARGET_SUBSTR in text else 0.0

def sample_completion(p, prompt_ids, n_new, temperature=1.0, rng=None):
    """Sample n_new tokens. Returns generated_ids, list of (input_window, sampled_token, p_old) per step."""
    rng = rng or np.random.default_rng()
    ids = list(prompt_ids); steps = []
    for _ in range(n_new):
        ctx = ids[-T:] if len(ids) >= T else ([0]*(T-len(ids)) + ids)
        x = np.array([ctx], dtype=np.int64)
        logits, _ = forward(p, x)
        last = logits[0, -1] / max(temperature, 1e-6)
        probs = softmax(last)
        tok = int(rng.choice(V, p=probs))
        steps.append((np.array(ctx, dtype=np.int64), tok, float(probs[tok])))
        ids.append(tok)
    return ids, steps

def kl_to_ref(p_cur, p_ref, n=4):
    kl = 0.0; ct = 0
    rng_kl = np.random.default_rng(123)
    for _ in range(n):
        x, _ = get_batch(2, rng_kl)
        l_c, _ = forward(p_cur, x); l_r, _ = forward(p_ref, x)
        pc = softmax(l_c, -1); pr = softmax(l_r, -1)
        kl += float((pc * (np.log(pc + 1e-12) - np.log(pr + 1e-12))).sum(-1).mean())
        ct += 1
    return kl / ct

def avg_reward(p, n=8, rng=None):
    rng = rng or np.random.default_rng(42)
    rs = []
    for _ in range(n):
        prm = PROMPTS[rng.integers(0, len(PROMPTS))]
        ids, _ = sample_completion(p, encode(prm), GEN_LEN, temperature=1.0, rng=rng)
        text = ''.join(itos[i] for i in ids[len(encode(prm)):])
        rs.append(reward_fn(text))
    return float(np.mean(rs))

rng_eval = np.random.default_rng(2024)
init_reward = avg_reward(params, n=24, rng=rng_eval)
print(f'GRPO initial avg reward: {init_reward:.3f}')
print('sample completions BEFORE training:')
for prm in PROMPTS:
    ids, _ = sample_completion(params, encode(prm), GEN_LEN, rng=np.random.default_rng(7))
    cont = ''.join(itos[i] for i in ids[len(encode(prm)):])
    print(f'  {prm!r:>12} -> {cont!r}  r={reward_fn(cont):.0f}')

opt_grpo = AdamW(params, wd=0.0)
rng_g = np.random.default_rng(99); t0 = time.time()
STEPS = 80
for step in range(STEPS):
    prm = PROMPTS[rng_g.integers(0, len(PROMPTS))]
    pid = encode(prm)
    # sample G completions from pi_old (= current params, snapshot logp_old)
    completions = []
    rewards = []
    for _ in range(GROUP):
        ids, steps_log = sample_completion(params, pid, GEN_LEN, rng=rng_g)
        cont = ''.join(itos[i] for i in ids[len(pid):])
        completions.append(steps_log); rewards.append(reward_fn(cont))
    rewards = np.array(rewards, dtype=np.float64)
    adv = (rewards - rewards.mean()) / (rewards.std() + 1e-6)
    if rewards.std() < 1e-9:
        continue                              # group degenerate; skip update
    # Build a single big batch: rows = (sample i, step t)
    rows_x, rows_tok, rows_logp_old, rows_adv = [], [], [], []
    for i, (steps_log, A_i) in enumerate(zip(completions, adv)):
        for (ctx, tok, p_old) in steps_log:
            rows_x.append(ctx); rows_tok.append(tok)
            rows_logp_old.append(math.log(p_old + 1e-12)); rows_adv.append(A_i)
    Bn = len(rows_x)
    Xb = np.stack(rows_x).astype(np.int64)            # (Bn, T)
    toks = np.array(rows_tok, dtype=np.int64)
    logp_old = np.array(rows_logp_old, dtype=np.float64)
    advs = np.array(rows_adv, dtype=np.float64)
    # Forward under CURRENT (= old at first inner step) policy
    logits, cache = forward(params, Xb)
    probs = softmax(logits, -1)                       # (Bn, T, V)
    last_probs = probs[:, -1, :]                      # action at the LAST position
    p_new = last_probs[np.arange(Bn), toks]
    logp_new = np.log(p_new + 1e-12)
    ratio = np.exp(logp_new - logp_old)               # (Bn,)
    # PPO clip mask: zero out the gradient where the clip is binding AGAINST improvement
    bind = ((advs > 0) & (ratio > 1 + eps_clip)) | ((advs < 0) & (ratio < 1 - eps_clip))
    mask = (~bind).astype(np.float64)
    # Gradient of L = -mean( ratio * adv ) wrt logits at last position:
    #   d ratio / d logp_new = ratio;  d logp_new / d logits = (1_{tok} - softmax(logits))
    # so d L / d logits = -mean( ratio * adv * (1_tok - p) ) per-row
    coef = (ratio * advs * mask) / max(1, Bn)         # (Bn,)
    dlogits = np.zeros_like(logits)                   # (Bn, T, V)
    # only the LAST position contributes (we sampled one token per step from there)
    delta = -last_probs.copy()
    delta[np.arange(Bn), toks] += 1.0                  # (Bn, V) = (1_tok - p)
    dlogits[:, -1, :] = -(coef[:, None] * delta).astype(logits.dtype)  # ascent -> negate
    # KL-to-ref penalty: add beta * d KL(pi || pi_ref) / d logits at every position.
    # d/dlogits KL(p||q) where p = softmax(logits), q fixed = p * (logp - logq - sum_y p(logp-logq))
    if beta_kl > 0:
        logits_ref, _ = forward(params_ref, Xb)
        log_p = np.log(probs + 1e-12); log_q = np.log(softmax(logits_ref, -1) + 1e-12)
        diff = log_p - log_q
        weighted = (probs * diff).sum(-1, keepdims=True)
        d_kl = probs * (diff - weighted)
        dlogits = dlogits + (beta_kl / Bn) * d_kl.astype(logits.dtype)
    grads = backward(params, cache, dlogits)
    gn = clip_global_norm(grads, 1.0)
    opt_grpo.step(params, grads, 1e-3)
    if step % 20 == 0:
        print(f'step {step:3d}  group_r mean={rewards.mean():.2f} std={rewards.std():.2f}  gn={gn:.3f}')
train_seconds = time.time() - t0

rng_eval2 = np.random.default_rng(2025)
final_reward = avg_reward(params, n=24, rng=rng_eval2)
kl_to_pi_ref = kl_to_ref(params, params_ref, n=4)
print(f'\nGRPO final avg reward : {final_reward:.3f}  (was {init_reward:.3f})')
print(f'KL(pi_theta || pi_ref): {kl_to_pi_ref:.4f}  (small = no collapse)')
print(f'training seconds      : {train_seconds:.1f}')
print('sample completions AFTER training:')
for prm in PROMPTS:
    ids, _ = sample_completion(params, encode(prm), GEN_LEN, rng=np.random.default_rng(7))
    cont = ''.join(itos[i] for i in ids[len(encode(prm)):])
    print(f'  {prm!r:>12} -> {cont!r}  r={reward_fn(cont):.0f}')

GRPO_INIT, GRPO_FINAL, GRPO_KL, GRPO_TIME = init_reward, final_reward, kl_to_pi_ref, train_seconds


### RLHF vs DPO vs GRPO comparison

All three optimize the same KL-regularized objective $\mathbb{E}_{x,y\sim\pi}[r(x,y)] - \beta D_{\mathrm{KL}}(\pi\|\pi_{\mathrm{ref}})$ — they differ in *how* they estimate the gradient.


In [ ]:
table = {
    'RLHF (PPO + RM)': {
        'gradient signal': 'r_phi(x,y) from a learned RM, advantage via GAE with V_psi',
        'models in memory': '4 (policy + value + reward + ref)',
        'pros': 'arbitrary, learned reward signals; well-tested in production',
        'cons': 'reward hacking; brittle infra; expensive',
        'used by': 'InstructGPT, GPT-3.5/4, Claude 1-3, Llama-2-Chat',
    },
    'DPO': {
        'gradient signal': 'closed-form preference loss; no RM, no online sampling',
        'models in memory': '2 (policy + ref)',
        'pros': 'one model, one loss; offline, deterministic, cheap',
        'cons': 'limited to pairwise preferences; bounded by dataset diversity',
        'used by': 'Zephyr, Tulu, most open-weight chat models since late 2023',
    },
    'GRPO': {
        'gradient signal': 'group-relative advantage (r_i - mean(r))/std(r); PPO clip',
        'models in memory': '2 (policy + ref); no value, no RM',
        'pros': 'scales to verifiable rewards (math/code); no RM bottleneck',
        'cons': 'needs G samples per prompt (compute trade-off vs value net)',
        'used by': 'DeepSeek-Math, DeepSeek-R1',
    },
}
for name, attrs in table.items():
    print(f'\n=== {name} ===')
    for k, v in attrs.items():
        print(f'  {k:<18}: {v}')


### Takeaways for fellowship interviews

- The policy gradient theorem is one log-derivative trick away from the RL objective; baselines are unbiased; GAE is the bias-variance dial.
- TRPO bounds policy improvement by KL; PPO is its first-order surrogate via per-sample ratio clipping.
- GRPO drops the value function by using the group itself as the baseline. Asymptotically unbiased; finite-$G$ bias is $O(1/G)$.
- RLHF, DPO, and GRPO are three coordinates on the same KL-regularized landscape: PPO+RM, closed-form inversion, and group-baseline-without-RM.
- Ch 27 pre-trained model + Ch 31 GRPO loop = the full *pre-train then align* pipeline used by every modern frontier lab.


In [ ]:
print('=' * 60)
print('CHAPTER 31 — final summary')
print('=' * 60)
print(f'  GRPO initial reward       : {GRPO_INIT:.3f}')
print(f'  GRPO final reward         : {GRPO_FINAL:.3f}')
print(f'  Reward delta              : {GRPO_FINAL - GRPO_INIT:+.3f}')
print(f'  KL(pi_theta || pi_ref)    : {GRPO_KL:.4f}')
print(f'  GRPO training time (s)    : {GRPO_TIME:.1f}')
print()
print('  31-chapter chain closed:')
print('    Ch 1-7   foundations (sets, vector spaces, calculus, linear maps)')
print('    Ch 8-15  probability + information theory + optimization')
print('    Ch 16-26 deep nets, transformer, tokenizer, attention')
print('    Ch 27    tiny GPT pre-training')
print('    Ch 28    SFT + DPO post-training')
print('    Ch 29-30 MDP + max-ent RL')
print('    Ch 31    Policy gradient -> PPO -> GRPO -> tiny GPT alignment')
print()
print('  GRPO loop ran on the Ch 27 model with no reward model and no value net,')
print('  measurably improved a verifiable reward, and stayed close to pi_ref.')
print('  This is the closing of the loop.')
